step1_prepare_transcript

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"

# New UBSN ROI encoding project
PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP_DIR = PROJECT_DIR / "step1_prepare_transcript"
CSV_DIR = STEP_DIR / "csv"
JSON_DIR = STEP_DIR / "json"

for d in [PROJECT_DIR, STEP_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"ROI name: {ROI_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step directory: {STEP_DIR}")


# =========================
# 2. Input transcript path
# =========================

TRANSCRIPT_PATH = DATA_ROOT / "align.csv"

if not TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Transcript file not found: {TRANSCRIPT_PATH}\n"
        "Please update TRANSCRIPT_PATH to the real 21styear transcript/alignment file."
    )

print(f"Transcript path: {TRANSCRIPT_PATH}")


# =========================
# 3. Load transcript
# =========================
# Expected format:
# col 0 = original word
# col 1 = cleaned word
# col 2 = onset
# col 3 = offset
# No header row

df = pd.read_csv(TRANSCRIPT_PATH, header=None)

print("\nLoaded transcript preview:")
print(df.head())


# =========================
# 4. Specify transcript columns directly
# =========================

orig_word_col = 0
clean_word_col = 1
onset_col = 2
offset_col = 3

print("\nUsing fixed transcript columns:")
print(f"  original_word_col = {orig_word_col}")
print(f"  clean_word_col    = {clean_word_col}")
print(f"  onset_col         = {onset_col}")
print(f"  offset_col        = {offset_col}")


# =========================
# 5. Build working dataframe
# =========================

work_df = df[[orig_word_col, clean_word_col, onset_col, offset_col]].copy()
work_df.columns = ["word_original", "word_clean", "onset", "offset"]

# Convert timing columns to numeric
work_df["onset"] = pd.to_numeric(work_df["onset"], errors="coerce")
work_df["offset"] = pd.to_numeric(work_df["offset"], errors="coerce")

# Drop rows missing critical information
work_df = work_df.dropna(subset=["word_original", "word_clean", "onset", "offset"]).copy()

# Standardize text columns
work_df["word_original"] = work_df["word_original"].astype(str).str.strip()
work_df["word_clean"] = work_df["word_clean"].astype(str).str.strip()

# Drop rows with empty strings
work_df = work_df[
    (work_df["word_original"] != "") &
    (work_df["word_clean"] != "")
].copy()


# =========================
# 6. Normalize model-facing word forms
# =========================

def normalize_word(w: str) -> str:
    w = str(w).strip().lower()

    # Remove punctuation only at word edges
    w = re.sub(r"^[^\w<]+|[^\w>]+$", "", w)

    # Normalize unknowns
    if w in {"<unk>", "unk", "[unk]"}:
        return "<unk>"

    return w


work_df["word_normalized"] = work_df["word_clean"].apply(normalize_word)

# Remove rows that become empty after normalization
work_df = work_df[work_df["word_normalized"] != ""].copy()

# Keep only valid timing rows
work_df = work_df[work_df["offset"] >= work_df["onset"]].copy()


# =========================
# 7. Sort and add timing-derived columns
# =========================

# Minimal stability improvement:
# sort by onset, then offset, then reset index
work_df = work_df.sort_values(["onset", "offset"]).reset_index(drop=True)

work_df["word_index"] = np.arange(len(work_df))
work_df["duration"] = work_df["offset"] - work_df["onset"]
work_df["word_midpoint"] = (work_df["onset"] + work_df["offset"]) / 2.0

# Reorder columns
work_df = work_df[
    [
        "word_index",
        "word_original",
        "word_clean",
        "word_normalized",
        "onset",
        "offset",
        "duration",
        "word_midpoint",
    ]
].copy()

print("\nCleaned transcript preview:")
print(work_df.head(10))


# =========================
# 8. Basic checks
# =========================

if len(work_df) == 0:
    raise ValueError("No valid transcript rows remain after cleaning.")

onset_non_decreasing = bool(np.all(np.diff(work_df["onset"].values) >= 0))
duration_non_negative = bool(np.all(work_df["duration"].values >= 0))
unique_word_index = bool(work_df["word_index"].is_unique)
finite_onsets = bool(np.isfinite(work_df["onset"].values).all())
finite_offsets = bool(np.isfinite(work_df["offset"].values).all())

if not onset_non_decreasing:
    raise ValueError("Onset times are not nondecreasing after sorting.")
if not duration_non_negative:
    raise ValueError("Negative durations detected after cleaning.")
if not unique_word_index:
    raise ValueError("word_index is not unique.")
if not finite_onsets or not finite_offsets:
    raise ValueError("Non-finite onset/offset values detected.")

# Informative warnings only
if (work_df["duration"] == 0).any():
    print("[WARNING] Some words have zero duration.")

if work_df["word_normalized"].eq("<unk>").any():
    print(f"[INFO] Number of <unk> tokens: {(work_df['word_normalized'] == '<unk>').sum()}")


# =========================
# 9. Save outputs
# =========================

clean_csv_path = CSV_DIR / "21styear_word_timing_clean.csv"
work_df.to_csv(clean_csv_path, index=False, encoding="utf-8-sig")

# Optional duplicate-friendly export for quick inspection
preview_csv_path = CSV_DIR / "21styear_word_timing_clean_preview_first50.csv"
work_df.head(50).to_csv(preview_csv_path, index=False, encoding="utf-8-sig")


# =========================
# 10. Step-end validation summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "encoding_step1_prepare_transcript_cleaned",
    "input_transcript_path": str(TRANSCRIPT_PATH),

    "n_words_total": int(len(work_df)),
    "n_unique_normalized_words": int(work_df["word_normalized"].nunique()),

    "first_onset": float(work_df["onset"].min()),
    "last_offset": float(work_df["offset"].max()),
    "mean_duration": float(work_df["duration"].mean()),
    "median_duration": float(work_df["duration"].median()),
    "min_duration": float(work_df["duration"].min()),
    "max_duration": float(work_df["duration"].max()),

    "columns_used": {
        "word_original": orig_word_col,
        "word_clean": clean_word_col,
        "onset": onset_col,
        "offset": offset_col,
    },

    "checks": {
        "onset_non_decreasing": onset_non_decreasing,
        "duration_non_negative": duration_non_negative,
        "unique_word_index": unique_word_index,
        "finite_onsets": finite_onsets,
        "finite_offsets": finite_offsets,
    },

    "outputs": {
        "clean_csv": str(clean_csv_path),
        "preview_csv": str(preview_csv_path),
    },
}

summary_path = JSON_DIR / "step1_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved cleaned transcript to: {clean_csv_path}")
print(f"Saved preview CSV to: {preview_csv_path}")
print(f"Saved summary to: {summary_path}")


# =========================
# 11. Final step-end checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)
print(f"n_words_total: {len(work_df)}")
print(f"onset_non_decreasing: {onset_non_decreasing}")
print(f"duration_non_negative: {duration_non_negative}")
print(f"unique_word_index: {unique_word_index}")
print(f"finite_onsets: {finite_onsets}")
print(f"finite_offsets: {finite_offsets}")
print(f"first_onset: {work_df['onset'].min():.6f}")
print(f"last_offset: {work_df['offset'].max():.6f}")
print(f"mean_duration: {work_df['duration'].mean():.6f}")
print(f"median_duration: {work_df['duration'].median():.6f}")

print("\nStep 1 completed successfully.")


Step2_extract_gpt2xl_embeddings.

b2b-linguistic-coupling：
https://github.com/hassonlab/b2b-linguistic-coupling


add_causal_lm_embs(...)

In [ ]:
from pathlib import Path
import json
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"

# New UBSN ROI encoding project
PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP1_DIR = PROJECT_DIR / "step1_prepare_transcript"
STEP2_DIR = PROJECT_DIR / "step2_extract_gpt2xl_alllayer_contextual_embeddings"
CSV_DIR = STEP2_DIR / "csv"
NPY_DIR = STEP2_DIR / "npy"
LAYER_DIR = NPY_DIR / "layers"
JSON_DIR = STEP2_DIR / "json"

for d in [PROJECT_DIR, STEP2_DIR, CSV_DIR, NPY_DIR, LAYER_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"ROI name: {ROI_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 1 directory: {STEP1_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")


# =========================
# 2. Input word table
# =========================

WORD_TABLE_PATH = STEP1_DIR / "csv" / "21styear_word_timing_clean.csv"
STEP1_SUMMARY_PATH = STEP1_DIR / "json" / "step1_summary.json"

if not WORD_TABLE_PATH.exists():
    raise FileNotFoundError(f"Word timing file not found: {WORD_TABLE_PATH}")
if not STEP1_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Step1 summary not found: {STEP1_SUMMARY_PATH}")

print(f"Word table path: {WORD_TABLE_PATH}")
print(f"Step1 summary path: {STEP1_SUMMARY_PATH}")

with open(STEP1_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step1_summary = json.load(f)


# =========================
# 3. Model setup
# =========================

MODEL_NAME = "gpt2-xl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Model name: {MODEL_NAME}")
print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
model.to(DEVICE)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_CONTEXT = int(tokenizer.model_max_length)
if MAX_CONTEXT is None or MAX_CONTEXT <= 0 or MAX_CONTEXT > 100000:
    MAX_CONTEXT = 1024

OVERLAP_TOKENS = 256
STEP_SIZE = MAX_CONTEXT - OVERLAP_TOKENS

print(f"MAX_CONTEXT = {MAX_CONTEXT}")
print(f"OVERLAP_TOKENS = {OVERLAP_TOKENS}")
print(f"STEP_SIZE = {STEP_SIZE}")

if STEP_SIZE <= 0:
    raise ValueError("STEP_SIZE must be > 0. Please re-check MAX_CONTEXT and OVERLAP_TOKENS.")


# =========================
# 4. Load cleaned transcript
# =========================

word_df = pd.read_csv(WORD_TABLE_PATH)

required_cols = [
    "word_index",
    "word_original",
    "word_clean",
    "word_normalized",
    "onset",
    "offset",
    "duration",
    "word_midpoint",
]
missing = [c for c in required_cols if c not in word_df.columns]
if missing:
    raise ValueError(f"Missing required columns in transcript file: {missing}")

word_df = word_df.copy().reset_index(drop=True)

print("\nLoaded cleaned transcript preview:")
print(word_df.head())
print(f"Number of words: {len(word_df)}")

if len(word_df) == 0:
    raise ValueError("The cleaned transcript is empty.")

if not word_df["word_index"].is_unique:
    raise ValueError("word_index is not unique in the transcript.")
if not np.all(np.diff(word_df["onset"].values) >= 0):
    raise ValueError("Transcript onset times are not nondecreasing.")
if not np.all(word_df["offset"].values >= word_df["onset"].values):
    raise ValueError("Found offset < onset in transcript.")


# =========================
# 5. Tokenize each word and build full token sequence
# =========================

all_token_ids = []
all_token_strs = []
token_to_word = []
word_token_start = []
word_token_end = []
word_n_subtokens = []

for word_idx, word in tqdm(
    enumerate(word_df["word_normalized"].astype(str).tolist()),
    total=len(word_df),
    desc="Tokenizing words"
):
    encoded = tokenizer(
        word,
        add_special_tokens=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    token_ids = encoded["input_ids"]
    token_strs = tokenizer.convert_ids_to_tokens(token_ids)

    if len(token_ids) == 0:
        # fallback
        token_ids = [tokenizer.eos_token_id]
        token_strs = [tokenizer.eos_token]

    start = len(all_token_ids)
    end = start + len(token_ids)

    word_token_start.append(start)
    word_token_end.append(end)
    word_n_subtokens.append(len(token_ids))

    all_token_ids.extend(token_ids)
    all_token_strs.extend(token_strs)
    token_to_word.extend([word_idx] * len(token_ids))

all_token_ids = np.asarray(all_token_ids, dtype=np.int64)
token_to_word = np.asarray(token_to_word, dtype=np.int32)
word_token_start = np.asarray(word_token_start, dtype=np.int32)
word_token_end = np.asarray(word_token_end, dtype=np.int32)
word_n_subtokens = np.asarray(word_n_subtokens, dtype=np.int32)

print("\nTokenization summary:")
print(f"Total words: {len(word_df)}")
print(f"Total subtokens: {len(all_token_ids)}")
print(f"Mean subtokens per word: {word_n_subtokens.mean():.3f}")
print(f"Median subtokens per word: {np.median(word_n_subtokens):.3f}")
print(f"Max subtokens per word: {word_n_subtokens.max()}")

if len(all_token_ids) == 0:
    raise ValueError("Tokenization produced zero tokens.")


# =========================
# 6. Build overlapping windows across full token sequence
# =========================

def build_window_starts(n_tokens: int, max_context: int, step_size: int):
    if n_tokens <= max_context:
        return [0]

    starts = list(range(0, n_tokens - max_context + 1, step_size))
    last_start = n_tokens - max_context
    if starts[-1] != last_start:
        starts.append(last_start)
    return starts


window_starts = build_window_starts(
    n_tokens=len(all_token_ids),
    max_context=MAX_CONTEXT,
    step_size=STEP_SIZE
)

print(f"\nNumber of windows: {len(window_starts)}")

# Assign each token to the earliest window that contains it
# so each token gets the largest available left context
token_assigned_window = np.full(len(all_token_ids), fill_value=-1, dtype=np.int32)

for win_idx, start in enumerate(window_starts):
    end = min(start + MAX_CONTEXT, len(all_token_ids))
    token_slice = np.arange(start, end, dtype=np.int32)
    unassigned = token_assigned_window[token_slice] == -1
    token_assigned_window[token_slice[unassigned]] = win_idx

if np.any(token_assigned_window < 0):
    raise RuntimeError("Some tokens were not assigned to any window.")


# =========================
# 7. Prepare output memmap
# =========================

n_words = len(word_df)
hidden_size = int(model.config.hidden_size)
n_layers_total = int(model.config.n_layer) + 1   # embedding layer + transformer outputs

ALL_LAYER_MEMMAP_PATH = NPY_DIR / "gpt2xl_word_embeddings_all_layers_memmap.npy"
all_layer_memmap = np.lib.format.open_memmap(
    ALL_LAYER_MEMMAP_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(n_layers_total, n_words, hidden_size),
)

print("\nEmbedding storage prepared:")
print(f"n_layers_total = {n_layers_total}")
print(f"hidden_size = {hidden_size}")
print(f"Memmap path = {ALL_LAYER_MEMMAP_PATH}")


# =========================
# 8. Extract contextual all-layer activations
# =========================

start_time = time.time()

with torch.no_grad():
    for win_idx, start in enumerate(tqdm(window_starts, desc="Running contextual windows")):
        end = min(start + MAX_CONTEXT, len(all_token_ids))
        window_token_ids = all_token_ids[start:end]

        input_ids = torch.tensor(window_token_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
        attention_mask = torch.ones_like(input_ids, device=DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )

        hidden_states = outputs.hidden_states

        if len(hidden_states) != n_layers_total:
            raise ValueError(
                f"Expected {n_layers_total} hidden-state tensors, got {len(hidden_states)}."
            )

        local_positions = []
        global_token_indices = []

        for local_pos in range(end - start):
            global_idx = start + local_pos
            if token_assigned_window[global_idx] == win_idx:
                local_positions.append(local_pos)
                global_token_indices.append(global_idx)

        if len(local_positions) == 0:
            continue

        local_positions = np.asarray(local_positions, dtype=np.int64)
        global_token_indices = np.asarray(global_token_indices, dtype=np.int64)
        local_word_ids = token_to_word[global_token_indices]

        # mean-pool later, so now accumulate token vectors into each word
        for layer_idx in range(n_layers_total):
            layer_arr = hidden_states[layer_idx][0, local_positions, :].detach().cpu().numpy().astype(np.float32)
            np.add.at(all_layer_memmap[layer_idx], local_word_ids, layer_arr)

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

elapsed_minutes = (time.time() - start_time) / 60.0
print(f"\nContextual all-layer extraction completed.")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")


# =========================
# 9. Mean-pool subtokens within each word
# =========================

subtoken_denominator = np.maximum(word_n_subtokens.astype(np.float32), 1.0)

for layer_idx in tqdm(range(n_layers_total), desc="Averaging subtokens into word embeddings"):
    all_layer_memmap[layer_idx] /= subtoken_denominator[:, None]

all_layer_memmap.flush()


# =========================
# 10. Save each layer separately
# =========================

layer_rows = []

for layer_idx in range(n_layers_total):
    layer_arr = np.array(all_layer_memmap[layer_idx], dtype=np.float32)
    layer_path = LAYER_DIR / f"layer_{layer_idx:02d}.npy"
    np.save(layer_path, layer_arr)

    layer_rows.append({
        "layer_index": layer_idx,
        "path": str(layer_path),
        "shape": str(layer_arr.shape),
    })

layer_df = pd.DataFrame(layer_rows)
layer_df.to_csv(CSV_DIR / "layer_file_manifest.csv", index=False, encoding="utf-8-sig")

print(f"\nSaved per-layer embeddings to: {LAYER_DIR}")


# =========================
# 11. Save word-token alignment table
# =========================

token_df = pd.DataFrame({
    "global_token_index": np.arange(len(all_token_ids)),
    "token_id": all_token_ids,
    "token_str": all_token_strs,
    "word_index": token_to_word,
    "assigned_window": token_assigned_window,
})

token_df_path = CSV_DIR / "token_word_alignment.csv"
token_df.to_csv(token_df_path, index=False, encoding="utf-8-sig")

word_token_df = word_df.copy()
word_token_df["token_start"] = word_token_start
word_token_df["token_end"] = word_token_end
word_token_df["n_subtokens"] = word_n_subtokens

word_token_df_path = CSV_DIR / "word_timing_with_token_spans.csv"
word_token_df.to_csv(word_token_df_path, index=False, encoding="utf-8-sig")

print(f"Saved token-word alignment table to: {token_df_path}")
print(f"Saved word-token span table to: {word_token_df_path}")


# =========================
# 12. Save JSON summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "alllayer_contextual_windows_cleaned",
    "input_word_table_path": str(WORD_TABLE_PATH),
    "source_step1_summary_path": str(STEP1_SUMMARY_PATH),

    "model_name": MODEL_NAME,
    "device": DEVICE,
    "max_context": int(MAX_CONTEXT),
    "overlap_tokens": int(OVERLAP_TOKENS),
    "step_size": int(STEP_SIZE),

    "n_words": int(n_words),
    "n_subtokens_total": int(len(all_token_ids)),
    "mean_subtokens_per_word": float(word_n_subtokens.mean()),
    "median_subtokens_per_word": float(np.median(word_n_subtokens)),
    "max_subtokens_per_word": int(word_n_subtokens.max()),

    "n_windows": int(len(window_starts)),
    "hidden_size": int(hidden_size),
    "n_layers_total": int(n_layers_total),

    "all_layer_memmap_shape": [int(n_layers_total), int(n_words), int(hidden_size)],
    "elapsed_minutes": float(elapsed_minutes),

    "outputs": {
        "all_layer_memmap": str(ALL_LAYER_MEMMAP_PATH),
        "layer_manifest_csv": str(CSV_DIR / "layer_file_manifest.csv"),
        "token_alignment_csv": str(token_df_path),
        "word_token_span_csv": str(word_token_df_path),
        "layer_dir": str(LAYER_DIR),
    },
}

summary_path = JSON_DIR / "step2_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved summary to: {summary_path}")


# =========================
# 13. Final step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

all_layer_shape_ok = tuple(all_layer_memmap.shape) == (n_layers_total, n_words, hidden_size)
all_word_assignment_ok = len(word_token_start) == n_words and len(word_token_end) == n_words
all_token_assignment_ok = np.all(token_assigned_window >= 0)
all_finite_sample_ok = np.isfinite(np.array(all_layer_memmap[0, :10, :10])).all()

print(f"all_layer_shape_ok: {all_layer_shape_ok}")
print(f"all_word_assignment_ok: {all_word_assignment_ok}")
print(f"all_token_assignment_ok: {all_token_assignment_ok}")
print(f"all_finite_sample_ok: {all_finite_sample_ok}")
print(f"n_words: {n_words}")
print(f"n_subtokens_total: {len(all_token_ids)}")
print(f"n_windows: {len(window_starts)}")
print(f"n_layers_total: {n_layers_total}")
print(f"hidden_size: {hidden_size}")
print(f"elapsed_minutes: {elapsed_minutes:.2f}")

if not all_layer_shape_ok:
    raise ValueError("Final memmap shape is incorrect.")
if not all_word_assignment_ok:
    raise ValueError("Word-token span arrays do not match n_words.")
if not all_token_assignment_ok:
    raise ValueError("Some tokens were not assigned to any window.")
if not all_finite_sample_ok:
    raise ValueError("Non-finite values detected in sampled embedding output.")

print("\nStep 2 completed successfully.")


This step extracts a contextual GPT-2 XL representation for each word by feeding the current word together with its preceding context into the causal language model and taking the hidden state of the word’s last subtoken as its embedding.

“Brain-to-brain coupling in a shared linguistic embedding space during natural conversations” coding from many papers and this is just one of them~

step3_reduce_embeddings_pca

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"

# New UBSN ROI encoding project
PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP2_DIR = PROJECT_DIR / "step2_extract_gpt2xl_alllayer_contextual_embeddings"
STEP3_DIR = PROJECT_DIR / "step3_reduce_embeddings_pca"

CSV_DIR = STEP3_DIR / "csv"
NPY_DIR = STEP3_DIR / "npy"
JSON_DIR = STEP3_DIR / "json"

for d in [PROJECT_DIR, STEP3_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"ROI name: {ROI_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 2 directory: {STEP2_DIR}")
print(f"Step 3 directory: {STEP3_DIR}")


# =========================
# 2. Input settings
# =========================

WORD_TABLE_PATH = STEP2_DIR / "csv" / "word_timing_with_token_spans.csv"
STEP2_SUMMARY_PATH = STEP2_DIR / "json" / "step2_summary.json"

# Use final layer from all-layer extraction
ALL_LAYER_MEMMAP_PATH = STEP2_DIR / "npy" / "gpt2xl_word_embeddings_all_layers_memmap.npy"

PCA_COMPONENTS = 50

# Keep the same split logic as before
TR = 1.5
STORY_START_TR = 14
STORY_N_TR = 2226
TRAIN_TRS = STORY_N_TR // 2   # 1113
TRAIN_TEST_SPLIT_TIME = STORY_START_TR * TR + TRAIN_TRS * TR   # 1690.5 s

for p in [WORD_TABLE_PATH, STEP2_SUMMARY_PATH, ALL_LAYER_MEMMAP_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

print(f"Word table path: {WORD_TABLE_PATH}")
print(f"All-layer memmap path: {ALL_LAYER_MEMMAP_PATH}")
print(f"PCA components: {PCA_COMPONENTS}")
print(f"Train/test split time (seconds): {TRAIN_TEST_SPLIT_TIME}")


# =========================
# 3. Load data
# =========================

word_df = pd.read_csv(WORD_TABLE_PATH)

with open(STEP2_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step2_summary = json.load(f)

required_cols = ["word_index", "onset", "offset", "word_midpoint"]
missing = [c for c in required_cols if c not in word_df.columns]
if missing:
    raise ValueError(f"Missing required columns in word table: {missing}")

n_layers_total = int(step2_summary["n_layers_total"])
n_words = int(step2_summary["n_words"])
hidden_size = int(step2_summary["hidden_size"])

all_layer_memmap = np.lib.format.open_memmap(
    ALL_LAYER_MEMMAP_PATH,
    mode="r"
)

expected_shape = (n_layers_total, n_words, hidden_size)
if tuple(all_layer_memmap.shape) != expected_shape:
    raise ValueError(
        f"All-layer memmap shape mismatch: got {tuple(all_layer_memmap.shape)}, "
        f"expected {expected_shape}"
    )

# Final layer = last transformer output
FINAL_LAYER_INDEX = n_layers_total - 1
embeddings = np.array(all_layer_memmap[FINAL_LAYER_INDEX], dtype=np.float32)

print("\nLoaded data:")
print("word_df shape:", word_df.shape)
print("all_layer_memmap shape:", tuple(all_layer_memmap.shape))
print("final layer index:", FINAL_LAYER_INDEX)
print("final layer embeddings shape:", embeddings.shape)

if len(word_df) != embeddings.shape[0]:
    raise ValueError(
        f"Mismatch between number of words ({len(word_df)}) and embedding rows ({embeddings.shape[0]})."
    )

if not np.isfinite(embeddings[:10, :10]).all():
    raise ValueError("Non-finite values detected in sampled final-layer embeddings.")


# =========================
# 4. Define train/test word split
# =========================

word_df = word_df.copy()
word_df["split"] = np.where(word_df["word_midpoint"] < TRAIN_TEST_SPLIT_TIME, "train", "test")

train_mask = word_df["split"] == "train"
test_mask = word_df["split"] == "test"

train_embeddings = embeddings[train_mask.values]
test_embeddings = embeddings[test_mask.values]

print("\nWord split summary:")
print(word_df["split"].value_counts())
print("train_embeddings shape:", train_embeddings.shape)
print("test_embeddings shape:", test_embeddings.shape)

if train_embeddings.shape[0] == 0 or test_embeddings.shape[0] == 0:
    raise ValueError("Train/test split produced zero words in one partition.")


# =========================
# 5. Fit PCA on train only
# =========================

start_time = time.time()

pca = PCA(n_components=PCA_COMPONENTS, svd_solver="full", random_state=0)
pca.fit(train_embeddings)

train_reduced = pca.transform(train_embeddings).astype(np.float32)
test_reduced = pca.transform(test_embeddings).astype(np.float32)
all_reduced = pca.transform(embeddings).astype(np.float32)

elapsed_minutes = (time.time() - start_time) / 60.0

print("\nPCA finished.")
print("train_reduced shape:", train_reduced.shape)
print("test_reduced shape:", test_reduced.shape)
print("all_reduced shape:", all_reduced.shape)
print(f"Elapsed minutes: {elapsed_minutes:.2f}")


# =========================
# 6. Save arrays
# =========================

np.save(NPY_DIR / "gpt2xl_word_embeddings_final_layer.npy", embeddings)
np.save(NPY_DIR / "gpt2xl_word_embeddings_pca50_all.npy", all_reduced)
np.save(NPY_DIR / "gpt2xl_word_embeddings_pca50_train.npy", train_reduced)
np.save(NPY_DIR / "gpt2xl_word_embeddings_pca50_test.npy", test_reduced)

np.save(NPY_DIR / "pca_components.npy", pca.components_.astype(np.float32))
np.save(NPY_DIR / "pca_mean.npy", pca.mean_.astype(np.float32))
np.save(NPY_DIR / "pca_explained_variance_ratio.npy", pca.explained_variance_ratio_.astype(np.float32))


# =========================
# 7. Save explained variance table
# =========================

ev_df = pd.DataFrame({
    "component": np.arange(1, PCA_COMPONENTS + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
})
ev_csv_path = CSV_DIR / "pca_explained_variance.csv"
ev_df.to_csv(ev_csv_path, index=False, encoding="utf-8-sig")

print(f"\nSaved explained variance table to: {ev_csv_path}")


# =========================
# 8. Save split word table
# =========================

split_word_table_path = CSV_DIR / "word_timing_with_split.csv"
word_df.to_csv(split_word_table_path, index=False, encoding="utf-8-sig")

print(f"Saved split word table to: {split_word_table_path}")


# =========================
# 9. Save JSON summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "final_layer_only_new_contextual_windows_cleaned",
    "input_all_layer_memmap_path": str(ALL_LAYER_MEMMAP_PATH),
    "input_word_table_path": str(WORD_TABLE_PATH),
    "model_name": step2_summary.get("model_name", "gpt2-xl"),
    "final_layer_index": int(FINAL_LAYER_INDEX),
    "pca_components": PCA_COMPONENTS,

    "tr": TR,
    "story_start_tr": STORY_START_TR,
    "story_n_tr": STORY_N_TR,
    "train_trs": TRAIN_TRS,
    "train_test_split_time": float(TRAIN_TEST_SPLIT_TIME),

    "n_words_total": int(len(word_df)),
    "n_train_words": int(train_mask.sum()),
    "n_test_words": int(test_mask.sum()),

    "final_layer_embedding_shape": list(embeddings.shape),
    "train_reduced_shape": list(train_reduced.shape),
    "test_reduced_shape": list(test_reduced.shape),
    "all_reduced_shape": list(all_reduced.shape),

    "explained_variance_first_component": float(pca.explained_variance_ratio_[0]),
    "explained_variance_cumulative_10": float(np.sum(pca.explained_variance_ratio_[:10])),
    "explained_variance_cumulative_20": float(np.sum(pca.explained_variance_ratio_[:20])),
    "explained_variance_cumulative_50": float(np.sum(pca.explained_variance_ratio_[:50])),

    "elapsed_minutes": float(elapsed_minutes),

    "outputs": {
        "final_layer_embedding_path": str(NPY_DIR / "gpt2xl_word_embeddings_final_layer.npy"),
        "pca_all_path": str(NPY_DIR / "gpt2xl_word_embeddings_pca50_all.npy"),
        "pca_train_path": str(NPY_DIR / "gpt2xl_word_embeddings_pca50_train.npy"),
        "pca_test_path": str(NPY_DIR / "gpt2xl_word_embeddings_pca50_test.npy"),
        "pca_components_path": str(NPY_DIR / "pca_components.npy"),
        "pca_mean_path": str(NPY_DIR / "pca_mean.npy"),
        "pca_explained_variance_ratio_path": str(NPY_DIR / "pca_explained_variance_ratio.npy"),
        "explained_variance_csv": str(ev_csv_path),
        "split_word_table_csv": str(split_word_table_path),
    },
}

summary_path = JSON_DIR / "step3_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved summary to: {summary_path}")


# =========================
# 10. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

shape_ok = (
    train_reduced.shape[1] == PCA_COMPONENTS and
    test_reduced.shape[1] == PCA_COMPONENTS and
    all_reduced.shape[1] == PCA_COMPONENTS
)
split_ok = (train_mask.sum() + test_mask.sum()) == len(word_df)
finite_ok = (
    np.isfinite(train_reduced).all() and
    np.isfinite(test_reduced).all() and
    np.isfinite(all_reduced).all()
)
ev_monotonic = np.all(np.diff(np.cumsum(pca.explained_variance_ratio_)) >= -1e-8)

print(f"shape_ok: {shape_ok}")
print(f"split_ok: {split_ok}")
print(f"finite_ok: {finite_ok}")
print(f"ev_monotonic: {ev_monotonic}")
print(f"n_train_words: {int(train_mask.sum())}")
print(f"n_test_words: {int(test_mask.sum())}")
print(f"final_layer_index: {FINAL_LAYER_INDEX}")
print(f"explained_variance_cumulative_50: {np.sum(pca.explained_variance_ratio_[:50]):.6f}")

if not shape_ok:
    raise ValueError("PCA output shapes are incorrect.")
if not split_ok:
    raise ValueError("Train/test split count does not sum to total words.")
if not finite_ok:
    raise ValueError("Non-finite values detected in PCA outputs.")
if not ev_monotonic:
    raise ValueError("Cumulative explained variance is not monotonic.")

print("\nStep 3 completed successfully.")


Step 4. Downsample word features to TR-level features — LANCZOS version

"Interpolation of the feature matrix
One challenge in fitting encoding models is that speech and BOLD data are sampled at very different frequencies. Approximately six words are spoken every two seconds, but only one brain image is recorded in that interval. To solve this problem, the stimulus matrix needs to be resampled to the same sampling frequency as the BOLD data. The procedure we provide for downsampling features to the fMRI acquisition rate can be thought of as comprising three steps. First, the discrete features for each word (or phoneme) are transformed into a continuous-time representation N(t) where t ∈ [0, T] and T indicates the length of the stimulus. This representation is zero at all timepoints except for the exact middle of each word (or phoneme), where it is equal to an infinitesimal-duration spike (Dirac δ-function) that is scaled by the feature value. Next, a low-pass antialiasing Lanczos filter is convolved with N(t) to get NLP(t). The cutoff frequency of this antialiasing filter is selected to match the Nyquist frequency of the fMRI data (half the acquisition rate, or 0.25 Hz). The cutoff frequency and filter roll-off (controlled by the number of lobes: more lobes yield a sharper roll-off, but at the cost of potentially increased noise) can be selected manually, although we recommend using the default values. Finally, NLP(t) is sampled at the fMRI acquisition times tr where r ∈ [1, 2….nTR] corresponds to the volume index in the fMRI acquisition. In practice, these three steps are accomplished simultaneously by way of a single matrix multiplication: the word- (or phoneme-) level stimulus matrix S (number of features by number of words/phonemes) is multiplied by a sparse “Lanczos” matrix L (number of words/phonemes by number of fMRI volumes). In essence, this assumes that the total brain response is the sum of responses to each word or phoneme. This approach has been widely used for language encoding models with natural stimuli10. An alternative to this approach would be to simply average the feature vectors for all the word or phonemes that appear within each 2-second period. However, that approach leads to discontinuities since words that fall infinitesimally before or after a boundary wind up in different time bins. The Lanczos method naturally accounts for this issue: if a word falls exactly at the boundary between two time bins, its features contribute equally to both (albeit scaled by 50%)."

Step4. Downsample word features to TR-level features — LANCZOS version

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"

# New UBSN ROI encoding project
PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP3_DIR = PROJECT_DIR / "step3_reduce_embeddings_pca"
STEP4_DIR = PROJECT_DIR / "step4_downsample_to_tr_lanczos"

CSV_DIR = STEP4_DIR / "csv"
NPY_DIR = STEP4_DIR / "npy"
JSON_DIR = STEP4_DIR / "json"

for d in [PROJECT_DIR, STEP4_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"ROI name: {ROI_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 3 directory: {STEP3_DIR}")
print(f"Step 4 directory: {STEP4_DIR}")


# =========================
# 2. Input settings
# =========================

WORD_TABLE_PATH = STEP3_DIR / "csv" / "word_timing_with_split.csv"
EMBEDDING_PATH = STEP3_DIR / "npy" / "gpt2xl_word_embeddings_pca50_all.npy"
STEP3_SUMMARY_PATH = STEP3_DIR / "json" / "step3_summary.json"

TR = 1.5
STORY_START_TR = 14
STORY_N_TR = 2226
TRAIN_TRS = STORY_N_TR // 2  # 1113

# Huth-style timing logic:
# first define TR onset times, then explicitly shift by TR/2.0
STORY_START_TIME = STORY_START_TR * TR
TR_ONSET_TIMES = STORY_START_TIME + np.arange(STORY_N_TR) * TR
TR_MIDPOINT_TIMES = TR_ONSET_TIMES + TR / 2.0

# Lanczos parameters
LANCZOS_WINDOW = 3
LANCZOS_CUTOFF_MULT = 1.0

for p in [WORD_TABLE_PATH, EMBEDDING_PATH, STEP3_SUMMARY_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

print(f"Word table path: {WORD_TABLE_PATH}")
print(f"Embedding path: {EMBEDDING_PATH}")
print(f"TR = {TR}")
print(f"STORY_START_TIME = {STORY_START_TIME}")
print(f"STORY_N_TR = {STORY_N_TR}")
print(f"LANCZOS_WINDOW = {LANCZOS_WINDOW}")
print(f"LANCZOS_CUTOFF_MULT = {LANCZOS_CUTOFF_MULT}")


# =========================
# 3. Huth-style Lanczos functions
# =========================

def lanczosfun(cutoff: float, t, window: int = 3) -> np.ndarray:
    """
    Compute the Lanczos kernel at time offset t.
    """
    t = np.asarray(t, dtype=np.float64) * cutoff
    val = np.zeros_like(t, dtype=np.float64)

    nonzero = t != 0
    val[nonzero] = (
        window
        * np.sin(np.pi * t[nonzero])
        * np.sin(np.pi * t[nonzero] / window)
        / (np.pi ** 2 * t[nonzero] ** 2)
    )
    val[~nonzero] = 1.0
    val[np.abs(t) > window] = 0.0
    return val


def interp2D_lanczos(
    data: np.ndarray,
    oldtime: np.ndarray,
    newtime: np.ndarray,
    window: int = 3,
    cutoff_mult: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Interpolate columns of data from oldtime to newtime using Lanczos interpolation.
    """
    cutoff = 1.0 / np.mean(np.diff(newtime)) * cutoff_mult

    print(f"\nDoing Lanczos interpolation with cutoff={cutoff:.6f} and {window} lobes.")

    lanczos_matrix = np.zeros((len(newtime), len(oldtime)), dtype=np.float64)
    for ndi in range(len(newtime)):
        lanczos_matrix[ndi, :] = lanczosfun(cutoff, newtime[ndi] - oldtime, window)

    newdata = np.dot(lanczos_matrix, data)
    return newdata.astype(np.float32), lanczos_matrix.astype(np.float32)


# =========================
# 4. Lightweight DataSequence-style container
# =========================

class EncodingDataSequence:
    """
    Minimal DataSequence-style container for this project.
    """

    def __init__(self, data: np.ndarray, data_times: np.ndarray, tr_times: np.ndarray):
        self.data = data
        self.data_times = data_times
        self.tr_times = tr_times

    def chunksums(self, interp: str = "lanczos", **kwargs) -> tuple[np.ndarray, np.ndarray]:
        if interp != "lanczos":
            raise ValueError("This project-specific DataSequence only supports interp='lanczos'.")
        return interp2D_lanczos(
            data=self.data,
            oldtime=self.data_times,
            newtime=self.tr_times,
            **kwargs,
        )


# =========================
# 5. Load data
# =========================

word_df = pd.read_csv(WORD_TABLE_PATH)
embeddings = np.load(EMBEDDING_PATH).astype(np.float32)

with open(STEP3_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step3_summary = json.load(f)

required_cols = ["word_index", "word_midpoint", "split"]
missing = [c for c in required_cols if c not in word_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(word_df) != embeddings.shape[0]:
    raise ValueError(
        f"Mismatch between word table rows ({len(word_df)}) and embedding rows ({embeddings.shape[0]})."
    )

print("\nLoaded inputs:")
print("word_df shape:", word_df.shape)
print("embeddings shape:", embeddings.shape)


# =========================
# 6. Prepare Huth-style timing variables
# =========================

word_midpoint_times = word_df["word_midpoint"].to_numpy(dtype=np.float64)
tr_midpoint_times = TR_MIDPOINT_TIMES.astype(np.float64)

print("\nTiming preview:")
print("First 5 word midpoint times:", word_midpoint_times[:5])
print("First 5 TR onset times:", TR_ONSET_TIMES[:5])
print("First 5 TR midpoint times:", tr_midpoint_times[:5])

if not np.all(np.diff(word_midpoint_times) >= 0):
    raise ValueError("word_midpoint_times must be nondecreasing.")
if len(tr_midpoint_times) != STORY_N_TR:
    raise ValueError(f"TR midpoint count mismatch: {len(tr_midpoint_times)} vs expected {STORY_N_TR}")


# =========================
# 7. Downsample to TR with Lanczos
# =========================

start_time = time.time()

ds = EncodingDataSequence(
    data=embeddings,
    data_times=word_midpoint_times,
    tr_times=tr_midpoint_times,
)

tr_features, lanczos_matrix = ds.chunksums(
    interp="lanczos",
    window=LANCZOS_WINDOW,
    cutoff_mult=LANCZOS_CUTOFF_MULT,
)

elapsed_minutes = (time.time() - start_time) / 60.0

print("\nLanczos downsampling finished.")
print("tr_features shape:", tr_features.shape)
print("lanczos_matrix shape:", lanczos_matrix.shape)
print(f"Elapsed minutes: {elapsed_minutes:.2f}")


# =========================
# 8. Recover train/test TR partitions
# =========================

tr_features_train = tr_features[:TRAIN_TRS].astype(np.float32)
tr_features_test = tr_features[TRAIN_TRS:].astype(np.float32)

print("\nTR split summary:")
print("tr_features_train shape:", tr_features_train.shape)
print("tr_features_test shape:", tr_features_test.shape)

if tr_features.shape[0] != STORY_N_TR:
    raise ValueError(f"TR feature rows mismatch: {tr_features.shape[0]} vs expected {STORY_N_TR}")
if tr_features_train.shape[0] != TRAIN_TRS:
    raise ValueError(f"Train TR count mismatch: {tr_features_train.shape[0]} vs expected {TRAIN_TRS}")
if tr_features_test.shape[0] != (STORY_N_TR - TRAIN_TRS):
    raise ValueError(
        f"Test TR count mismatch: {tr_features_test.shape[0]} vs expected {STORY_N_TR - TRAIN_TRS}"
    )


# =========================
# 9. Save arrays
# =========================

np.save(NPY_DIR / "tr_features_pca50_lanczos_all.npy", tr_features)
np.save(NPY_DIR / "tr_features_pca50_lanczos_train.npy", tr_features_train)
np.save(NPY_DIR / "tr_features_pca50_lanczos_test.npy", tr_features_test)
np.save(NPY_DIR / "lanczos_interpolation_matrix.npy", lanczos_matrix)
np.save(NPY_DIR / "tr_onset_times.npy", TR_ONSET_TIMES.astype(np.float32))
np.save(NPY_DIR / "tr_midpoint_times.npy", TR_MIDPOINT_TIMES.astype(np.float32))


# =========================
# 10. Save TR time table
# =========================

tr_time_table = pd.DataFrame({
    "tr_index": np.arange(STORY_N_TR),
    "tr_onset_time": TR_ONSET_TIMES,
    "tr_midpoint_time": TR_MIDPOINT_TIMES,
    "split": np.where(np.arange(STORY_N_TR) < TRAIN_TRS, "train", "test"),
})

tr_time_table_path = CSV_DIR / "tr_time_table.csv"
tr_time_table.to_csv(tr_time_table_path, index=False, encoding="utf-8-sig")

print(f"\nSaved TR time table to: {tr_time_table_path}")


# =========================
# 11. Save summary CSV
# =========================

summary_df = pd.DataFrame({
    "pipeline_type": ["final_layer_only_new_contextual_windows_cleaned"],
    "method": ["Lanczos interpolation"],
    "n_input_words": [len(word_df)],
    "input_feature_dim": [embeddings.shape[1]],
    "output_trs_total": [tr_features.shape[0]],
    "train_trs": [tr_features_train.shape[0]],
    "test_trs": [tr_features_test.shape[0]],
    "output_feature_dim": [tr_features.shape[1]],
    "lanczos_window": [LANCZOS_WINDOW],
    "lanczos_cutoff_mult": [LANCZOS_CUTOFF_MULT],
    "elapsed_minutes": [elapsed_minutes],
})

summary_csv_path = CSV_DIR / "tr_feature_summary.csv"
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

print(f"Saved TR feature summary to: {summary_csv_path}")


# =========================
# 12. Save JSON summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "final_layer_only_new_contextual_windows_cleaned",
    "method": "Lanczos interpolation",
    "implementation_style": "Huth-inspired DataSequence-style resampling",
    "input_word_table_path": str(WORD_TABLE_PATH),
    "input_embedding_path": str(EMBEDDING_PATH),
    "step3_summary_path": str(STEP3_SUMMARY_PATH),

    "timing_logic": {
        "word_time_definition": "word midpoint times",
        "tr_time_definition": "TR onset times shifted by TR/2.0 to obtain TR midpoint times",
        "story_start_tr": int(STORY_START_TR),
        "story_n_tr": int(STORY_N_TR),
        "tr_seconds": float(TR),
        "story_start_time_seconds": float(STORY_START_TIME),
    },

    "lanczos_parameters": {
        "window": int(LANCZOS_WINDOW),
        "cutoff_mult": float(LANCZOS_CUTOFF_MULT),
    },

    "shapes": {
        "embeddings_input": list(embeddings.shape),
        "tr_features_all": list(tr_features.shape),
        "tr_features_train": list(tr_features_train.shape),
        "tr_features_test": list(tr_features_test.shape),
        "lanczos_matrix": list(lanczos_matrix.shape),
    },

    "train_test_split": {
        "train_trs": int(TRAIN_TRS),
        "test_trs": int(STORY_N_TR - TRAIN_TRS),
    },

    "outputs": {
        "tr_features_all": str(NPY_DIR / "tr_features_pca50_lanczos_all.npy"),
        "tr_features_train": str(NPY_DIR / "tr_features_pca50_lanczos_train.npy"),
        "tr_features_test": str(NPY_DIR / "tr_features_pca50_lanczos_test.npy"),
        "lanczos_matrix": str(NPY_DIR / "lanczos_interpolation_matrix.npy"),
        "tr_onset_times": str(NPY_DIR / "tr_onset_times.npy"),
        "tr_midpoint_times": str(NPY_DIR / "tr_midpoint_times.npy"),
        "tr_time_table": str(tr_time_table_path),
        "summary_csv": str(summary_csv_path),
    },

    "elapsed_minutes": float(elapsed_minutes),
}

summary_json_path = JSON_DIR / "step4_lanczos_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved JSON summary to: {summary_json_path}")


# =========================
# 13. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

shape_ok = (
    tr_features.shape == (STORY_N_TR, embeddings.shape[1]) and
    tr_features_train.shape == (TRAIN_TRS, embeddings.shape[1]) and
    tr_features_test.shape == (STORY_N_TR - TRAIN_TRS, embeddings.shape[1])
)
finite_ok = (
    np.isfinite(tr_features).all() and
    np.isfinite(tr_features_train).all() and
    np.isfinite(tr_features_test).all() and
    np.isfinite(lanczos_matrix).all()
)
split_ok = (
    len(tr_time_table) == STORY_N_TR and
    (tr_time_table["split"].value_counts().to_dict().get("train", 0) == TRAIN_TRS) and
    (tr_time_table["split"].value_counts().to_dict().get("test", 0) == STORY_N_TR - TRAIN_TRS)
)
timing_ok = (
    np.all(np.diff(TR_ONSET_TIMES) > 0) and
    np.all(np.diff(TR_MIDPOINT_TIMES) > 0)
)

print(f"shape_ok: {shape_ok}")
print(f"finite_ok: {finite_ok}")
print(f"split_ok: {split_ok}")
print(f"timing_ok: {timing_ok}")
print(f"tr_features shape: {tr_features.shape}")
print(f"tr_features_train shape: {tr_features_train.shape}")
print(f"tr_features_test shape: {tr_features_test.shape}")
print(f"lanczos_matrix shape: {lanczos_matrix.shape}")
print(f"elapsed_minutes: {elapsed_minutes:.2f}")

if not shape_ok:
    raise ValueError("Output TR feature shapes are incorrect.")
if not finite_ok:
    raise ValueError("Non-finite values detected in TR features or Lanczos matrix.")
if not split_ok:
    raise ValueError("TR split table is inconsistent with expected train/test counts.")
if not timing_ok:
    raise ValueError("TR timing arrays are not strictly increasing.")

print("\nStep 4 completed successfully.")


step5_build_delayed_design_matrix

FIR/delay

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"

# New UBSN ROI encoding project
PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP4_DIR = PROJECT_DIR / "step4_downsample_to_tr_lanczos"
STEP5_DIR = PROJECT_DIR / "step5_build_delayed_design_matrix"

CSV_DIR = STEP5_DIR / "csv"
NPY_DIR = STEP5_DIR / "npy"
JSON_DIR = STEP5_DIR / "json"

for d in [PROJECT_DIR, STEP5_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Project base directory: {BASE_DIR}")
print(f"ROI name: {ROI_NAME}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Step 4 directory: {STEP4_DIR}")
print(f"Step 5 directory: {STEP5_DIR}")


# =========================
# 2. Input settings
# =========================

X_ALL_PATH = STEP4_DIR / "npy" / "tr_features_pca50_lanczos_all.npy"
X_TRAIN_PATH_IN = STEP4_DIR / "npy" / "tr_features_pca50_lanczos_train.npy"
X_TEST_PATH_IN = STEP4_DIR / "npy" / "tr_features_pca50_lanczos_test.npy"
STEP4_SUMMARY_PATH = STEP4_DIR / "json" / "step4_lanczos_summary.json"

TRIM = 5
N_DELAYS = 4

for p in [X_ALL_PATH, X_TRAIN_PATH_IN, X_TEST_PATH_IN, STEP4_SUMMARY_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

print(f"Input all path: {X_ALL_PATH}")
print(f"Input train path: {X_TRAIN_PATH_IN}")
print(f"Input test path: {X_TEST_PATH_IN}")
print(f"TRIM = {TRIM}")
print(f"N_DELAYS = {N_DELAYS}")


# =========================
# 3. Load step4 outputs
# =========================

X_all = np.load(X_ALL_PATH).astype(np.float32)
X_train_story = np.load(X_TRAIN_PATH_IN).astype(np.float32)
X_test_story = np.load(X_TEST_PATH_IN).astype(np.float32)

with open(STEP4_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step4_summary = json.load(f)

print("\nLoaded matrices:")
print("X_all shape:", X_all.shape)
print("X_train_story shape:", X_train_story.shape)
print("X_test_story shape:", X_test_story.shape)

if X_all.shape[0] != (X_train_story.shape[0] + X_test_story.shape[0]):
    raise ValueError(
        f"Mismatch: X_all rows={X_all.shape[0]} but train+test={X_train_story.shape[0] + X_test_story.shape[0]}"
    )

feature_dim = int(X_all.shape[1])
print("feature_dim =", feature_dim)


# =========================
# 4. Huth-style helper functions
# =========================

def zs(v: np.ndarray) -> np.ndarray:
    """
    Huth-style column-wise z-score.
    """
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def make_delayed(stim: np.ndarray, delays) -> np.ndarray:
    """
    Positive delays shift the stimulus forward in time,
    then concatenate delayed copies column-wise.
    """
    stim = np.asarray(stim, dtype=np.float32)
    nt, ndim = stim.shape

    delayed_list = []
    for d in delays:
        dstim = np.zeros((nt, ndim), dtype=np.float32)
        if d < 0:
            dstim[:d, :] = stim[-d:, :]
        elif d > 0:
            dstim[d:, :] = stim[:-d, :]
        else:
            dstim[:, :] = stim
        delayed_list.append(dstim)

    return np.hstack(delayed_list).astype(np.float32)


# =========================
# 5. Apply Huth-style trim and z-score
# =========================

# Original Huth-inspired logic:
#   stim = zscore(downsampled_feat[s][5+trim:-trim])
X_train_trimmed = X_train_story[5 + TRIM : -TRIM].astype(np.float32)
X_test_trimmed = X_test_story[5 + TRIM : -TRIM].astype(np.float32)

print("\nAfter Huth-style trim:")
print("X_train_trimmed shape:", X_train_trimmed.shape)
print("X_test_trimmed shape:", X_test_trimmed.shape)

if X_train_trimmed.shape[0] <= 0 or X_test_trimmed.shape[0] <= 0:
    raise ValueError("Trimmed train/test matrices have non-positive time dimension.")

X_train_trimmed = zs(X_train_trimmed)
X_test_trimmed = zs(X_test_trimmed)

print("\nAfter column-wise z-score:")
print("X_train_trimmed mean(abs):", float(np.mean(np.abs(X_train_trimmed))))
print("X_test_trimmed mean(abs):", float(np.mean(np.abs(X_test_trimmed))))


# =========================
# 6. Build delayed design matrices
# =========================

delays = range(1, N_DELAYS + 1)

X_train_delayed = make_delayed(X_train_trimmed, delays)
X_test_delayed = make_delayed(X_test_trimmed, delays)

print("\nDelayed design matrices:")
print("X_train_delayed shape:", X_train_delayed.shape)
print("X_test_delayed shape:", X_test_delayed.shape)

expected_dim = feature_dim * N_DELAYS
if X_train_delayed.shape[1] != expected_dim:
    raise ValueError(
        f"Unexpected train delayed dim: {X_train_delayed.shape[1]} vs expected {expected_dim}"
    )
if X_test_delayed.shape[1] != expected_dim:
    raise ValueError(
        f"Unexpected test delayed dim: {X_test_delayed.shape[1]} vs expected {expected_dim}"
    )


# =========================
# 7. Save delayed design matrices
# =========================

X_TRAIN_OUT = NPY_DIR / "X_train_delayed_lanczos_huthstyle.npy"
X_TEST_OUT = NPY_DIR / "X_test_delayed_lanczos_huthstyle.npy"

np.save(X_TRAIN_OUT, X_train_delayed)
np.save(X_TEST_OUT, X_test_delayed)

print(f"\nSaved train delayed matrix to: {X_TRAIN_OUT}")
print(f"Saved test delayed matrix to: {X_TEST_OUT}")


# =========================
# 8. Save delayed-column manifest
# =========================

column_rows = []
for delay_idx, delay in enumerate(delays, start=1):
    for base_feat_idx in range(feature_dim):
        column_rows.append({
            "delayed_column_index": (delay_idx - 1) * feature_dim + base_feat_idx,
            "base_feature_index": base_feat_idx,
            "delay": int(delay),
        })

columns_df = pd.DataFrame(column_rows)
columns_csv_path = CSV_DIR / "delayed_feature_columns_huthstyle.csv"
columns_df.to_csv(columns_csv_path, index=False, encoding="utf-8-sig")

print(f"Saved delayed feature column manifest to: {columns_csv_path}")


# =========================
# 9. Save summary CSV
# =========================

summary_df = pd.DataFrame({
    "pipeline_type": ["final_layer_only_new_contextual_windows_huthstyle_fir"],
    "input_all_shape": [str(tuple(X_all.shape))],
    "input_train_shape_pretrim": [str(tuple(X_train_story.shape))],
    "input_test_shape_pretrim": [str(tuple(X_test_story.shape))],
    "input_train_shape_posttrim": [str(tuple(X_train_trimmed.shape))],
    "input_test_shape_posttrim": [str(tuple(X_test_trimmed.shape))],
    "output_train_shape": [str(tuple(X_train_delayed.shape))],
    "output_test_shape": [str(tuple(X_test_delayed.shape))],
    "trim": [TRIM],
    "n_delays": [N_DELAYS],
    "n_base_features": [feature_dim],
    "output_feature_dim": [expected_dim],
    "train_mean_abs": [float(np.mean(np.abs(X_train_delayed)))],
    "test_mean_abs": [float(np.mean(np.abs(X_test_delayed)))],
})

summary_csv_path = CSV_DIR / "delayed_design_summary_huthstyle.csv"
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

print(f"Saved delayed design summary to: {summary_csv_path}")


# =========================
# 10. Save JSON summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "final_layer_only_new_contextual_windows_huthstyle_fir",
    "input_source": "Huth-style Lanczos-downsampled TR features",
    "trim": int(TRIM),
    "n_delays": int(N_DELAYS),
    "delay_definition": "range(1, ndelays+1)",
    "n_base_features": int(feature_dim),
    "output_feature_dim": int(expected_dim),

    "input_all_shape": list(X_all.shape),
    "input_train_shape_pretrim": list(X_train_story.shape),
    "input_test_shape_pretrim": list(X_test_story.shape),
    "input_train_shape_posttrim": list(X_train_trimmed.shape),
    "input_test_shape_posttrim": list(X_test_trimmed.shape),
    "output_train_shape": list(X_train_delayed.shape),
    "output_test_shape": list(X_test_delayed.shape),

    "original_huth_inspirations": {
        "encoding_utils_apply_zscore_and_hrf": [
            "stim = [zscore(downsampled_feat[s][5+trim:-trim]) for s in stories]",
            "delays = range(1, ndelays+1)",
            "delstim = make_delayed(stim, delays)"
        ],
        "ridge_utils_npp_zscore": "z-score each feature column",
        "ridge_utils_utils_make_delayed": "positive delays shift stimulus forward in time and hstack delayed copies"
    },

    "outputs": {
        "train_delayed_path": str(X_TRAIN_OUT),
        "test_delayed_path": str(X_TEST_OUT),
        "columns_csv": str(columns_csv_path),
        "summary_csv": str(summary_csv_path),
    },
}

summary_json_path = JSON_DIR / "step5_summary_huthstyle.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved summary to: {summary_json_path}")


# =========================
# 11. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

shape_ok = (
    X_train_delayed.shape[1] == expected_dim and
    X_test_delayed.shape[1] == expected_dim
)
finite_ok = (
    np.isfinite(X_train_delayed).all() and
    np.isfinite(X_test_delayed).all()
)
trim_ok = (
    X_train_trimmed.shape[0] == X_train_story.shape[0] - (5 + 2 * TRIM) and
    X_test_trimmed.shape[0] == X_test_story.shape[0] - (5 + 2 * TRIM)
)

print(f"shape_ok: {shape_ok}")
print(f"finite_ok: {finite_ok}")
print(f"trim_ok: {trim_ok}")
print(f"X_train_trimmed shape: {X_train_trimmed.shape}")
print(f"X_test_trimmed shape: {X_test_trimmed.shape}")
print(f"X_train_delayed shape: {X_train_delayed.shape}")
print(f"X_test_delayed shape: {X_test_delayed.shape}")
print(f"expected_dim: {expected_dim}")

if not shape_ok:
    raise ValueError("Delayed design matrices have incorrect output dimensions.")
if not finite_ok:
    raise ValueError("Non-finite values found in delayed design matrices.")
if not trim_ok:
    raise ValueError("Unexpected post-trim time dimension.")

print("\nStep 5 completed successfully.")


ALL previous stpes for model fitting now are finished

You do not have to adjust anything from the Step1-Step5 in this piepiline!!!!!!!!!!!!!!!!!!!!!!!

step6_prepare_fmri_response_multiROI

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

ENC_PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP5_DIR = ENC_PROJECT_DIR / "step5_build_delayed_design_matrix"
STEP6_DIR = ENC_PROJECT_DIR / "step6_prepare_fmri_response"

CSV_DIR = STEP6_DIR / "csv"
NPY_DIR = STEP6_DIR / "npy"
JSON_DIR = STEP6_DIR / "json"

for d in [ENC_PROJECT_DIR, STEP6_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Encoding project directory: {ENC_PROJECT_DIR}")
print(f"Step 5 directory: {STEP5_DIR}")
print(f"Step 6 directory: {STEP6_DIR}")


# =========================
# 2. BrainIAK cleaned inputs
# =========================

SRM_PROJECT_DIR = BASE_DIR / "runs"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

RUN_DIR = SRM_PROJECT_DIR / RUN_NAME
STEP2_BRAINIAK_DIR = RUN_DIR / "step2_split_zscore"
STEP1_BRAINIAK_DIR = RUN_DIR / "step1_setup_roi_cleaned"

train_resp_path = STEP2_BRAINIAK_DIR / "npy" / "train_data.npy"
test_resp_path = STEP2_BRAINIAK_DIR / "npy" / "test_data.npy"
step2_brainiak_config_path = STEP2_BRAINIAK_DIR / "step2_config.json"

for p in [train_resp_path, test_resp_path, step2_brainiak_config_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing BrainIAK input file: {p}")

print(f"BrainIAK train response path: {train_resp_path}")
print(f"BrainIAK test response path: {test_resp_path}")


# =========================
# 3. Encoding Step5 inputs
# =========================

X_train_path = STEP5_DIR / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
X_test_path = STEP5_DIR / "npy" / "X_test_delayed_lanczos_huthstyle.npy"
step5_summary_path = STEP5_DIR / "json" / "step5_summary_huthstyle.json"

for p in [X_train_path, X_test_path, step5_summary_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing Step5 input file: {p}")

print(f"Encoding train design path: {X_train_path}")
print(f"Encoding test design path: {X_test_path}")


# =========================
# 4. Load data
# =========================

Y_train_list = np.load(train_resp_path, allow_pickle=True)
Y_test_list = np.load(test_resp_path, allow_pickle=True)

Y_train_list = [np.asarray(x, dtype=np.float32) for x in Y_train_list]   # voxels x time
Y_test_list = [np.asarray(x, dtype=np.float32) for x in Y_test_list]     # voxels x time

X_train = np.load(X_train_path).astype(np.float32)   # time x delayed_features
X_test = np.load(X_test_path).astype(np.float32)

with open(step2_brainiak_config_path, "r", encoding="utf-8") as f:
    step2_brainiak_config = json.load(f)

with open(step5_summary_path, "r", encoding="utf-8") as f:
    step5_summary = json.load(f)

print("\nLoaded inputs:")
print(f"Number of subjects = {len(Y_train_list)}")
print(f"Y_train subject 1 shape (voxels x time) = {Y_train_list[0].shape}")
print(f"Y_test subject 1 shape (voxels x time) = {Y_test_list[0].shape}")
print(f"X_train shape = {X_train.shape}")
print(f"X_test shape = {X_test.shape}")

if len(Y_train_list) == 0 or len(Y_test_list) == 0:
    raise ValueError("Empty BrainIAK response list.")
if len(Y_train_list) != len(Y_test_list):
    raise ValueError("Mismatch in number of train/test response subjects.")
if not all(y.shape == Y_train_list[0].shape for y in Y_train_list):
    raise ValueError("Inconsistent Y_train subject shapes.")
if not all(y.shape == Y_test_list[0].shape for y in Y_test_list):
    raise ValueError("Inconsistent Y_test subject shapes.")
if not all(np.isfinite(y).all() for y in Y_train_list):
    raise ValueError("Non-finite values found in Y_train_list.")
if not all(np.isfinite(y).all() for y in Y_test_list):
    raise ValueError("Non-finite values found in Y_test_list.")


# =========================
# 5. Huth-style trim settings
# =========================

TRIM = int(step5_summary["trim"])

print(f"\nRecovered TRIM from Step5 summary: {TRIM}")

# BrainIAK Step2 outputs are [voxels x time], where time=1113 for train/test
# Huth-style response trim should mirror stimulus side:
#   Y = Y[:, 5+trim : -trim]
# because Step5 used X_story[5+trim : -trim]
#
# Since responses are [voxels x time], trim on axis=1.
def trim_response_huthstyle(Y_vox_time: np.ndarray, trim: int) -> np.ndarray:
    return Y_vox_time[:, 5 + trim : -trim].astype(np.float32)


# =========================
# 6. Apply trim to each subject
# =========================

Y_train_trimmed_list = [trim_response_huthstyle(y, TRIM) for y in Y_train_list]
Y_test_trimmed_list = [trim_response_huthstyle(y, TRIM) for y in Y_test_list]

print("\nAfter Huth-style response trim:")
print(f"Y_train_trimmed subject 1 shape = {Y_train_trimmed_list[0].shape}")
print(f"Y_test_trimmed subject 1 shape = {Y_test_trimmed_list[0].shape}")

if not all(y.shape == Y_train_trimmed_list[0].shape for y in Y_train_trimmed_list):
    raise ValueError("Inconsistent trimmed Y_train shapes across subjects.")
if not all(y.shape == Y_test_trimmed_list[0].shape for y in Y_test_trimmed_list):
    raise ValueError("Inconsistent trimmed Y_test shapes across subjects.")


# =========================
# 7. Convert to encoding-friendly layout
# =========================
# Original BrainIAK response is [voxels x time]
# Encoding expects [time x voxels]

Y_train_time_vox_list = [y.T.astype(np.float32) for y in Y_train_trimmed_list]
Y_test_time_vox_list = [y.T.astype(np.float32) for y in Y_test_trimmed_list]

print("\nConverted to time x voxels:")
print(f"Y_train_time_vox subject 1 shape = {Y_train_time_vox_list[0].shape}")
print(f"Y_test_time_vox subject 1 shape = {Y_test_time_vox_list[0].shape}")


# =========================
# 8. Build group-average responses
# =========================

Y_train_group = np.mean(np.stack(Y_train_time_vox_list, axis=0), axis=0).astype(np.float32)
Y_test_group = np.mean(np.stack(Y_test_time_vox_list, axis=0), axis=0).astype(np.float32)

print("\nGroup-average responses:")
print(f"Y_train_group shape = {Y_train_group.shape}")
print(f"Y_test_group shape = {Y_test_group.shape}")


# =========================
# 9. Critical alignment checks
# =========================

print("\nCritical alignment checks:")
print(f"X_train timepoints = {X_train.shape[0]}")
print(f"Y_train_group timepoints = {Y_train_group.shape[0]}")
print(f"X_test timepoints = {X_test.shape[0]}")
print(f"Y_test_group timepoints = {Y_test_group.shape[0]}")

if X_train.shape[0] != Y_train_group.shape[0]:
    raise ValueError(
        f"Train time mismatch: X_train has {X_train.shape[0]} timepoints, "
        f"but Y_train_group has {Y_train_group.shape[0]}."
    )
if X_test.shape[0] != Y_test_group.shape[0]:
    raise ValueError(
        f"Test time mismatch: X_test has {X_test.shape[0]} timepoints, "
        f"but Y_test_group has {Y_test_group.shape[0]}."
    )


# =========================
# 10. Save outputs
# =========================

Y_train_group_path = NPY_DIR / "Y_train_group_huthstyle.npy"
Y_test_group_path = NPY_DIR / "Y_test_group_huthstyle.npy"
Y_train_list_path = NPY_DIR / "Y_train_list_huthstyle.npy"
Y_test_list_path = NPY_DIR / "Y_test_list_huthstyle.npy"

np.save(Y_train_group_path, Y_train_group)
np.save(Y_test_group_path, Y_test_group)
np.save(Y_train_list_path, np.array(Y_train_time_vox_list, dtype=object), allow_pickle=True)
np.save(Y_test_list_path, np.array(Y_test_time_vox_list, dtype=object), allow_pickle=True)

print(f"\nSaved Y_train_group to: {Y_train_group_path}")
print(f"Saved Y_test_group to: {Y_test_group_path}")
print(f"Saved Y_train_list to: {Y_train_list_path}")
print(f"Saved Y_test_list to: {Y_test_list_path}")


# =========================
# 11. Save shape table
# =========================

shape_rows = []
for i, (yt, ys) in enumerate(zip(Y_train_time_vox_list, Y_test_time_vox_list)):
    shape_rows.append({
        "subject_index": i,
        "train_shape": str(tuple(yt.shape)),
        "test_shape": str(tuple(ys.shape)),
        "n_timepoints_train": int(yt.shape[0]),
        "n_voxels": int(yt.shape[1]),
        "n_timepoints_test": int(ys.shape[0]),
    })

shape_df = pd.DataFrame(shape_rows)
shape_csv_path = CSV_DIR / "subject_response_shapes_huthstyle.csv"
shape_df.to_csv(shape_csv_path, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame({
    "n_subjects": [len(Y_train_time_vox_list)],
    "trim": [TRIM],
    "train_group_shape": [str(tuple(Y_train_group.shape))],
    "test_group_shape": [str(tuple(Y_test_group.shape))],
    "X_train_shape": [str(tuple(X_train.shape))],
    "X_test_shape": [str(tuple(X_test.shape))],
    "alignment_train_ok": [bool(X_train.shape[0] == Y_train_group.shape[0])],
    "alignment_test_ok": [bool(X_test.shape[0] == Y_test_group.shape[0])],
})

summary_csv_path = CSV_DIR / "response_preparation_summary_huthstyle.csv"
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

print(f"Saved shape table to: {shape_csv_path}")
print(f"Saved summary table to: {summary_csv_path}")


# =========================
# 12. Save JSON summary
# =========================

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "prepare_fmri_response_huthstyle_cleaned",
    "source_brainiak_run": RUN_NAME,
    "source_brainiak_step2_config": str(step2_brainiak_config_path),
    "source_step5_summary": str(step5_summary_path),

    "trim": int(TRIM),
    "brainiak_input_layout": "voxels x time",
    "encoding_output_layout": "time x voxels",

    "n_subjects": int(len(Y_train_time_vox_list)),
    "train_subject0_shape_before_trim": list(Y_train_list[0].shape),
    "test_subject0_shape_before_trim": list(Y_test_list[0].shape),
    "train_subject0_shape_after_trim": list(Y_train_trimmed_list[0].shape),
    "test_subject0_shape_after_trim": list(Y_test_trimmed_list[0].shape),
    "train_subject0_shape_time_vox": list(Y_train_time_vox_list[0].shape),
    "test_subject0_shape_time_vox": list(Y_test_time_vox_list[0].shape),

    "Y_train_group_shape": list(Y_train_group.shape),
    "Y_test_group_shape": list(Y_test_group.shape),
    "X_train_shape": list(X_train.shape),
    "X_test_shape": list(X_test.shape),

    "alignment": {
        "train_ok": bool(X_train.shape[0] == Y_train_group.shape[0]),
        "test_ok": bool(X_test.shape[0] == Y_test_group.shape[0]),
    },

    "outputs": {
        "Y_train_group": str(Y_train_group_path),
        "Y_test_group": str(Y_test_group_path),
        "Y_train_list": str(Y_train_list_path),
        "Y_test_list": str(Y_test_list_path),
        "shape_csv": str(shape_csv_path),
        "summary_csv": str(summary_csv_path),
    },
}

summary_json_path = JSON_DIR / "step6_summary_huthstyle.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved summary to: {summary_json_path}")


# =========================
# 13. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

shape_ok = (
    all(y.shape == Y_train_time_vox_list[0].shape for y in Y_train_time_vox_list) and
    all(y.shape == Y_test_time_vox_list[0].shape for y in Y_test_time_vox_list)
)
finite_ok = (
    all(np.isfinite(y).all() for y in Y_train_time_vox_list) and
    all(np.isfinite(y).all() for y in Y_test_time_vox_list) and
    np.isfinite(Y_train_group).all() and
    np.isfinite(Y_test_group).all()
)
alignment_ok = (
    X_train.shape[0] == Y_train_group.shape[0] and
    X_test.shape[0] == Y_test_group.shape[0]
)

print(f"shape_ok: {shape_ok}")
print(f"finite_ok: {finite_ok}")
print(f"alignment_ok: {alignment_ok}")
print(f"Y_train_group shape: {Y_train_group.shape}")
print(f"Y_test_group shape: {Y_test_group.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

if not shape_ok:
    raise ValueError("Subject response shapes are inconsistent after trimming/transposing.")
if not finite_ok:
    raise ValueError("Non-finite values detected in prepared fMRI responses.")
if not alignment_ok:
    raise ValueError("Stimulus and response time dimensions do not align.")

print("\nStep 6 completed successfully.")


step7_fit_encoding_model_huthstyle_multiROI

step7a this is the very previous step for "alpha" selection!!!!

In [ ]:
from pathlib import Path
import json
import random
import time
import itertools as itools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

ENC_PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP5_DIR = ENC_PROJECT_DIR / "step5_build_delayed_design_matrix"
STEP6_DIR = ENC_PROJECT_DIR / "step6_prepare_fmri_response"
STEP7A_DIR = ENC_PROJECT_DIR / "step7a_select_alpha_huthstyle"

FIG_DIR = STEP7A_DIR / "figures"
CSV_DIR = STEP7A_DIR / "csv"
NPY_DIR = STEP7A_DIR / "npy"
JSON_DIR = STEP7A_DIR / "json"

for d in [STEP7A_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Encoding project directory: {ENC_PROJECT_DIR}")
print(f"Step 5 directory: {STEP5_DIR}")
print(f"Step 6 directory: {STEP6_DIR}")
print(f"Step 7a directory: {STEP7A_DIR}")


# =========================
# 2. Huth-style alpha selection settings
# =========================

ALPHAS = np.logspace(1, 5, 20).astype(np.float32)
NBOOTS = 15
CHUNKLEN = 40
NCHUNKS = 6
SINGCUTOFF = 1e-10
USE_CORR = True
SEED = 0
SINGLE_ALPHA = True

print("\nHuth-style alpha-selection parameters:")
print("ALPHAS =", ALPHAS)
print("NBOOTS =", NBOOTS)
print("CHUNKLEN =", CHUNKLEN)
print("NCHUNKS =", NCHUNKS)
print("SINGCUTOFF =", SINGCUTOFF)
print("USE_CORR =", USE_CORR)
print("SINGLE_ALPHA =", SINGLE_ALPHA)
print("SEED =", SEED)


# =========================
# 3. Load Huth-style X and Y
# =========================

X_TRAIN_PATH = STEP5_DIR / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
X_TEST_PATH = STEP5_DIR / "npy" / "X_test_delayed_lanczos_huthstyle.npy"

Y_TRAIN_PATH = STEP6_DIR / "npy" / "Y_train_group_huthstyle.npy"
Y_TEST_PATH = STEP6_DIR / "npy" / "Y_test_group_huthstyle.npy"

required_paths = [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_PATH, Y_TEST_PATH]
for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input file: {p}")

X_train = np.load(X_TRAIN_PATH).astype(np.float32)
X_test = np.load(X_TEST_PATH).astype(np.float32)
Y_train = np.load(Y_TRAIN_PATH).astype(np.float32)
Y_test = np.load(Y_TEST_PATH).astype(np.float32)

print("\nLoaded matrices:")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

if X_train.shape[0] != Y_train.shape[0]:
    raise ValueError("X_train and Y_train mismatch.")
if X_test.shape[0] != Y_test.shape[0]:
    raise ValueError("X_test and Y_test mismatch.")
if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
    raise ValueError("Non-finite values found in X matrices.")
if not np.isfinite(Y_train).all() or not np.isfinite(Y_test).all():
    raise ValueError("Non-finite values found in Y matrices.")


# =========================
# 4. Huth-inspired helper functions
# =========================

def zs(v: np.ndarray) -> np.ndarray:
    """
    Huth-style column-wise z-score.
    """
    v = np.asarray(v, dtype=np.float64)
    s = v.std(axis=0, ddof=0)
    m = v - v.mean(axis=0, keepdims=True)
    for i in range(len(s)):
        if s[i] != 0.0:
            m[:, i] /= s[i]
    return np.nan_to_num(m).astype(np.float32)


def columnwise_corr(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    Columnwise Pearson r using Huth-style z-scoring.
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return np.nan_to_num((zs(y_true) * zs(y_pred)).mean(axis=0)).astype(np.float32)


def ridge_fit_predict_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    alpha: float,
    singcutoff: float = 1e-10,
):
    """
    Huth-style SVD ridge fit + predict.
    """
    Rstim = np.asarray(Rstim, dtype=np.float64)
    Pstim = np.asarray(Pstim, dtype=np.float64)
    Rresp = np.asarray(Rresp, dtype=np.float64)

    U, S, Vh = np.linalg.svd(Rstim, full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))

    U = U[:, :ngoodS]
    S = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR = np.dot(U.T, np.nan_to_num(Rresp))
    wt = Vh.T.dot(np.diag(S / (S**2 + float(alpha)**2))).dot(UR)
    pred = np.dot(Pstim, wt).astype(np.float32)

    return wt.astype(np.float32), pred


def ridge_corr_huthstyle(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    Presp: np.ndarray,
    alphas: np.ndarray,
    singcutoff: float = 1e-10,
    use_corr: bool = True,
) -> np.ndarray:
    """
    Evaluate each alpha on held-out validation data.

    Returns
    -------
    scores : [n_alphas, n_targets]
    """
    scores = []

    for alpha in alphas:
        _, pred = ridge_fit_predict_huthstyle(
            Rstim=Rstim,
            Pstim=Pstim,
            Rresp=Rresp,
            alpha=float(alpha),
            singcutoff=singcutoff,
        )

        if use_corr:
            sc = columnwise_corr(Presp, pred)
        else:
            # kept for branch compatibility
            ss_res = np.sum((Presp - pred) ** 2, axis=0)
            ss_tot = np.sum((Presp - Presp.mean(axis=0, keepdims=True)) ** 2, axis=0)
            r2 = 1.0 - ss_res / np.maximum(ss_tot, 1e-8)
            sc = np.sign(r2) * np.sqrt(np.abs(r2))

        scores.append(sc.astype(np.float32))

    return np.stack(scores, axis=0).astype(np.float32)


def make_bootstrap_chunk_splits(
    n_timepoints: int,
    chunklen: int,
    nchunks: int,
    nboots: int,
    seed: int = 0,
):
    """
    Huth-style bootstrap chunk sampling.
    """
    rng = random.Random(seed)

    allinds = list(range(n_timepoints))
    indchunks = list(zip(*[iter(allinds)] * chunklen))

    if len(indchunks) < nchunks:
        raise ValueError(
            f"Not enough chunks for bootstrap: len(indchunks)={len(indchunks)}, nchunks={nchunks}"
        )

    valinds = []
    for _ in range(nboots):
        heldchunks = rng.sample(indchunks, nchunks)
        heldinds = np.array(sorted(itools.chain(*heldchunks)), dtype=int)
        valinds.append(heldinds)

    return valinds


# =========================
# 5. Build validation chunk splits
# =========================

valinds = make_bootstrap_chunk_splits(
    n_timepoints=X_train.shape[0],
    chunklen=CHUNKLEN,
    nchunks=NCHUNKS,
    nboots=NBOOTS,
    seed=SEED,
)

print("\nBootstrap validation setup:")
print(f"Number of bootstrap splits: {len(valinds)}")
print(f"Validation chunk size per bootstrap: {len(valinds[0])}")


# =========================
# 6. Huth-style alpha search
# =========================

start_time = time.time()

Rcmats = []
alpha_rows = []

for bi, heldinds in enumerate(valinds):
    print(f"\nBootstrap {bi+1}/{NBOOTS}")

    notheldinds = np.array(
        sorted(list(set(range(X_train.shape[0])) - set(heldinds))),
        dtype=int
    )

    RRstim = X_train[notheldinds, :]
    PRstim = X_train[heldinds, :]
    RRresp = Y_train[notheldinds, :]
    PRresp = Y_train[heldinds, :]

    Rcmat = ridge_corr_huthstyle(
        RRstim,
        PRstim,
        RRresp,
        PRresp,
        alphas=ALPHAS,
        singcutoff=SINGCUTOFF,
        use_corr=USE_CORR,
    )

    Rcmats.append(Rcmat)

    mean_across_voxels = Rcmat.mean(axis=1)
    for ai, alpha in enumerate(ALPHAS):
        alpha_rows.append({
            "bootstrap_index": bi,
            "alpha": float(alpha),
            "mean_corr_across_voxels": float(mean_across_voxels[ai]),
        })

alpha_bootstrap_df = pd.DataFrame(alpha_rows)
alpha_bootstrap_df.to_csv(CSV_DIR / "alpha_bootstrap_details_huthstyle.csv", index=False)

# Stack bootstrap results: [n_alphas, n_voxels, nboots]
allRcorrs = np.dstack(Rcmats).astype(np.float32)
np.save(NPY_DIR / "all_bootstrap_corrs_huthstyle.npy", allRcorrs)

print("\nallRcorrs shape:", allRcorrs.shape)


# =========================
# 7. Select best alpha (single_alpha=True style)
# =========================

meanbootcorr = allRcorrs.mean(axis=2).mean(axis=1)
stdb_bootcorr = allRcorrs.mean(axis=1).std(axis=1)

bestalphaind = int(np.argmax(meanbootcorr))
BEST_ALPHA = float(ALPHAS[bestalphaind])

alpha_summary_df = pd.DataFrame({
    "alpha": ALPHAS.astype(float),
    "bootstrap_mean_corr_across_voxels": meanbootcorr.astype(float),
    "bootstrap_std_corr_across_voxels": stdb_bootcorr.astype(float),
    "is_best_alpha": [i == bestalphaind for i in range(len(ALPHAS))],
})
alpha_summary_df.to_csv(CSV_DIR / "alpha_selection_summary_huthstyle.csv", index=False)

print("\nHuth-style single-alpha selection:")
print(alpha_summary_df.to_string(index=False))
print(f"\nBest alpha selected: {BEST_ALPHA}")


# =========================
# 8. Final full-train sanity check on held-out test
# =========================

wt, Y_test_pred = ridge_fit_predict_huthstyle(
    Rstim=X_train,
    Pstim=X_test,
    Rresp=Y_train,
    alpha=BEST_ALPHA,
    singcutoff=SINGCUTOFF,
)

voxel_corrs = columnwise_corr(Y_test, Y_test_pred)

mean_r = float(np.mean(voxel_corrs))
median_r = float(np.median(voxel_corrs))
p90_r = float(np.percentile(voxel_corrs, 90))
p95_r = float(np.percentile(voxel_corrs, 95))
positive_fraction = float(np.mean(voxel_corrs > 0))

print("\nHeld-out test sanity-check performance summary:")
print("mean_r =", mean_r)
print("median_r =", median_r)
print("p90_r =", p90_r)
print("p95_r =", p95_r)
print("positive_fraction =", positive_fraction)


# =========================
# 9. Save outputs
# =========================

np.save(NPY_DIR / "weights_huthstyle.npy", wt)
np.save(NPY_DIR / "Y_test_pred_huthstyle.npy", Y_test_pred)
np.save(NPY_DIR / "voxel_corrs_huthstyle.npy", voxel_corrs)
np.save(NPY_DIR / "best_alpha_huthstyle.npy", np.array([BEST_ALPHA], dtype=np.float32))


# =========================
# 10. Save summary JSON
# =========================

elapsed_minutes = (time.time() - start_time) / 60.0

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "step7a_huthstyle_single_alpha_selection_cleaned",

    "x_train_shape": list(X_train.shape),
    "x_test_shape": list(X_test.shape),
    "y_train_shape": list(Y_train.shape),
    "y_test_shape": list(Y_test.shape),

    "alphas": [float(a) for a in ALPHAS],
    "nboots": int(NBOOTS),
    "chunklen": int(CHUNKLEN),
    "nchunks": int(NCHUNKS),
    "singcutoff": float(SINGCUTOFF),
    "use_corr": bool(USE_CORR),
    "single_alpha": bool(SINGLE_ALPHA),
    "seed": int(SEED),

    "best_alpha": float(BEST_ALPHA),

    "mean_r": mean_r,
    "median_r": median_r,
    "p90_r": p90_r,
    "p95_r": p95_r,
    "positive_fraction": positive_fraction,

    "elapsed_minutes": float(elapsed_minutes),

    "outputs": {
        "all_bootstrap_corrs": str(NPY_DIR / "all_bootstrap_corrs_huthstyle.npy"),
        "best_alpha": str(NPY_DIR / "best_alpha_huthstyle.npy"),
        "weights": str(NPY_DIR / "weights_huthstyle.npy"),
        "y_test_pred": str(NPY_DIR / "Y_test_pred_huthstyle.npy"),
        "voxel_corrs": str(NPY_DIR / "voxel_corrs_huthstyle.npy"),
        "alpha_bootstrap_details_csv": str(CSV_DIR / "alpha_bootstrap_details_huthstyle.csv"),
        "alpha_selection_summary_csv": str(CSV_DIR / "alpha_selection_summary_huthstyle.csv"),
    },
}

with open(JSON_DIR / "step7a_summary_huthstyle.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved summary JSON to: {JSON_DIR / 'step7a_summary_huthstyle.json'}")


# =========================
# 11. Quick figure
# =========================

plt.figure(figsize=(8, 5))
plt.plot(ALPHAS, meanbootcorr, marker="o")
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("bootstrap mean corr across voxels")
plt.title("Alpha selection summary")
plt.tight_layout()
plt.savefig(FIG_DIR / "alpha_selection_curve_huthstyle.png", dpi=200)
plt.close()


# =========================
# 12. Step-end validation checks
# =========================

print("\n" + "=" * 80)
print("STEP-END VALIDATION CHECKS")
print("=" * 80)

allRcorrs_shape_ok = (
    allRcorrs.shape[0] == len(ALPHAS) and
    allRcorrs.shape[1] == Y_train.shape[1] and
    allRcorrs.shape[2] == NBOOTS
)
finite_ok = (
    np.isfinite(allRcorrs).all() and
    np.isfinite(meanbootcorr).all() and
    np.isfinite(voxel_corrs).all() and
    np.isfinite(Y_test_pred).all()
)
best_alpha_ok = BEST_ALPHA in [float(a) for a in ALPHAS]

print(f"allRcorrs_shape_ok: {allRcorrs_shape_ok}")
print(f"finite_ok: {finite_ok}")
print(f"best_alpha_ok: {best_alpha_ok}")
print(f"BEST_ALPHA: {BEST_ALPHA}")
print(f"weights shape: {wt.shape}")
print(f"Y_test_pred shape: {Y_test_pred.shape}")
print(f"mean_r: {mean_r:.6f}")
print(f"median_r: {median_r:.6f}")
print(f"positive_fraction: {positive_fraction:.6f}")

if not allRcorrs_shape_ok:
    raise ValueError("allRcorrs shape is incorrect.")
if not finite_ok:
    raise ValueError("Non-finite values detected in Step7a outputs.")
if not best_alpha_ok:
    raise ValueError("Selected alpha is not one of the candidate ALPHAS.")

print("\nStep 7a completed successfully.")


We prioritized cross-validated correlation as the main model-selection criterion, while monitoring the predicted-to-observed variance ratio to avoid overly shrunk solutions with implausibly flat predicted time courses.

STEP7B POSSIBLE

In [ ]:
from pathlib import Path
import json
import time
import itertools as itools
import random

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import zscore as scipy_zscore


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

ENC_PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP5_DIR = ENC_PROJECT_DIR / "step5_build_delayed_design_matrix"
STEP6_DIR = ENC_PROJECT_DIR / "step6_prepare_fmri_response"
STEP7A_DIR = ENC_PROJECT_DIR / "step7a_select_alpha_huthstyle"
STEP7B_DIR = ENC_PROJECT_DIR / "step7b_raw_bold_encoding"

FIG_DIR = STEP7B_DIR / "figures"
CSV_DIR = STEP7B_DIR / "csv"
NPY_DIR = STEP7B_DIR / "npy"
JSON_DIR = STEP7B_DIR / "json"

for d in [STEP7B_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Encoding project directory: {ENC_PROJECT_DIR}")
print(f"Step 7b directory: {STEP7B_DIR}")


# =========================
# 2. Load inputs
# =========================

X_TRAIN_PATH = STEP5_DIR / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
X_TEST_PATH  = STEP5_DIR / "npy" / "X_test_delayed_lanczos_huthstyle.npy"

# [IMPORTANT] Load per-subject response lists (time x voxels), not group average.
# Huth-style: fit a separate encoding model for each subject individually.
Y_TRAIN_LIST_PATH = STEP6_DIR / "npy" / "Y_train_list_huthstyle.npy"
Y_TEST_LIST_PATH  = STEP6_DIR / "npy" / "Y_test_list_huthstyle.npy"

# Also load group-average responses for group-level analysis
Y_TRAIN_GROUP_PATH = STEP6_DIR / "npy" / "Y_train_group_huthstyle.npy"
Y_TEST_GROUP_PATH  = STEP6_DIR / "npy" / "Y_test_group_huthstyle.npy"

STEP7A_SUMMARY_PATH = STEP7A_DIR / "json" / "step7a_summary_huthstyle.json"

for p in [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_LIST_PATH, Y_TEST_LIST_PATH,
          Y_TRAIN_GROUP_PATH, Y_TEST_GROUP_PATH, STEP7A_SUMMARY_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

X_train = np.load(X_TRAIN_PATH).astype(np.float32)
X_test  = np.load(X_TEST_PATH).astype(np.float32)

Y_train_list = [np.asarray(x, dtype=np.float32) for x in np.load(Y_TRAIN_LIST_PATH, allow_pickle=True)]
Y_test_list  = [np.asarray(x, dtype=np.float32) for x in np.load(Y_TEST_LIST_PATH,  allow_pickle=True)]

Y_train_group = np.load(Y_TRAIN_GROUP_PATH).astype(np.float32)
Y_test_group  = np.load(Y_TEST_GROUP_PATH).astype(np.float32)

with open(STEP7A_SUMMARY_PATH, "r", encoding="utf-8") as f:
    step7a_summary = json.load(f)

N_SUBJECTS = len(Y_train_list)
N_VOXELS   = Y_train_list[0].shape[1]

print(f"\nLoaded inputs:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"N_SUBJECTS: {N_SUBJECTS}")
print(f"Y_train subject 0 shape (time x voxels): {Y_train_list[0].shape}")
print(f"Y_test  subject 0 shape (time x voxels): {Y_test_list[0].shape}")
print(f"Y_train_group shape: {Y_train_group.shape}")
print(f"Y_test_group  shape: {Y_test_group.shape}")

if X_train.shape[0] != Y_train_list[0].shape[0]:
    raise ValueError("Train time mismatch between X and Y.")
if X_test.shape[0] != Y_test_list[0].shape[0]:
    raise ValueError("Test time mismatch between X and Y.")


# =========================
# 3. Alpha selection settings
# =========================

# [IMPORTANT - Huth lab] Use a log-spaced alpha grid matching ridge.py convention.
# For individual subjects we use per-voxel alpha (single_alpha=False),
# which is the standard Huth lab approach for subject-level encoding models.
# For group-level analysis we use single_alpha=True (same as Step 7a).
ALPHAS      = np.logspace(1, 5, 20).astype(np.float32)
NBOOTS      = 15
CHUNKLEN    = 40
NCHUNKS     = 6
SINGCUTOFF  = 1e-10
USE_CORR    = True
SEED        = 0

# Re-use the group-level best alpha found in Step 7a for group analysis
GROUP_BEST_ALPHA = float(step7a_summary["best_alpha"])

print(f"\nAlpha grid: {ALPHAS}")
print(f"GROUP_BEST_ALPHA (from Step 7a): {GROUP_BEST_ALPHA}")
print(f"NBOOTS={NBOOTS}, CHUNKLEN={CHUNKLEN}, NCHUNKS={NCHUNKS}")


# =========================
# 4. Core Huth-style ridge functions
# =========================

# [IMPORTANT - Huth lab] z-score helper, column-wise, ddof=0 (population).
# Matches the lambda `zs = lambda v: (v-v.mean(0))/v.std(0)` in ridge.py.
def zs(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float64)
    mu = v.mean(axis=0, keepdims=True)
    sd = v.std(axis=0, ddof=0, keepdims=True)
    sd[sd < 1e-12] = 1.0
    return np.nan_to_num((v - mu) / sd).astype(np.float32)


# [IMPORTANT - Huth lab] SVD-based ridge regression weight computation.
# Directly mirrors the `ridge()` function in ridge.py.
def ridge_fit(
    stim: np.ndarray,
    resp: np.ndarray,
    valphas: np.ndarray,
    singcutoff: float = 1e-10,
) -> np.ndarray:
    """
    Fit ridge regression weights via SVD.

    Parameters
    ----------
    stim    : (T, N) training stimulus
    resp    : (T, M) training responses
    valphas : (M,)   per-voxel regularization parameters
    Returns
    -------
    wt : (N, M) regression weights
    """
    U, S, Vh = np.linalg.svd(stim.astype(np.float64), full_matrices=False)

    ngoodS = int(np.sum(S > singcutoff))
    U  = U[:, :ngoodS]
    S  = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR = U.T @ np.nan_to_num(resp.astype(np.float64))

    ualphas = np.unique(valphas)
    wt = np.zeros((stim.shape[1], resp.shape[1]), dtype=np.float32)

    for ua in ualphas:
        selvox = np.where(valphas == ua)[0]
        D      = S / (S ** 2 + float(ua) ** 2)
        awt    = Vh.T @ np.diag(D) @ UR[:, selvox]
        wt[:, selvox] = awt.astype(np.float32)

    return wt


# [IMPORTANT - Huth lab] Ridge correlation across alphas without storing weights.
# Mirrors `ridge_corr()` in ridge.py for efficient bootstrap alpha selection.
def ridge_corr_alphas(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    Presp: np.ndarray,
    alphas: np.ndarray,
    singcutoff: float = 1e-10,
    use_corr: bool = True,
) -> np.ndarray:
    """
    Evaluate all alphas on held-out validation data without storing weights.

    Returns
    -------
    Rcorrs : (n_alphas, n_voxels)
    """
    U, S, Vh = np.linalg.svd(Rstim.astype(np.float64), full_matrices=False)

    ngoodS = int(np.sum(S > singcutoff))
    U  = U[:, :ngoodS]
    S  = S[:ngoodS]
    Vh = Vh[:ngoodS, :]

    UR   = U.T @ np.nan_to_num(Rresp.astype(np.float64))
    PVh  = Pstim.astype(np.float64) @ Vh.T
    zPre = zs(Presp)

    Rcorrs = []
    for a in alphas:
        D    = S / (S ** 2 + float(a) ** 2)
        pred = (PVh * D[None, :]) @ UR

        if use_corr:
            rc = np.nan_to_num((zPre * zs(pred)).mean(axis=0)).astype(np.float32)
        else:
            resvar = (Presp - pred).var(axis=0)
            Rsq    = 1.0 - resvar / np.maximum(Presp.var(axis=0), 1e-8)
            rc     = (np.sign(Rsq) * np.sqrt(np.abs(Rsq))).astype(np.float32)

        Rcorrs.append(rc)

    return np.stack(Rcorrs, axis=0).astype(np.float32)   # (n_alphas, n_voxels)


# [IMPORTANT - Huth lab] Bootstrap chunk split matching ridge.py exactly.
# indchunks[:nchunks] are held out; the rest are used for training within each boot.
def make_bootstrap_splits(
    n_timepoints: int,
    chunklen: int,
    nchunks: int,
    nboots: int,
    seed: int = 0,
):
    rng = random.Random(seed)
    allinds   = list(range(n_timepoints))
    indchunks = list(zip(*[iter(allinds)] * chunklen))

    if len(indchunks) < nchunks:
        raise ValueError(
            f"Not enough chunks: len(indchunks)={len(indchunks)}, nchunks={nchunks}"
        )

    valinds = []
    for _ in range(nboots):
        held   = rng.sample(indchunks, nchunks)
        heldinds = np.array(sorted(itools.chain(*held)), dtype=int)
        valinds.append(heldinds)
    return valinds


# [IMPORTANT - Huth lab] Per-voxel alpha selection via bootstrap cross-validation.
# Mirrors `bootstrap_ridge(..., single_alpha=False)` in ridge.py.
def select_per_voxel_alpha(
    Rstim: np.ndarray,
    Rresp: np.ndarray,
    alphas: np.ndarray,
    nboots: int,
    chunklen: int,
    nchunks: int,
    singcutoff: float = 1e-10,
    use_corr: bool = True,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Run bootstrap cross-validation to select the best alpha per voxel.

    Returns
    -------
    valphas     : (n_voxels,)  best alpha per voxel
    allRcorrs   : (n_alphas, n_voxels, nboots)
    """
    n_timepoints = Rstim.shape[0]
    valinds      = make_bootstrap_splits(n_timepoints, chunklen, nchunks, nboots, seed)

    Rcmats = []
    for bi, heldinds in enumerate(valinds):
        notheldinds = np.array(
            sorted(set(range(n_timepoints)) - set(heldinds.tolist())), dtype=int
        )

        RRstim = Rstim[notheldinds, :]
        PRstim = Rstim[heldinds,    :]
        RRresp = Rresp[notheldinds, :]
        PRresp = Rresp[heldinds,    :]

        Rcmat = ridge_corr_alphas(
            RRstim, PRstim, RRresp, PRresp,
            alphas=alphas, singcutoff=singcutoff, use_corr=use_corr,
        )
        Rcmats.append(Rcmat)
        print(f"  Bootstrap {bi+1}/{nboots} | mean best corr: {Rcmat.max(axis=0).mean():.5f}")

    # allRcorrs: (n_alphas, n_voxels, nboots)
    allRcorrs = np.stack(Rcmats, axis=2).astype(np.float32)

    # [IMPORTANT - Huth lab] Select best alpha per voxel by averaging across bootstraps.
    # Mirrors ridge.py line: bestalphainds = np.argmax(meanbootcorrs, 0)
    meanbootcorrs = allRcorrs.mean(axis=2)          # (n_alphas, n_voxels)
    bestalphainds = np.argmax(meanbootcorrs, axis=0) # (n_voxels,)
    valphas       = alphas[bestalphainds]             # (n_voxels,)

    return valphas, allRcorrs


# =========================
# 5. Individual-level encoding model fitting
#    Per-subject, per-voxel alpha selection (Huth lab standard)
# =========================

print("\n" + "=" * 70)
print("INDIVIDUAL-LEVEL ENCODING MODEL FITTING (raw BOLD)")
print("=" * 70)

indiv_rows     = []
all_voxel_corrs = []   # list of (n_voxels,) arrays, one per subject

start_total = time.time()

for subj_idx in range(N_SUBJECTS):
    print(f"\n--- Subject {subj_idx+1}/{N_SUBJECTS} ---")

    Y_train_subj = Y_train_list[subj_idx].astype(np.float32)  # (time, voxels)
    Y_test_subj  = Y_test_list[subj_idx].astype(np.float32)

    # [IMPORTANT - Huth lab] Per-voxel alpha selection via bootstrap ridge.
    # This is single_alpha=False mode from ridge.py.
    t0 = time.time()
    valphas, allRcorrs_subj = select_per_voxel_alpha(
        Rstim    = X_train,
        Rresp    = Y_train_subj,
        alphas   = ALPHAS,
        nboots   = NBOOTS,
        chunklen = CHUNKLEN,
        nchunks  = NCHUNKS,
        singcutoff = SINGCUTOFF,
        use_corr   = USE_CORR,
        seed       = SEED + subj_idx,   # different seed per subject for independence
    )

    # Fit final weights on full training set using per-voxel alphas
    wt = ridge_fit(X_train, Y_train_subj, valphas, singcutoff=SINGCUTOFF)

    # Predict on held-out test set
    Y_pred_subj = (X_test.astype(np.float64) @ wt.astype(np.float64)).astype(np.float32)

    # [IMPORTANT - Huth lab] Evaluate with Pearson r (use_corr=True), column-wise.
    # Mirrors ridge.py: corrs = np.array([np.corrcoef(Presp[:,ii], nnpred[:,ii])[0,1] ...])
    voxel_corrs = np.array([
        float(np.corrcoef(Y_test_subj[:, ii], Y_pred_subj[:, ii])[0, 1])
        for ii in range(N_VOXELS)
    ], dtype=np.float32)
    voxel_corrs = np.nan_to_num(voxel_corrs)

    all_voxel_corrs.append(voxel_corrs)

    elapsed = time.time() - t0
    print(f"  mean_r={np.mean(voxel_corrs):.5f} | "
          f"median_r={np.median(voxel_corrs):.5f} | "
          f"p90_r={np.percentile(voxel_corrs, 90):.5f} | "
          f"positive_frac={np.mean(voxel_corrs > 0):.5f} | "
          f"elapsed={elapsed/60:.2f} min")

    # Save per-subject weights and predictions
    subj_npy_dir = NPY_DIR / "individual"
    subj_npy_dir.mkdir(exist_ok=True)
    np.save(subj_npy_dir / f"subj{subj_idx:02d}_weights_raw.npy",      wt)
    np.save(subj_npy_dir / f"subj{subj_idx:02d}_valphas_raw.npy",      valphas)
    np.save(subj_npy_dir / f"subj{subj_idx:02d}_voxel_corrs_raw.npy",  voxel_corrs)
    np.save(subj_npy_dir / f"subj{subj_idx:02d}_Y_pred_raw.npy",       Y_pred_subj)

    indiv_rows.append({
        "subject_index"   : subj_idx,
        "mean_r"          : float(np.mean(voxel_corrs)),
        "median_r"        : float(np.median(voxel_corrs)),
        "std_r"           : float(np.std(voxel_corrs)),
        "p10_r"           : float(np.percentile(voxel_corrs, 10)),
        "p25_r"           : float(np.percentile(voxel_corrs, 25)),
        "p75_r"           : float(np.percentile(voxel_corrs, 75)),
        "p90_r"           : float(np.percentile(voxel_corrs, 90)),
        "p95_r"           : float(np.percentile(voxel_corrs, 95)),
        "positive_frac"   : float(np.mean(voxel_corrs > 0)),
        "elapsed_minutes" : float((time.time() - t0) / 60),
    })

# Stack into array: (n_subjects, n_voxels)
all_voxel_corrs_arr = np.stack(all_voxel_corrs, axis=0).astype(np.float32)
np.save(NPY_DIR / "all_subjects_voxel_corrs_raw.npy", all_voxel_corrs_arr)

indiv_df = pd.DataFrame(indiv_rows)
indiv_df.to_csv(CSV_DIR / "individual_encoding_summary_raw.csv", index=False, encoding="utf-8-sig")

print(f"\nIndividual fitting complete. Total elapsed: {(time.time()-start_total)/60:.2f} min")


# =========================
# 6. Group-level encoding model fitting
#    Single alpha (from Step 7a), fit on group-average BOLD
# =========================

print("\n" + "=" * 70)
print("GROUP-LEVEL ENCODING MODEL FITTING (raw BOLD, group average)")
print("=" * 70)

# [IMPORTANT - Huth lab] For group-level analysis, use single best alpha
# selected in Step 7a (single_alpha=True mode). This ensures fair comparison
# between raw and SRM conditions at the group level.
group_valphas = np.full(N_VOXELS, fill_value=GROUP_BEST_ALPHA, dtype=np.float32)

wt_group = ridge_fit(X_train, Y_train_group, group_valphas, singcutoff=SINGCUTOFF)
Y_pred_group = (X_test.astype(np.float64) @ wt_group.astype(np.float64)).astype(np.float32)

group_voxel_corrs = np.array([
    float(np.corrcoef(Y_test_group[:, ii], Y_pred_group[:, ii])[0, 1])
    for ii in range(N_VOXELS)
], dtype=np.float32)
group_voxel_corrs = np.nan_to_num(group_voxel_corrs)

np.save(NPY_DIR / "group_voxel_corrs_raw.npy",  group_voxel_corrs)
np.save(NPY_DIR / "group_weights_raw.npy",       wt_group)
np.save(NPY_DIR / "group_Y_pred_raw.npy",        Y_pred_group)

group_mean_r    = float(np.mean(group_voxel_corrs))
group_median_r  = float(np.median(group_voxel_corrs))
group_p90_r     = float(np.percentile(group_voxel_corrs, 90))
group_pos_frac  = float(np.mean(group_voxel_corrs > 0))

print(f"Group mean_r   = {group_mean_r:.5f}")
print(f"Group median_r = {group_median_r:.5f}")
print(f"Group p90_r    = {group_p90_r:.5f}")
print(f"Group positive_frac = {group_pos_frac:.5f}")

group_summary_df = pd.DataFrame([{
    "mean_r"       : group_mean_r,
    "median_r"     : group_median_r,
    "std_r"        : float(np.std(group_voxel_corrs)),
    "p90_r"        : group_p90_r,
    "p95_r"        : float(np.percentile(group_voxel_corrs, 95)),
    "positive_frac": group_pos_frac,
    "best_alpha"   : GROUP_BEST_ALPHA,
}])
group_summary_df.to_csv(CSV_DIR / "group_encoding_summary_raw.csv", index=False, encoding="utf-8-sig")


# =========================
# 7. Figures
# =========================

# --- Shared axis limits for consistency across all figures ---
# [IMPORTANT] Unified axis ranges so that plots can be directly compared
# between Step 7b (raw) and Step 7c (SRM) without visual distortion.
CORR_VMIN = -0.20
CORR_VMAX =  0.60
MEAN_R_MIN =  0.00
MEAN_R_MAX =  0.25

subject_indices = np.arange(N_SUBJECTS)
mean_r_per_subj = indiv_df["mean_r"].values

# ---- Figure 1: Per-subject mean encoding r (bar + strip) ----
# Inspired by Bhattacharjee et al. (2025) Fig. 1b style (individual participant bars).
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(subject_indices, mean_r_per_subj, color="#4C72B0", alpha=0.75, label="Individual mean r")
ax.axhline(group_mean_r, color="crimson", linewidth=2.0, linestyle="--",
           label=f"Group mean r = {group_mean_r:.5f}")
ax.set_xlabel("Subject index", fontsize=12)
ax.set_ylabel("Mean encoding r (Pearson)", fontsize=12)
ax.set_title("Raw BOLD encoding: per-subject mean r", fontsize=13)
ax.set_xticks(subject_indices)
ax.set_xticklabels([f"S{i}" for i in subject_indices], fontsize=8, rotation=45)
ax.set_ylim(MEAN_R_MIN, MEAN_R_MAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_per_subject_mean_r_raw.png", dpi=200)
plt.close()
print("Saved fig1_per_subject_mean_r_raw.png")

# ---- Figure 2: Violin plot of voxel-wise corr distribution per subject ----
# Shows full distribution shape, not just mean — allows readers to see spread.
fig, ax = plt.subplots(figsize=(14, 5))
parts = ax.violinplot(
    [all_voxel_corrs[i] for i in range(N_SUBJECTS)],
    positions=subject_indices,
    showmedians=True,
    showextrema=False,
)
for pc in parts["bodies"]:
    pc.set_facecolor("#4C72B0")
    pc.set_alpha(0.6)
parts["cmedians"].set_color("white")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.axhline(group_mean_r, color="crimson", linewidth=1.5, linestyle="--",
           label=f"Group mean r = {group_mean_r:.5f}")
ax.set_xlabel("Subject index", fontsize=12)
ax.set_ylabel("Voxel-wise encoding r", fontsize=12)
ax.set_title("Raw BOLD encoding: voxel-wise r distribution per subject", fontsize=13)
ax.set_xticks(subject_indices)
ax.set_xticklabels([f"S{i}" for i in subject_indices], fontsize=8, rotation=45)
ax.set_ylim(CORR_VMIN, CORR_VMAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_voxelwise_corr_violin_raw.png", dpi=200)
plt.close()
print("Saved fig2_voxelwise_corr_violin_raw.png")

# ---- Figure 3: Group-level voxel-wise corr histogram ----
# Shows the distribution of encoding performance across all voxels
# for the group-average analysis. Useful to communicate SNR level.
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(group_voxel_corrs, bins=60, color="#4C72B0", alpha=0.75, edgecolor="white")
ax.axvline(group_mean_r,   color="crimson", linewidth=2.0, linestyle="--",
           label=f"Mean r = {group_mean_r:.5f}")
ax.axvline(group_median_r, color="orange",  linewidth=2.0, linestyle=":",
           label=f"Median r = {group_median_r:.5f}")
ax.axvline(0, color="gray", linewidth=1.0, linestyle="-")
ax.set_xlabel("Encoding r (Pearson)", fontsize=12)
ax.set_ylabel("Number of voxels", fontsize=12)
ax.set_title("Group-level raw BOLD encoding: voxel-wise r histogram", fontsize=13)
ax.set_xlim(CORR_VMIN, CORR_VMAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_group_voxelwise_corr_histogram_raw.png", dpi=200)
plt.close()
print("Saved fig3_group_voxelwise_corr_histogram_raw.png")

# ---- Figure 4: Cross-subject mean map — mean r across subjects per voxel ----
# [IMPORTANT - Bhattacharjee et al. 2025 style] Computing across-subject mean
# gives a more robust estimate of encoding performance than any single subject.
mean_r_across_subjects = all_voxel_corrs_arr.mean(axis=0)  # (n_voxels,)
np.save(NPY_DIR / "mean_r_across_subjects_raw.npy", mean_r_across_subjects)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(mean_r_across_subjects, bins=60, color="#2196F3", alpha=0.75, edgecolor="white")
ax.axvline(float(np.mean(mean_r_across_subjects)),   color="crimson", linewidth=2.0,
           linestyle="--", label=f"Mean = {np.mean(mean_r_across_subjects):.5f}")
ax.axvline(float(np.median(mean_r_across_subjects)), color="orange",  linewidth=2.0,
           linestyle=":",  label=f"Median = {np.median(mean_r_across_subjects):.5f}")
ax.axvline(0, color="gray", linewidth=1.0)
ax.set_xlabel("Mean encoding r across subjects (Pearson)", fontsize=12)
ax.set_ylabel("Number of voxels", fontsize=12)
ax.set_title("Raw BOLD: mean encoding r across subjects (voxel-wise)", fontsize=13)
ax.set_xlim(CORR_VMIN, CORR_VMAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_mean_r_across_subjects_histogram_raw.png", dpi=200)
plt.close()
print("Saved fig4_mean_r_across_subjects_histogram_raw.png")

# ---- Figure 5: Summary statistics boxplot across subjects ----
# Each dot is a subject's mean r. Allows readers to see inter-subject variability.
fig, ax = plt.subplots(figsize=(5, 6))
bp = ax.boxplot(mean_r_per_subj, widths=0.4, patch_artist=True,
                medianprops=dict(color="white", linewidth=2))
bp["boxes"][0].set_facecolor("#4C72B0")
bp["boxes"][0].set_alpha(0.7)
# Overlay individual subject dots (jitter for readability)
rng = np.random.default_rng(42)
jitter = rng.uniform(-0.08, 0.08, size=N_SUBJECTS)
ax.scatter(np.ones(N_SUBJECTS) + jitter, mean_r_per_subj,
           color="steelblue", alpha=0.8, s=40, zorder=5)
ax.axhline(group_mean_r, color="crimson", linewidth=1.5, linestyle="--",
           label=f"Group mean r = {group_mean_r:.5f}")
ax.set_ylabel("Mean encoding r (Pearson)", fontsize=12)
ax.set_xticks([1])
ax.set_xticklabels(["Raw BOLD"], fontsize=11)
ax.set_title("Raw BOLD: across-subject distribution of mean r", fontsize=12)
ax.set_ylim(MEAN_R_MIN, MEAN_R_MAX)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_summary_boxplot_raw.png", dpi=200)
plt.close()
print("Saved fig5_summary_boxplot_raw.png")


# =========================
# 8. Save JSON summary
# =========================

total_elapsed = (time.time() - start_total) / 60.0

summary = {
    "dataset"       : "21styear",
    "roi_target"    : ROI_TARGET,
    "pipeline_type" : "step7b_raw_bold_encoding_per_voxel_alpha",
    "n_subjects"    : int(N_SUBJECTS),
    "n_voxels"      : int(N_VOXELS),

    "alpha_settings": {
        "alphas"        : [float(a) for a in ALPHAS],
        "nboots"        : int(NBOOTS),
        "chunklen"      : int(CHUNKLEN),
        "nchunks"       : int(NCHUNKS),
        "singcutoff"    : float(SINGCUTOFF),
        "use_corr"      : bool(USE_CORR),
        "seed"          : int(SEED),
        "individual_mode": "per_voxel_alpha (single_alpha=False, Huth lab style)",
        "group_mode"    : "single_alpha=True, reuse Step7a best alpha",
        "group_best_alpha": float(GROUP_BEST_ALPHA),
    },

    "individual_results": {
        "mean_r_across_subjects"  : float(np.mean(mean_r_per_subj)),
        "std_r_across_subjects"   : float(np.std(mean_r_per_subj)),
        "min_subject_mean_r"      : float(np.min(mean_r_per_subj)),
        "max_subject_mean_r"      : float(np.max(mean_r_per_subj)),
    },

    "group_results": {
        "mean_r"        : group_mean_r,
        "median_r"      : group_median_r,
        "p90_r"         : group_p90_r,
        "positive_frac" : group_pos_frac,
        "best_alpha"    : float(GROUP_BEST_ALPHA),
    },

    "total_elapsed_minutes": float(total_elapsed),

    "outputs": {
        "all_subjects_voxel_corrs" : str(NPY_DIR / "all_subjects_voxel_corrs_raw.npy"),
        "mean_r_across_subjects"   : str(NPY_DIR / "mean_r_across_subjects_raw.npy"),
        "group_voxel_corrs"        : str(NPY_DIR / "group_voxel_corrs_raw.npy"),
        "group_weights"            : str(NPY_DIR / "group_weights_raw.npy"),
        "individual_summary_csv"   : str(CSV_DIR / "individual_encoding_summary_raw.csv"),
        "group_summary_csv"        : str(CSV_DIR / "group_encoding_summary_raw.csv"),
    },
}

with open(JSON_DIR / "step7b_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved Step 7b summary to: {JSON_DIR / 'step7b_summary.json'}")


# =========================
# 9. Step-end validation checks
# =========================

print("\n" + "=" * 70)
print("STEP-END VALIDATION CHECKS")
print("=" * 70)

n_subj_ok       = len(all_voxel_corrs) == N_SUBJECTS
shapes_ok       = all(v.shape == (N_VOXELS,) for v in all_voxel_corrs)
finite_indiv_ok = all(np.isfinite(v).all() for v in all_voxel_corrs)
finite_group_ok = np.isfinite(group_voxel_corrs).all()
corr_range_ok   = (
    np.all(all_voxel_corrs_arr >= -1.0) and
    np.all(all_voxel_corrs_arr <=  1.0)
)

print(f"n_subj_ok:       {n_subj_ok}  ({len(all_voxel_corrs)} subjects)")
print(f"shapes_ok:       {shapes_ok}")
print(f"finite_indiv_ok: {finite_indiv_ok}")
print(f"finite_group_ok: {finite_group_ok}")
print(f"corr_range_ok:   {corr_range_ok}")
print(f"mean_r across subjects (individual): {np.mean(mean_r_per_subj):.5f}")
print(f"group mean_r:                        {group_mean_r:.5f}")

if not n_subj_ok:
    raise ValueError("Subject count mismatch in voxel_corrs output.")
if not shapes_ok:
    raise ValueError("Inconsistent voxel_corrs shapes across subjects.")
if not finite_indiv_ok:
    raise ValueError("Non-finite values in individual voxel_corrs.")
if not finite_group_ok:
    raise ValueError("Non-finite values in group voxel_corrs.")
if not corr_range_ok:
    raise ValueError("Correlation values out of [-1, 1] range.")

print("\nStep 7b completed successfully.")


In [ ]:
"""
Step 7c: SRM-Reconstructed BOLD Encoding Model Fitting
=======================================================
This step implements the core SRM+LLM encoding analysis inspired by
Bhattacharjee et al. (2025, Nature Computational Science).

The key idea (Bhattacharjee / Hasson lab):
    Each subject's BOLD signal is first projected into the SRM shared space,
    then reconstructed back into the original voxel space:
        X_reconstructed = W @ W.T @ X_original     [equation (2) in the paper]
    This reconstruction acts as a denoising operation — it retains only the
    stimulus-driven, cross-subject shared signal and discards idiosyncratic noise.

We then fit the same LLM-based encoding model (ridge regression, Huth lab style)
on the SRM-reconstructed BOLD, and compare encoding performance (Pearson r)
against the raw BOLD baseline from Step 7b.

Conditions compared:
    - Condition A (Step 7b): LLM → ridge → raw BOLD          [baseline]
    - Condition B (Step 7c): LLM → ridge → SRM-reconstructed BOLD  [this step]

Both individual-level (per-voxel alpha) and group-level (single alpha) analyses
are performed to enable fair comparison at both levels.

Sanity checks are embedded throughout to catch data quality issues early.
"""

from pathlib import Path
import json
import time
import itertools as itools
import random

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

ENC_PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
SRM_PROJECT_DIR = BASE_DIR / "runs"

STEP5_DIR  = ENC_PROJECT_DIR / "step5_build_delayed_design_matrix"
STEP6_DIR  = ENC_PROJECT_DIR / "step6_prepare_fmri_response"
STEP7A_DIR = ENC_PROJECT_DIR / "step7a_select_alpha_huthstyle"
STEP7B_DIR = ENC_PROJECT_DIR / "step7b_raw_bold_encoding"
STEP7C_DIR = ENC_PROJECT_DIR / "step7c_srm_reconstructed_encoding"

# SRM weight matrices from BrainIAK pipeline
SRM_STEP3B_DIR = SRM_PROJECT_DIR / RUN_NAME / "step3b_estimate_srm"

FIG_DIR = STEP7C_DIR / "figures"
CSV_DIR = STEP7C_DIR / "csv"
NPY_DIR = STEP7C_DIR / "npy"
JSON_DIR = STEP7C_DIR / "json"

for d in [STEP7C_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(__doc__)
print(f"Encoding project : {ENC_PROJECT_DIR}")
print(f"SRM project      : {SRM_PROJECT_DIR}")
print(f"Step 7c directory: {STEP7C_DIR}")


# =========================
# 2. Load all inputs
# =========================

X_TRAIN_PATH       = STEP5_DIR / "npy" / "X_train_delayed_lanczos_huthstyle.npy"
X_TEST_PATH        = STEP5_DIR / "npy" / "X_test_delayed_lanczos_huthstyle.npy"
Y_TRAIN_LIST_PATH  = STEP6_DIR / "npy" / "Y_train_list_huthstyle.npy"
Y_TEST_LIST_PATH   = STEP6_DIR / "npy" / "Y_test_list_huthstyle.npy"
Y_TRAIN_GROUP_PATH = STEP6_DIR / "npy" / "Y_train_group_huthstyle.npy"
Y_TEST_GROUP_PATH  = STEP6_DIR / "npy" / "Y_test_group_huthstyle.npy"
W_LIST_PATH        = SRM_STEP3B_DIR / "npy" / "weight_matrices.npy"
STEP7A_JSON_PATH   = STEP7A_DIR / "json" / "step7a_summary_huthstyle.json"
STEP7B_JSON_PATH   = STEP7B_DIR / "json" / "step7b_summary.json"

for p in [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_LIST_PATH, Y_TEST_LIST_PATH,
          Y_TRAIN_GROUP_PATH, Y_TEST_GROUP_PATH, W_LIST_PATH,
          STEP7A_JSON_PATH, STEP7B_JSON_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

print("\nLoading inputs...")
t_load = time.time()

X_train = np.load(X_TRAIN_PATH).astype(np.float32)
X_test  = np.load(X_TEST_PATH).astype(np.float32)

Y_train_list  = [np.asarray(x, dtype=np.float32)
                 for x in np.load(Y_TRAIN_LIST_PATH, allow_pickle=True)]
Y_test_list   = [np.asarray(x, dtype=np.float32)
                 for x in np.load(Y_TEST_LIST_PATH,  allow_pickle=True)]
Y_train_group = np.load(Y_TRAIN_GROUP_PATH).astype(np.float32)
Y_test_group  = np.load(Y_TEST_GROUP_PATH).astype(np.float32)

W_list = [np.asarray(w, dtype=np.float32)
          for w in np.load(W_LIST_PATH, allow_pickle=True)]

with open(STEP7A_JSON_PATH, "r", encoding="utf-8") as f:
    step7a_summary = json.load(f)
with open(STEP7B_JSON_PATH, "r", encoding="utf-8") as f:
    step7b_summary = json.load(f)

print(f"Inputs loaded in {time.time()-t_load:.1f}s")

N_SUBJECTS = len(Y_train_list)
N_VOXELS   = Y_train_list[0].shape[1]
N_FEATURES = W_list[0].shape[1]   # SRM shared dimensionality (K)

GROUP_BEST_ALPHA = float(step7a_summary["best_alpha"])

print(f"\nN_SUBJECTS : {N_SUBJECTS}")
print(f"N_VOXELS   : {N_VOXELS}")
print(f"SRM K (features): {N_FEATURES}")
print(f"X_train shape  : {X_train.shape}")
print(f"X_test  shape  : {X_test.shape}")
print(f"W shape (subj 0): {W_list[0].shape}  [voxels x K]")
print(f"GROUP_BEST_ALPHA: {GROUP_BEST_ALPHA:.5f}")


# =========================
# 3. Sanity check: inputs
# =========================

print("\n" + "=" * 60)
print("SANITY CHECK — INPUT DATA")
print("=" * 60)

# [SANITY] Basic shape consistency
assert len(Y_train_list) == len(Y_test_list) == len(W_list), \
    "Mismatch in number of subjects across Y_train, Y_test, W_list."
assert X_train.shape[0] == Y_train_list[0].shape[0], \
    f"Train time mismatch: X={X_train.shape[0]}, Y={Y_train_list[0].shape[0]}"
assert X_test.shape[0] == Y_test_list[0].shape[0], \
    f"Test time mismatch: X={X_test.shape[0]}, Y={Y_test_list[0].shape[0]}"
assert all(w.shape[0] == N_VOXELS for w in W_list), \
    "W matrix voxel dimension mismatch across subjects."
assert all(w.shape[1] == N_FEATURES for w in W_list), \
    "W matrix feature dimension mismatch across subjects."

# [SANITY] W matrices should be approximately orthonormal: W.T @ W ≈ I_K
# Inspired by BrainIAK SRM constraint: W_i^T W_i = I_k
print("\nChecking W matrix orthonormality (W.T @ W ≈ I_K):")
for si in range(min(3, N_SUBJECTS)):
    WtW = W_list[si].T.astype(np.float64) @ W_list[si].astype(np.float64)
    off_diag_err = float(np.abs(WtW - np.eye(N_FEATURES)).max())
    print(f"  Subject {si}: max off-diagonal error = {off_diag_err:.6f}  "
          f"{'OK' if off_diag_err < 0.05 else 'WARNING: large error'}")

# [SANITY] Y data should be approximately z-scored per voxel
print("\nChecking Y data per-voxel z-score (mean≈0, std≈1):")
for si in [0, N_SUBJECTS//2, N_SUBJECTS-1]:
    y = Y_train_list[si]
    pv_mean = float(np.abs(y.mean(axis=0)).mean())
    pv_std  = float(y.std(axis=0).mean())
    print(f"  Subject {si}: mean of |col-means| = {pv_mean:.5f}, "
          f"mean of col-stds = {pv_std:.5f}  "
          f"{'OK' if pv_mean < 0.05 and 0.90 < pv_std < 1.10 else 'WARNING'}")

# [SANITY] No NaN or Inf anywhere
assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), \
    "Non-finite values in X matrices."
assert all(np.isfinite(y).all() for y in Y_train_list), \
    "Non-finite values in Y_train_list."
assert all(np.isfinite(w).all() for w in W_list), \
    "Non-finite values in W_list."

print("\nAll input sanity checks passed.")


# =========================
# 4. SRM reconstruction
#    X_reconstructed = W @ W.T @ X_original
#    [Bhattacharjee et al. 2025, equation (2); Hasson lab]
# =========================

print("\n" + "=" * 60)
print("SRM RECONSTRUCTION:  X_recon = W @ W.T @ X")
print("Inspired by Bhattacharjee et al. (2025) equation (2)")
print("=" * 60)

t_recon = time.time()

Y_train_recon_list = []
Y_test_recon_list  = []

for si, W in enumerate(W_list):
    W64 = W.astype(np.float64)

    # [IMPORTANT - Bhattacharjee / Hasson lab]
    # Project into shared space then immediately project back.
    # This is a low-rank projection that retains only the stimulus-driven
    # shared signal and discards participant-specific noise.
    # W shape: (voxels, K)  →  W @ W.T shape: (voxels, voxels)
    # We avoid forming W @ W.T explicitly (memory) by doing two matmuls:
    #   step1: shared = W.T @ Y      shape: (time, K)
    #   step2: recon  = W  @ shared  shape: (time, voxels)   [then .T if needed]

    # Y_train shape: (time, voxels)
    shared_train = Y_train_list[si].astype(np.float64) @ W64   # (time, K)
    recon_train  = shared_train @ W64.T                         # (time, voxels)

    shared_test  = Y_test_list[si].astype(np.float64) @ W64    # (time, K)
    recon_test   = shared_test  @ W64.T                         # (time, voxels)

    Y_train_recon_list.append(recon_train.astype(np.float32))
    Y_test_recon_list.append(recon_test.astype(np.float32))

print(f"Reconstruction done in {time.time()-t_recon:.1f}s")
print(f"Y_train_recon shape (subj 0): {Y_train_recon_list[0].shape}")
print(f"Y_test_recon  shape (subj 0): {Y_test_recon_list[0].shape}")


# =========================
# 5. Sanity check: SRM reconstruction
# =========================

print("\n" + "=" * 60)
print("SANITY CHECK — SRM RECONSTRUCTION")
print("=" * 60)

# [SANITY] Reconstruction should reduce variance (low-rank projection).
# The reconstructed signal should have LESS variance than the original,
# since we project into K-dimensional subspace (K << n_voxels).
print("\nVariance comparison (original vs reconstructed):")
for si in [0, N_SUBJECTS//2, N_SUBJECTS-1]:
    var_orig  = float(np.var(Y_train_list[si]))
    var_recon = float(np.var(Y_train_recon_list[si]))
    ratio = var_recon / max(var_orig, 1e-8)
    print(f"  Subject {si}: var_orig={var_orig:.5f}, var_recon={var_recon:.5f}, "
          f"ratio={ratio:.5f}  "
          f"{'OK (reduced)' if ratio < 1.0 else 'WARNING: recon > orig'}")

# [SANITY] Correlation between original and reconstructed mean timecourse.
# Should be moderate-to-high (reflecting shared signal).
print("\nCorrelation between original and reconstructed mean timecourse:")
for si in [0, N_SUBJECTS//2, N_SUBJECTS-1]:
    orig_mean  = Y_train_list[si].mean(axis=1)
    recon_mean = Y_train_recon_list[si].mean(axis=1)
    r = float(np.corrcoef(orig_mean, recon_mean)[0, 1])
    print(f"  Subject {si}: r(orig_mean, recon_mean) = {r:.5f}  "
          f"{'OK' if r > 0.1 else 'WARNING: very low correlation'}")

# [SANITY] Reconstructed signal should not be all-zero or near-zero
print("\nReconstructed signal magnitude check:")
for si in [0, N_SUBJECTS//2, N_SUBJECTS-1]:
    rms = float(np.sqrt(np.mean(Y_train_recon_list[si] ** 2)))
    print(f"  Subject {si}: RMS of reconstructed train = {rms:.5f}  "
          f"{'OK' if rms > 1e-3 else 'WARNING: near-zero signal'}")

# [SANITY] Build reconstructed group average
Y_train_recon_group = np.mean(
    np.stack(Y_train_recon_list, axis=0), axis=0
).astype(np.float32)
Y_test_recon_group = np.mean(
    np.stack(Y_test_recon_list, axis=0), axis=0
).astype(np.float32)

# Group recon should correlate strongly with group original
r_group_train = float(np.corrcoef(
    Y_train_recon_group.mean(axis=1),
    Y_train_group.mean(axis=1)
)[0, 1])
print(f"\nGroup-level r(recon_mean, orig_mean) [train]: {r_group_train:.5f}  "
      f"{'OK' if r_group_train > 0.3 else 'WARNING: unexpectedly low'}")

print("\nAll reconstruction sanity checks passed.")


# =========================
# 6. Core Huth-style ridge functions
#    (same as Step 7b — reproduced here for self-contained step)
# =========================

# [IMPORTANT - Huth lab] Column-wise z-score, ddof=0.
# Matches `zs = lambda v: (v-v.mean(0))/v.std(0)` in ridge.py.
def zs(v: np.ndarray) -> np.ndarray:
    v  = np.asarray(v, dtype=np.float64)
    mu = v.mean(axis=0, keepdims=True)
    sd = v.std(axis=0, ddof=0, keepdims=True)
    sd[sd < 1e-12] = 1.0
    return np.nan_to_num((v - mu) / sd).astype(np.float32)


# [IMPORTANT - Huth lab] SVD-based ridge fit. Mirrors `ridge()` in ridge.py.
def ridge_fit(
    stim: np.ndarray,
    resp: np.ndarray,
    valphas: np.ndarray,
    singcutoff: float = 1e-10,
) -> np.ndarray:
    U, S, Vh = np.linalg.svd(stim.astype(np.float64), full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))
    U, S, Vh = U[:, :ngoodS], S[:ngoodS], Vh[:ngoodS, :]
    UR = U.T @ np.nan_to_num(resp.astype(np.float64))
    wt = np.zeros((stim.shape[1], resp.shape[1]), dtype=np.float32)
    for ua in np.unique(valphas):
        selvox = np.where(valphas == ua)[0]
        D = S / (S ** 2 + float(ua) ** 2)
        wt[:, selvox] = (Vh.T @ np.diag(D) @ UR[:, selvox]).astype(np.float32)
    return wt


# [IMPORTANT - Huth lab] Evaluate all alphas without storing weights.
# Mirrors `ridge_corr()` in ridge.py.
def ridge_corr_alphas(
    Rstim: np.ndarray,
    Pstim: np.ndarray,
    Rresp: np.ndarray,
    Presp: np.ndarray,
    alphas: np.ndarray,
    singcutoff: float = 1e-10,
    use_corr: bool = True,
) -> np.ndarray:
    U, S, Vh = np.linalg.svd(Rstim.astype(np.float64), full_matrices=False)
    ngoodS = int(np.sum(S > singcutoff))
    U, S, Vh = U[:, :ngoodS], S[:ngoodS], Vh[:ngoodS, :]
    UR   = U.T @ np.nan_to_num(Rresp.astype(np.float64))
    PVh  = Pstim.astype(np.float64) @ Vh.T
    zPre = zs(Presp)
    Rcorrs = []
    for a in alphas:
        D    = S / (S ** 2 + float(a) ** 2)
        pred = (PVh * D[None, :]) @ UR
        if use_corr:
            rc = np.nan_to_num((zPre * zs(pred)).mean(axis=0)).astype(np.float32)
        else:
            resvar = (Presp - pred).var(axis=0)
            Rsq    = 1.0 - resvar / np.maximum(Presp.var(axis=0), 1e-8)
            rc     = (np.sign(Rsq) * np.sqrt(np.abs(Rsq))).astype(np.float32)
        Rcorrs.append(rc)
    return np.stack(Rcorrs, axis=0).astype(np.float32)


# [IMPORTANT - Huth lab] Bootstrap chunk splits. Mirrors ridge.py exactly.
def make_bootstrap_splits(n_timepoints, chunklen, nchunks, nboots, seed=0):
    rng = random.Random(seed)
    allinds   = list(range(n_timepoints))
    indchunks = list(zip(*[iter(allinds)] * chunklen))
    if len(indchunks) < nchunks:
        raise ValueError(f"Not enough chunks: {len(indchunks)} < {nchunks}")
    valinds = []
    for _ in range(nboots):
        held     = rng.sample(indchunks, nchunks)
        heldinds = np.array(sorted(itools.chain(*held)), dtype=int)
        valinds.append(heldinds)
    return valinds


# [IMPORTANT - Huth lab] Per-voxel alpha via bootstrap CV.
# Mirrors `bootstrap_ridge(..., single_alpha=False)` in ridge.py.
def select_per_voxel_alpha(
    Rstim, Rresp, alphas, nboots, chunklen, nchunks,
    singcutoff=1e-10, use_corr=True, seed=0,
):
    n_tp    = Rstim.shape[0]
    valinds = make_bootstrap_splits(n_tp, chunklen, nchunks, nboots, seed)
    Rcmats  = []
    for bi, heldinds in enumerate(valinds):
        notheldinds = np.array(
            sorted(set(range(n_tp)) - set(heldinds.tolist())), dtype=int
        )
        Rcmat = ridge_corr_alphas(
            Rstim[notheldinds], Rstim[heldinds],
            Rresp[notheldinds], Rresp[heldinds],
            alphas=alphas, singcutoff=singcutoff, use_corr=use_corr,
        )
        Rcmats.append(Rcmat)
        print(f"    Boot {bi+1}/{nboots} | mean best corr: "
              f"{Rcmat.max(axis=0).mean():.5f}")
    allRcorrs     = np.stack(Rcmats, axis=2).astype(np.float32)
    meanbootcorrs = allRcorrs.mean(axis=2)
    bestalphainds = np.argmax(meanbootcorrs, axis=0)
    valphas       = alphas[bestalphainds]
    return valphas, allRcorrs


# =========================
# 7. Alpha settings
#    (keep identical to Step 7b for fair comparison)
# =========================

ALPHAS     = np.logspace(1, 5, 20).astype(np.float32)
NBOOTS     = 15
CHUNKLEN   = 40
NCHUNKS    = 6
SINGCUTOFF = 1e-10
USE_CORR   = True
SEED       = 0

print(f"\nAlpha grid: logspace(1,5,20), range [{ALPHAS.min():.1f}, {ALPHAS.max():.1f}]")
print(f"Bootstrap: NBOOTS={NBOOTS}, CHUNKLEN={CHUNKLEN}, NCHUNKS={NCHUNKS}")


# =========================
# 8. Individual-level encoding on SRM-reconstructed BOLD
#    Per-voxel alpha (Huth lab, single_alpha=False)
# =========================

print("\n" + "=" * 60)
print("INDIVIDUAL-LEVEL ENCODING (SRM-reconstructed BOLD)")
print("=" * 60)

indiv_rows      = []
all_voxel_corrs = []
start_total     = time.time()

for si in range(N_SUBJECTS):
    print(f"\n--- Subject {si+1}/{N_SUBJECTS} ---")
    t_subj = time.time()

    Y_train_s = Y_train_recon_list[si]   # (time, voxels) — reconstructed
    Y_test_s  = Y_test_recon_list[si]

    # [IMPORTANT - Huth lab] Per-voxel alpha selection on reconstructed BOLD.
    # Using seed offset per subject for independence across subjects.
    valphas, _ = select_per_voxel_alpha(
        Rstim=X_train, Rresp=Y_train_s,
        alphas=ALPHAS, nboots=NBOOTS, chunklen=CHUNKLEN, nchunks=NCHUNKS,
        singcutoff=SINGCUTOFF, use_corr=USE_CORR, seed=SEED + si,
    )

    # Fit final weights on full training set
    wt     = ridge_fit(X_train, Y_train_s, valphas, singcutoff=SINGCUTOFF)
    Y_pred = (X_test.astype(np.float64) @ wt.astype(np.float64)).astype(np.float32)

    # [IMPORTANT - Huth lab] Evaluate with Pearson r, column-wise.
    voxel_corrs = np.array([
        float(np.corrcoef(Y_test_s[:, ii], Y_pred[:, ii])[0, 1])
        for ii in range(N_VOXELS)
    ], dtype=np.float32)
    voxel_corrs = np.nan_to_num(voxel_corrs)
    all_voxel_corrs.append(voxel_corrs)

    elapsed = time.time() - t_subj
    print(f"  mean_r={np.mean(voxel_corrs):.5f} | "
          f"median_r={np.median(voxel_corrs):.5f} | "
          f"p90_r={np.percentile(voxel_corrs,90):.5f} | "
          f"pos_frac={np.mean(voxel_corrs>0):.5f} | "
          f"elapsed={elapsed/60:.2f} min")

    # Save per-subject outputs
    subj_dir = NPY_DIR / "individual"
    subj_dir.mkdir(exist_ok=True)
    np.save(subj_dir / f"subj{si:02d}_weights_srm.npy",     wt)
    np.save(subj_dir / f"subj{si:02d}_valphas_srm.npy",     valphas)
    np.save(subj_dir / f"subj{si:02d}_voxel_corrs_srm.npy", voxel_corrs)
    np.save(subj_dir / f"subj{si:02d}_Y_pred_srm.npy",      Y_pred)

    indiv_rows.append({
        "subject_index" : si,
        "mean_r"        : float(np.mean(voxel_corrs)),
        "median_r"      : float(np.median(voxel_corrs)),
        "std_r"         : float(np.std(voxel_corrs)),
        "p10_r"         : float(np.percentile(voxel_corrs, 10)),
        "p25_r"         : float(np.percentile(voxel_corrs, 25)),
        "p75_r"         : float(np.percentile(voxel_corrs, 75)),
        "p90_r"         : float(np.percentile(voxel_corrs, 90)),
        "p95_r"         : float(np.percentile(voxel_corrs, 95)),
        "positive_frac" : float(np.mean(voxel_corrs > 0)),
        "elapsed_min"   : float((time.time() - t_subj) / 60),
    })

all_voxel_corrs_arr = np.stack(all_voxel_corrs, axis=0).astype(np.float32)
np.save(NPY_DIR / "all_subjects_voxel_corrs_srm.npy", all_voxel_corrs_arr)

indiv_df = pd.DataFrame(indiv_rows)
indiv_df.to_csv(CSV_DIR / "individual_encoding_summary_srm.csv",
                index=False, encoding="utf-8-sig")

print(f"\nIndividual fitting done. Total elapsed: "
      f"{(time.time()-start_total)/60:.2f} min")


# =========================
# 9. Group-level encoding on SRM-reconstructed BOLD
#    Single alpha (from Step 7a), fit on group-average reconstructed BOLD
# =========================

print("\n" + "=" * 60)
print("GROUP-LEVEL ENCODING (SRM-reconstructed BOLD, group average)")
print("=" * 60)

t_group = time.time()

# [IMPORTANT - Bhattacharjee et al. 2025]
# Group average of SRM-reconstructed BOLD isolates the shared stimulus-driven
# signal while further suppressing participant-specific noise.
group_valphas = np.full(N_VOXELS, fill_value=GROUP_BEST_ALPHA, dtype=np.float32)

wt_group   = ridge_fit(X_train, Y_train_recon_group, group_valphas,
                       singcutoff=SINGCUTOFF)
Y_pred_grp = (X_test.astype(np.float64) @ wt_group.astype(np.float64)).astype(np.float32)

group_voxel_corrs = np.array([
    float(np.corrcoef(Y_test_recon_group[:, ii], Y_pred_grp[:, ii])[0, 1])
    for ii in range(N_VOXELS)
], dtype=np.float32)
group_voxel_corrs = np.nan_to_num(group_voxel_corrs)

np.save(NPY_DIR / "group_voxel_corrs_srm.npy",  group_voxel_corrs)
np.save(NPY_DIR / "group_weights_srm.npy",       wt_group)
np.save(NPY_DIR / "group_Y_pred_srm.npy",        Y_pred_grp)
np.save(NPY_DIR / "Y_train_recon_group.npy",     Y_train_recon_group)
np.save(NPY_DIR / "Y_test_recon_group.npy",      Y_test_recon_group)

group_mean_r   = float(np.mean(group_voxel_corrs))
group_median_r = float(np.median(group_voxel_corrs))
group_p90_r    = float(np.percentile(group_voxel_corrs, 90))
group_pos_frac = float(np.mean(group_voxel_corrs > 0))

print(f"Group mean_r   = {group_mean_r:.5f}  "
      f"(elapsed: {time.time()-t_group:.1f}s)")
print(f"Group median_r = {group_median_r:.5f}")
print(f"Group p90_r    = {group_p90_r:.5f}")
print(f"Group positive_frac = {group_pos_frac:.5f}")

group_summary_df = pd.DataFrame([{
    "mean_r"        : group_mean_r,
    "median_r"      : group_median_r,
    "std_r"         : float(np.std(group_voxel_corrs)),
    "p90_r"         : group_p90_r,
    "p95_r"         : float(np.percentile(group_voxel_corrs, 95)),
    "positive_frac" : group_pos_frac,
    "best_alpha"    : GROUP_BEST_ALPHA,
}])
group_summary_df.to_csv(CSV_DIR / "group_encoding_summary_srm.csv",
                        index=False, encoding="utf-8-sig")


# =========================
# 10. Sanity check: encoding results
# =========================

print("\n" + "=" * 60)
print("SANITY CHECK — ENCODING RESULTS")
print("=" * 60)

mean_r_per_subj_srm = indiv_df["mean_r"].values

# [SANITY] Load Step 7b individual results for direct comparison
raw_corrs_path = STEP7B_DIR / "npy" / "all_subjects_voxel_corrs_raw.npy"
raw_indiv_csv  = STEP7B_DIR / "csv" / "individual_encoding_summary_raw.csv"
raw_group_path = STEP7B_DIR / "npy" / "group_voxel_corrs_raw.npy"

if raw_corrs_path.exists() and raw_group_path.exists():
    raw_voxel_corrs_arr = np.load(raw_corrs_path)
    raw_group_corrs     = np.load(raw_group_path)
    raw_indiv_df        = pd.read_csv(raw_indiv_csv)
    mean_r_per_subj_raw = raw_indiv_df["mean_r"].values

    delta_indiv = mean_r_per_subj_srm - mean_r_per_subj_raw
    delta_group = group_mean_r - float(np.mean(raw_group_corrs))

    print(f"\nIndividual-level comparison (SRM vs raw):")
    print(f"  raw  mean_r across subjects : {np.mean(mean_r_per_subj_raw):.5f}")
    print(f"  SRM  mean_r across subjects : {np.mean(mean_r_per_subj_srm):.5f}")
    print(f"  delta (SRM - raw)           : {np.mean(delta_indiv):.5f}")
    print(f"  subjects improved           : "
          f"{np.sum(delta_indiv > 0)}/{N_SUBJECTS}")

    print(f"\nGroup-level comparison (SRM vs raw):")
    print(f"  raw  group mean_r : {np.mean(raw_group_corrs):.5f}")
    print(f"  SRM  group mean_r : {group_mean_r:.5f}")
    print(f"  delta (SRM - raw) : {delta_group:.5f}")

    # [SANITY] Correlation values must be in [-1, 1]
    assert np.all(all_voxel_corrs_arr >= -1.0) and \
           np.all(all_voxel_corrs_arr <=  1.0), \
        "SRM encoding correlations out of [-1, 1] range."

    # [SANITY] Warn if SRM is dramatically worse than raw (suggests bug)
    if np.mean(delta_indiv) < -0.05:
        print("\n[WARNING] SRM individual mean_r is substantially LOWER than raw. "
              "Please check W matrix orientation and reconstruction logic.")
    else:
        print("\nSRM reconstruction sanity check passed.")
else:
    print("Step 7b outputs not found — skipping raw vs SRM comparison.")
    delta_indiv = np.zeros(N_SUBJECTS)
    delta_group = 0.0
    mean_r_per_subj_raw = np.zeros(N_SUBJECTS)
    raw_group_corrs     = np.zeros(N_VOXELS)


# =========================
# 11. Save Δ arrays (SRM - raw)
# =========================

mean_r_across_subjects_srm = all_voxel_corrs_arr.mean(axis=0)  # (n_voxels,)
np.save(NPY_DIR / "mean_r_across_subjects_srm.npy", mean_r_across_subjects_srm)

if raw_corrs_path.exists():
    mean_r_across_subjects_raw = raw_voxel_corrs_arr.mean(axis=0)
    delta_voxelwise_indiv = mean_r_across_subjects_srm - mean_r_across_subjects_raw
    delta_voxelwise_group = group_voxel_corrs - raw_group_corrs

    np.save(NPY_DIR / "delta_mean_r_voxelwise_individual.npy", delta_voxelwise_indiv)
    np.save(NPY_DIR / "delta_voxelwise_group.npy",             delta_voxelwise_group)
    print(f"\nSaved delta arrays (SRM - raw).")


# =========================
# 12. Figures
# =========================

# [IMPORTANT] Use identical axis limits as Step 7b for direct visual comparison.
CORR_VMIN  = -0.20
CORR_VMAX  =  0.60
MEAN_R_MIN =  0.00
MEAN_R_MAX =  0.25
DELTA_VMIN = -0.10
DELTA_VMAX =  0.15

subject_indices = np.arange(N_SUBJECTS)

# ---- Figure 1: SRM vs Raw per-subject mean r (side-by-side bars) ----
# [Inspired by Bhattacharjee et al. 2025, Fig. 2 style]
# Each subject has two bars: raw (blue) and SRM (orange).
fig, ax = plt.subplots(figsize=(14, 5))
width = 0.38
ax.bar(subject_indices - width/2, mean_r_per_subj_raw, width,
       color="#4C72B0", alpha=0.80, label="Raw BOLD")
ax.bar(subject_indices + width/2, mean_r_per_subj_srm, width,
       color="#DD8452", alpha=0.80, label="SRM-reconstructed")
ax.axhline(float(np.mean(mean_r_per_subj_raw)), color="#4C72B0",
           linewidth=1.5, linestyle="--", alpha=0.7,
           label=f"Raw mean = {np.mean(mean_r_per_subj_raw):.5f}")
ax.axhline(float(np.mean(mean_r_per_subj_srm)), color="#DD8452",
           linewidth=1.5, linestyle="--", alpha=0.7,
           label=f"SRM mean = {np.mean(mean_r_per_subj_srm):.5f}")
ax.set_xlabel("Subject index", fontsize=12)
ax.set_ylabel("Mean encoding r (Pearson)", fontsize=12)
ax.set_title("Raw vs SRM-reconstructed BOLD: per-subject mean r",
             fontsize=13)
ax.set_xticks(subject_indices)
ax.set_xticklabels([f"S{i}" for i in subject_indices], fontsize=8, rotation=45)
ax.set_ylim(MEAN_R_MIN, MEAN_R_MAX)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_raw_vs_srm_per_subject_mean_r.png", dpi=200)
plt.close()
print("Saved fig1_raw_vs_srm_per_subject_mean_r.png")

# ---- Figure 2: Violin plot — raw vs SRM voxel-wise r distribution ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, corrs_list, label, color in zip(
    axes,
    [
        [np.load(raw_corrs_path)[i] for i in range(N_SUBJECTS)]
        if raw_corrs_path.exists() else [np.zeros(N_VOXELS)]*N_SUBJECTS,
        all_voxel_corrs
    ],
    ["Raw BOLD", "SRM-reconstructed"],
    ["#4C72B0", "#DD8452"],
):
    parts = ax.violinplot(corrs_list, positions=subject_indices,
                          showmedians=True, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    parts["cmedians"].set_color("white")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Subject index", fontsize=11)
    ax.set_ylabel("Voxel-wise encoding r", fontsize=11)
    ax.set_title(f"{label}: voxel-wise r distribution", fontsize=12)
    ax.set_xticks(subject_indices)
    ax.set_xticklabels([f"S{i}" for i in subject_indices], fontsize=7, rotation=45)
    ax.set_ylim(CORR_VMIN, CORR_VMAX)
plt.suptitle("Voxel-wise encoding r: Raw vs SRM-reconstructed",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_violin_raw_vs_srm.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved fig2_violin_raw_vs_srm.png")

# ---- Figure 3: Scatter — per-subject raw r vs SRM r ----
# Points above diagonal = SRM improved that subject.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(mean_r_per_subj_raw, mean_r_per_subj_srm,
           color="#2196F3", alpha=0.85, s=60, zorder=5)
for si in range(N_SUBJECTS):
    ax.annotate(f"S{si}", (mean_r_per_subj_raw[si], mean_r_per_subj_srm[si]),
                fontsize=7, alpha=0.7, xytext=(3, 3), textcoords="offset points")
lim_lo = min(MEAN_R_MIN, float(min(mean_r_per_subj_raw.min(),
                                    mean_r_per_subj_srm.min())) - 0.005)
lim_hi = MEAN_R_MAX
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi],
        color="gray", linewidth=1.2, linestyle="--", label="y = x")
ax.set_xlabel("Raw BOLD mean encoding r", fontsize=12)
ax.set_ylabel("SRM-reconstructed mean encoding r", fontsize=12)
ax.set_title("Per-subject: SRM vs Raw encoding r", fontsize=12)
ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_scatter_raw_vs_srm_per_subject.png", dpi=200)
plt.close()
print("Saved fig3_scatter_raw_vs_srm_per_subject.png")

# ---- Figure 4: Delta histogram (SRM - raw) per subject ----
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(subject_indices, delta_indiv,
       color=["#DD8452" if d > 0 else "#4C72B0" for d in delta_indiv],
       alpha=0.80)
ax.axhline(0, color="gray", linewidth=1.0, linestyle="--")
ax.axhline(float(np.mean(delta_indiv)), color="crimson", linewidth=2.0,
           linestyle="--",
           label=f"Mean Δ = {np.mean(delta_indiv):.5f}")
ax.set_xlabel("Subject index", fontsize=12)
ax.set_ylabel("Δ mean r  (SRM − Raw)", fontsize=12)
ax.set_title("Per-subject improvement: SRM − Raw", fontsize=12)
ax.set_xticks(subject_indices)
ax.set_xticklabels([f"S{i}" for i in subject_indices], fontsize=8, rotation=45)
ax.set_ylim(DELTA_VMIN, DELTA_VMAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_delta_mean_r_per_subject.png", dpi=200)
plt.close()
print("Saved fig4_delta_mean_r_per_subject.png")

# ---- Figure 5: Group-level histogram — raw vs SRM voxel-wise r ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(raw_group_corrs, bins=60, color="#4C72B0", alpha=0.60,
        label=f"Raw  (mean={np.mean(raw_group_corrs):.5f})", edgecolor="none")
ax.hist(group_voxel_corrs, bins=60, color="#DD8452", alpha=0.60,
        label=f"SRM  (mean={group_mean_r:.5f})", edgecolor="none")
ax.axvline(float(np.mean(raw_group_corrs)), color="#4C72B0",
           linewidth=2.0, linestyle="--")
ax.axvline(group_mean_r, color="#DD8452",
           linewidth=2.0, linestyle="--")
ax.axvline(0, color="gray", linewidth=1.0)
ax.set_xlabel("Encoding r (Pearson)", fontsize=12)
ax.set_ylabel("Number of voxels", fontsize=12)
ax.set_title("Group-level voxel-wise r: Raw vs SRM-reconstructed",
             fontsize=13)
ax.set_xlim(CORR_VMIN, CORR_VMAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_group_histogram_raw_vs_srm.png", dpi=200)
plt.close()
print("Saved fig5_group_histogram_raw_vs_srm.png")

# ---- Figure 6: Summary boxplot — raw vs SRM individual mean r ----
# Each dot is one subject. This is the clearest single summary figure.
fig, ax = plt.subplots(figsize=(6, 6))
data_to_plot = [mean_r_per_subj_raw, mean_r_per_subj_srm]
bp = ax.boxplot(data_to_plot, widths=0.4, patch_artist=True,
                medianprops=dict(color="white", linewidth=2))
colors = ["#4C72B0", "#DD8452"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
rng = np.random.default_rng(42)
for xi, (data, color) in enumerate(zip(data_to_plot, colors), start=1):
    jitter = rng.uniform(-0.08, 0.08, size=len(data))
    ax.scatter(np.ones(len(data)) * xi + jitter, data,
               color=color, alpha=0.85, s=40, zorder=5)
# Connect paired subject dots
for si in range(N_SUBJECTS):
    ax.plot([1, 2], [mean_r_per_subj_raw[si], mean_r_per_subj_srm[si]],
            color="gray", linewidth=0.6, alpha=0.4)
ax.set_ylabel("Mean encoding r (Pearson)", fontsize=12)
ax.set_xticks([1, 2])
ax.set_xticklabels(["Raw BOLD", "SRM-reconstructed"], fontsize=11)
ax.set_title("Individual mean r: Raw vs SRM", fontsize=12)
ax.set_ylim(MEAN_R_MIN, MEAN_R_MAX)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig6_summary_boxplot_raw_vs_srm.png", dpi=200)
plt.close()
print("Saved fig6_summary_boxplot_raw_vs_srm.png")


# =========================
# 13. Save JSON summary
# =========================

total_elapsed = (time.time() - start_total) / 60.0

summary = {
    "dataset"       : "21styear",
    "roi_target"    : ROI_TARGET,
    "pipeline_type" : "step7c_srm_reconstructed_bold_encoding",
    "srm_run_name"  : RUN_NAME,
    "n_subjects"    : int(N_SUBJECTS),
    "n_voxels"      : int(N_VOXELS),
    "srm_k_features": int(N_FEATURES),

    "reconstruction": {
        "method"  : "W @ W.T @ X  (Bhattacharjee et al. 2025, eq. 2)",
        "W_shape" : list(W_list[0].shape),
    },

    "alpha_settings": {
        "alphas"          : [float(a) for a in ALPHAS],
        "nboots"          : int(NBOOTS),
        "chunklen"        : int(CHUNKLEN),
        "nchunks"         : int(NCHUNKS),
        "individual_mode" : "per_voxel_alpha (single_alpha=False, Huth lab)",
        "group_mode"      : "single_alpha=True, reuse Step7a best alpha",
        "group_best_alpha": float(GROUP_BEST_ALPHA),
    },

    "individual_results_srm": {
        "mean_r_across_subjects" : float(np.mean(mean_r_per_subj_srm)),
        "std_r_across_subjects"  : float(np.std(mean_r_per_subj_srm)),
        "min_subject_mean_r"     : float(np.min(mean_r_per_subj_srm)),
        "max_subject_mean_r"     : float(np.max(mean_r_per_subj_srm)),
    },

    "individual_results_raw": {
        "mean_r_across_subjects" : float(np.mean(mean_r_per_subj_raw)),
    },

    "individual_delta": {
        "mean_delta"       : float(np.mean(delta_indiv)),
        "subjects_improved": int(np.sum(delta_indiv > 0)),
        "pct_improved"     : float(np.mean(delta_indiv > 0) * 100),
    },

    "group_results_srm": {
        "mean_r"        : group_mean_r,
        "median_r"      : group_median_r,
        "p90_r"         : group_p90_r,
        "positive_frac" : group_pos_frac,
    },

    "group_delta": {
        "delta_mean_r" : float(delta_group),
    },

    "total_elapsed_minutes": float(total_elapsed),

    "outputs": {
        "all_subjects_voxel_corrs_srm"   : str(NPY_DIR / "all_subjects_voxel_corrs_srm.npy"),
        "mean_r_across_subjects_srm"     : str(NPY_DIR / "mean_r_across_subjects_srm.npy"),
        "group_voxel_corrs_srm"          : str(NPY_DIR / "group_voxel_corrs_srm.npy"),
        "delta_voxelwise_individual"     : str(NPY_DIR / "delta_mean_r_voxelwise_individual.npy"),
        "delta_voxelwise_group"          : str(NPY_DIR / "delta_voxelwise_group.npy"),
        "individual_summary_csv"         : str(CSV_DIR / "individual_encoding_summary_srm.csv"),
        "group_summary_csv"              : str(CSV_DIR / "group_encoding_summary_srm.csv"),
    },
}

with open(JSON_DIR / "step7c_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved Step 7c summary to: {JSON_DIR / 'step7c_summary.json'}")


# =========================
# 14. Step-end validation checks
# =========================

print("\n" + "=" * 60)
print("STEP-END VALIDATION CHECKS")
print("=" * 60)

n_subj_ok       = len(all_voxel_corrs) == N_SUBJECTS
shapes_ok       = all(v.shape == (N_VOXELS,) for v in all_voxel_corrs)
finite_ok       = all(np.isfinite(v).all() for v in all_voxel_corrs)
finite_group_ok = np.isfinite(group_voxel_corrs).all()
range_ok        = (np.all(all_voxel_corrs_arr >= -1.0) and
                   np.all(all_voxel_corrs_arr <=  1.0))

print(f"n_subj_ok       : {n_subj_ok}  ({len(all_voxel_corrs)} subjects)")
print(f"shapes_ok       : {shapes_ok}")
print(f"finite_ok       : {finite_ok}")
print(f"finite_group_ok : {finite_group_ok}")
print(f"range_ok        : {range_ok}")
print(f"mean_r (individual SRM) : {np.mean(mean_r_per_subj_srm):.5f}")
print(f"mean_r (individual raw) : {np.mean(mean_r_per_subj_raw):.5f}")
print(f"mean delta (SRM - raw)  : {np.mean(delta_indiv):.5f}")
print(f"group mean_r SRM        : {group_mean_r:.5f}")
print(f"group mean_r raw        : {float(np.mean(raw_group_corrs)):.5f}")
print(f"group delta (SRM - raw) : {delta_group:.5f}")

if not n_subj_ok:
    raise ValueError("Subject count mismatch.")
if not shapes_ok:
    raise ValueError("Inconsistent voxel_corrs shapes.")
if not finite_ok:
    raise ValueError("Non-finite values in individual voxel_corrs.")
if not finite_group_ok:
    raise ValueError("Non-finite values in group voxel_corrs.")
if not range_ok:
    raise ValueError("Correlation values out of [-1, 1] range.")

print(f"\nTotal elapsed: {total_elapsed:.2f} min")
print("\nStep 7c completed successfully.")


step7d_statistical_summary_raw_vs_srm

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, wilcoxon, binomtest


"""
Step 7d: Statistical Summary of Raw vs SRM Encoding
===================================================
This step performs a pure statistics / reporting pass on the outputs from:
    - Step 7b: raw BOLD encoding
    - Step 7c: SRM-reconstructed BOLD encoding

No model is fit here. We only load saved outputs, compute descriptive and
paired-comparison statistics, and save summary tables + figures in the same
style as the earlier Step 7 cells.
"""


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

ENC_PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
STEP7B_DIR = ENC_PROJECT_DIR / "step7b_raw_bold_encoding"
STEP7C_DIR = ENC_PROJECT_DIR / "step7c_srm_reconstructed_encoding"
STEP7D_DIR = ENC_PROJECT_DIR / "step7d_statistical_summary_raw_vs_srm"

FIG_DIR = STEP7D_DIR / "figures"
CSV_DIR = STEP7D_DIR / "csv"
NPY_DIR = STEP7D_DIR / "npy"
JSON_DIR = STEP7D_DIR / "json"

for d in [STEP7D_DIR, FIG_DIR, CSV_DIR, NPY_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(__doc__)
print(f"Encoding project directory: {ENC_PROJECT_DIR}")
print(f"Step 7d directory: {STEP7D_DIR}")


# =========================
# 2. Load Step 7b / 7c outputs
# =========================

RAW_ALL_SUBJ_PATH = STEP7B_DIR / "npy" / "all_subjects_voxel_corrs_raw.npy"
RAW_GROUP_PATH = STEP7B_DIR / "npy" / "group_voxel_corrs_raw.npy"
SRM_ALL_SUBJ_PATH = STEP7C_DIR / "npy" / "all_subjects_voxel_corrs_srm.npy"
SRM_GROUP_PATH = STEP7C_DIR / "npy" / "group_voxel_corrs_srm.npy"

STEP7B_JSON_PATH = STEP7B_DIR / "json" / "step7b_summary.json"
STEP7C_JSON_PATH = STEP7C_DIR / "json" / "step7c_summary.json"

for p in [
    RAW_ALL_SUBJ_PATH,
    RAW_GROUP_PATH,
    SRM_ALL_SUBJ_PATH,
    SRM_GROUP_PATH,
    STEP7B_JSON_PATH,
    STEP7C_JSON_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

start_total = time.time()

raw_voxel_corrs_arr = np.load(RAW_ALL_SUBJ_PATH).astype(np.float32)
raw_group_corrs = np.load(RAW_GROUP_PATH).astype(np.float32)
srm_voxel_corrs_arr = np.load(SRM_ALL_SUBJ_PATH).astype(np.float32)
srm_group_corrs = np.load(SRM_GROUP_PATH).astype(np.float32)

with open(STEP7B_JSON_PATH, "r", encoding="utf-8") as f:
    step7b_summary = json.load(f)
with open(STEP7C_JSON_PATH, "r", encoding="utf-8") as f:
    step7c_summary = json.load(f)

print("\nLoaded saved encoding outputs.")
print(f"raw_voxel_corrs_arr shape: {raw_voxel_corrs_arr.shape}")
print(f"srm_voxel_corrs_arr shape: {srm_voxel_corrs_arr.shape}")
print(f"raw_group_corrs shape: {raw_group_corrs.shape}")
print(f"srm_group_corrs shape: {srm_group_corrs.shape}")


# =========================
# 3. Basic shape checks
# =========================

if raw_voxel_corrs_arr.shape != srm_voxel_corrs_arr.shape:
    raise ValueError(
        f"Raw/SRM subject array shape mismatch: {raw_voxel_corrs_arr.shape} vs {srm_voxel_corrs_arr.shape}"
    )
if raw_group_corrs.shape != srm_group_corrs.shape:
    raise ValueError(
        f"Raw/SRM group array shape mismatch: {raw_group_corrs.shape} vs {srm_group_corrs.shape}"
    )

N_SUBJECTS, N_VOXELS = raw_voxel_corrs_arr.shape
subject_indices = np.arange(N_SUBJECTS)

if not np.isfinite(raw_voxel_corrs_arr).all():
    raise ValueError("Non-finite values detected in raw_voxel_corrs_arr.")
if not np.isfinite(srm_voxel_corrs_arr).all():
    raise ValueError("Non-finite values detected in srm_voxel_corrs_arr.")


# =========================
# 4. Helper functions
# =========================

def summarize_vector(x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=np.float64).ravel()
    return {
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
        "std": float(np.std(x)),
        "p10": float(np.percentile(x, 10)),
        "p25": float(np.percentile(x, 25)),
        "p75": float(np.percentile(x, 75)),
        "p90": float(np.percentile(x, 90)),
        "p95": float(np.percentile(x, 95)),
        "positive_frac": float(np.mean(x > 0)),
    }


def bootstrap_mean_ci(x: np.ndarray, n_boot: int = 10000, seed: int = 0) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float64).ravel()
    means = np.empty(n_boot, dtype=np.float64)
    for bi in range(n_boot):
        sample = rng.choice(x, size=len(x), replace=True)
        means[bi] = sample.mean()
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(lo), float(hi)


def paired_permutation_pvalue(x: np.ndarray, n_perm: int = 20000, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float64).ravel()
    observed = float(np.mean(x))
    abs_observed = abs(observed)
    null_means = np.empty(n_perm, dtype=np.float64)
    for pi in range(n_perm):
        flips = rng.choice([-1.0, 1.0], size=len(x), replace=True)
        null_means[pi] = np.mean(x * flips)
    p = (np.sum(np.abs(null_means) >= abs_observed) + 1.0) / (n_perm + 1.0)
    return float(p)


# =========================
# 5. Subject-level descriptive statistics
# =========================

mean_r_per_subj_raw = raw_voxel_corrs_arr.mean(axis=1)
mean_r_per_subj_srm = srm_voxel_corrs_arr.mean(axis=1)
median_r_per_subj_raw = np.median(raw_voxel_corrs_arr, axis=1)
median_r_per_subj_srm = np.median(srm_voxel_corrs_arr, axis=1)

delta_indiv = mean_r_per_subj_srm - mean_r_per_subj_raw
delta_indiv_median = median_r_per_subj_srm - median_r_per_subj_raw

subject_stats_df = pd.DataFrame({
    "subject_index": subject_indices,
    "raw_mean_r": mean_r_per_subj_raw.astype(float),
    "srm_mean_r": mean_r_per_subj_srm.astype(float),
    "delta_mean_r": delta_indiv.astype(float),
    "raw_median_r": median_r_per_subj_raw.astype(float),
    "srm_median_r": median_r_per_subj_srm.astype(float),
    "delta_median_r": delta_indiv_median.astype(float),
    "improved": (delta_indiv > 0),
})
subject_stats_df.to_csv(CSV_DIR / "subject_level_statistics_raw_vs_srm.csv", index=False)

print("\nSubject-level summary:")
print(subject_stats_df.to_string(index=False))


# =========================
# 6. Subject-level inferential statistics
# =========================

ttest_res = ttest_rel(mean_r_per_subj_srm, mean_r_per_subj_raw)
if np.allclose(delta_indiv, 0.0):
    wilcoxon_stat = 0.0
    wilcoxon_p = 1.0
else:
    wilcoxon_res = wilcoxon(mean_r_per_subj_srm, mean_r_per_subj_raw, zero_method="wilcox")
    wilcoxon_stat = float(wilcoxon_res.statistic)
    wilcoxon_p = float(wilcoxon_res.pvalue)

n_improved = int(np.sum(delta_indiv > 0))
n_nonzero = int(np.sum(delta_indiv != 0))
if n_nonzero == 0:
    sign_p = 1.0
else:
    sign_p = float(binomtest(n_improved, n=n_nonzero, p=0.5, alternative="two-sided").pvalue)

delta_ci_lo, delta_ci_hi = bootstrap_mean_ci(delta_indiv, n_boot=10000, seed=0)
perm_p = paired_permutation_pvalue(delta_indiv, n_perm=20000, seed=0)

subject_inference_df = pd.DataFrame({
    "metric": ["paired_mean_r_srm_minus_raw"],
    "n_subjects": [int(N_SUBJECTS)],
    "mean_delta": [float(np.mean(delta_indiv))],
    "median_delta": [float(np.median(delta_indiv))],
    "std_delta": [float(np.std(delta_indiv))],
    "bootstrap_ci_2p5": [delta_ci_lo],
    "bootstrap_ci_97p5": [delta_ci_hi],
    "paired_t_stat": [float(ttest_res.statistic)],
    "paired_t_p": [float(ttest_res.pvalue)],
    "wilcoxon_stat": [wilcoxon_stat],
    "wilcoxon_p": [wilcoxon_p],
    "sign_test_p": [sign_p],
    "permutation_p": [perm_p],
    "subjects_improved": [n_improved],
    "pct_improved": [float(np.mean(delta_indiv > 0) * 100.0)],
})
subject_inference_df.to_csv(CSV_DIR / "subject_level_inference_raw_vs_srm.csv", index=False)

print("\nSubject-level inference summary:")
print(subject_inference_df.to_string(index=False))


# =========================
# 7. Voxel-level descriptive statistics
# =========================

mean_r_across_subjects_raw = raw_voxel_corrs_arr.mean(axis=0)
mean_r_across_subjects_srm = srm_voxel_corrs_arr.mean(axis=0)
delta_voxelwise_subject_mean = mean_r_across_subjects_srm - mean_r_across_subjects_raw
delta_voxelwise_group = srm_group_corrs - raw_group_corrs

np.save(NPY_DIR / "mean_r_across_subjects_raw.npy", mean_r_across_subjects_raw.astype(np.float32))
np.save(NPY_DIR / "mean_r_across_subjects_srm.npy", mean_r_across_subjects_srm.astype(np.float32))
np.save(NPY_DIR / "delta_voxelwise_subject_mean.npy", delta_voxelwise_subject_mean.astype(np.float32))
np.save(NPY_DIR / "delta_voxelwise_group.npy", delta_voxelwise_group.astype(np.float32))

voxel_summary_df = pd.DataFrame({
    "summary_scope": ["mean_across_subjects", "group_average"],
    "raw_mean": [float(np.mean(mean_r_across_subjects_raw)), float(np.mean(raw_group_corrs))],
    "srm_mean": [float(np.mean(mean_r_across_subjects_srm)), float(np.mean(srm_group_corrs))],
    "delta_mean": [float(np.mean(delta_voxelwise_subject_mean)), float(np.mean(delta_voxelwise_group))],
    "raw_median": [float(np.median(mean_r_across_subjects_raw)), float(np.median(raw_group_corrs))],
    "srm_median": [float(np.median(mean_r_across_subjects_srm)), float(np.median(srm_group_corrs))],
    "delta_positive_frac": [
        float(np.mean(delta_voxelwise_subject_mean > 0)),
        float(np.mean(delta_voxelwise_group > 0)),
    ],
})
voxel_summary_df.to_csv(CSV_DIR / "voxel_level_statistics_raw_vs_srm.csv", index=False)


# =========================
# 8. Plot settings
# =========================

CORR_VMIN = float(min(raw_voxel_corrs_arr.min(), srm_voxel_corrs_arr.min(), raw_group_corrs.min(), srm_group_corrs.min()))
CORR_VMAX = float(max(raw_voxel_corrs_arr.max(), srm_voxel_corrs_arr.max(), raw_group_corrs.max(), srm_group_corrs.max()))
CORR_PAD = 0.02
CORR_VMIN -= CORR_PAD
CORR_VMAX += CORR_PAD

MEAN_R_MIN = float(min(mean_r_per_subj_raw.min(), mean_r_per_subj_srm.min()) - 0.01)
MEAN_R_MAX = float(max(mean_r_per_subj_raw.max(), mean_r_per_subj_srm.max()) + 0.01)
DELTA_MIN = float(min(delta_indiv.min(), delta_voxelwise_subject_mean.min(), delta_voxelwise_group.min()) - 0.01)
DELTA_MAX = float(max(delta_indiv.max(), delta_voxelwise_subject_mean.max(), delta_voxelwise_group.max()) + 0.01)


# =========================
# 9. Figures
# =========================

# ---- Figure 1: Paired subject summary (same style family as Step 7c) ----
fig, ax = plt.subplots(figsize=(6, 6))
data_to_plot = [mean_r_per_subj_raw, mean_r_per_subj_srm]
bp = ax.boxplot(data_to_plot, widths=0.4, patch_artist=True,
                medianprops=dict(color="white", linewidth=2))
colors = ["#4C72B0", "#DD8452"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
rng = np.random.default_rng(42)
for xi, (data, color) in enumerate(zip(data_to_plot, colors), start=1):
    jitter = rng.uniform(-0.08, 0.08, size=len(data))
    ax.scatter(np.ones(len(data)) * xi + jitter, data,
               color=color, alpha=0.85, s=40, zorder=5)
for si in range(N_SUBJECTS):
    ax.plot([1, 2], [mean_r_per_subj_raw[si], mean_r_per_subj_srm[si]],
            color="gray", linewidth=0.6, alpha=0.4)
ax.set_ylabel("Mean encoding r (Pearson)", fontsize=12)
ax.set_xticks([1, 2])
ax.set_xticklabels(["Raw BOLD", "SRM-reconstructed"], fontsize=11)
ax.set_title("Subject-level mean r: Raw vs SRM", fontsize=12)
ax.set_ylim(MEAN_R_MIN, MEAN_R_MAX)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_subject_boxplot_raw_vs_srm.png", dpi=200)
plt.close()
print("Saved fig1_subject_boxplot_raw_vs_srm.png")

# ---- Figure 2: Subject delta bar plot ----
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(subject_indices, delta_indiv,
       color=["#DD8452" if d > 0 else "#4C72B0" for d in delta_indiv],
       alpha=0.80)
ax.axhline(0, color="gray", linewidth=1.0, linestyle="--")
ax.axhline(float(np.mean(delta_indiv)), color="crimson", linewidth=2.0,
           linestyle="--",
           label=f"Mean delta = {np.mean(delta_indiv):.5f}")
ax.set_xlabel("Subject index", fontsize=12)
ax.set_ylabel("Delta mean r (SRM - Raw)", fontsize=12)
ax.set_title("Subject-level improvement: SRM - Raw", fontsize=12)
ax.set_xticks(subject_indices)
ax.set_xticklabels([f"S{i}" for i in subject_indices], fontsize=8, rotation=45)
ax.set_ylim(DELTA_MIN, DELTA_MAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_subject_delta_barplot.png", dpi=200)
plt.close()
print("Saved fig2_subject_delta_barplot.png")

# ---- Figure 3: Histogram of voxel-wise delta averaged across subjects ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(delta_voxelwise_subject_mean, bins=60, color="#DD8452", alpha=0.70,
        edgecolor="none", label=f"Mean={np.mean(delta_voxelwise_subject_mean):.5f}")
ax.axvline(0, color="gray", linewidth=1.0)
ax.axvline(float(np.mean(delta_voxelwise_subject_mean)), color="#C44E52",
           linewidth=2.0, linestyle="--")
ax.set_xlabel("Delta encoding r (SRM - Raw)", fontsize=12)
ax.set_ylabel("Number of voxels", fontsize=12)
ax.set_title("Voxel-wise delta r averaged across subjects", fontsize=13)
ax.set_xlim(DELTA_MIN, DELTA_MAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_voxel_delta_hist_subjectmean.png", dpi=200)
plt.close()
print("Saved fig3_voxel_delta_hist_subjectmean.png")

# ---- Figure 4: Group-level histogram raw vs SRM ----
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(raw_group_corrs, bins=60, color="#4C72B0", alpha=0.60,
        label=f"Raw  (mean={np.mean(raw_group_corrs):.5f})", edgecolor="none")
ax.hist(srm_group_corrs, bins=60, color="#DD8452", alpha=0.60,
        label=f"SRM  (mean={np.mean(srm_group_corrs):.5f})", edgecolor="none")
ax.axvline(float(np.mean(raw_group_corrs)), color="#4C72B0", linewidth=2.0, linestyle="--")
ax.axvline(float(np.mean(srm_group_corrs)), color="#DD8452", linewidth=2.0, linestyle="--")
ax.axvline(0, color="gray", linewidth=1.0)
ax.set_xlabel("Encoding r (Pearson)", fontsize=12)
ax.set_ylabel("Number of voxels", fontsize=12)
ax.set_title("Group-level voxel-wise r: Raw vs SRM", fontsize=13)
ax.set_xlim(CORR_VMIN, CORR_VMAX)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_group_histogram_raw_vs_srm.png", dpi=200)
plt.close()
print("Saved fig4_group_histogram_raw_vs_srm.png")


# =========================
# 10. Save summary JSON
# =========================

total_elapsed = (time.time() - start_total) / 60.0

summary = {
    "dataset": step7c_summary.get("dataset", "21styear"),
    "roi_target": ROI_TARGET,
    "pipeline_type": "step7d_statistical_summary_raw_vs_srm",
    "n_subjects": int(N_SUBJECTS),
    "n_voxels": int(N_VOXELS),
    "source_step7b_summary": str(STEP7B_JSON_PATH),
    "source_step7c_summary": str(STEP7C_JSON_PATH),
    "subject_level_descriptive": {
        "raw": summarize_vector(mean_r_per_subj_raw),
        "srm": summarize_vector(mean_r_per_subj_srm),
        "delta": summarize_vector(delta_indiv),
    },
    "subject_level_inference": {
        "paired_t_stat": float(ttest_res.statistic),
        "paired_t_p": float(ttest_res.pvalue),
        "wilcoxon_stat": float(wilcoxon_stat),
        "wilcoxon_p": float(wilcoxon_p),
        "sign_test_p": float(sign_p),
        "permutation_p": float(perm_p),
        "bootstrap_ci_2p5": float(delta_ci_lo),
        "bootstrap_ci_97p5": float(delta_ci_hi),
        "subjects_improved": int(n_improved),
        "subjects_not_improved": int(N_SUBJECTS - n_improved),
    },
    "voxel_level_descriptive": {
        "mean_across_subjects_raw": summarize_vector(mean_r_across_subjects_raw),
        "mean_across_subjects_srm": summarize_vector(mean_r_across_subjects_srm),
        "mean_across_subjects_delta": summarize_vector(delta_voxelwise_subject_mean),
        "group_raw": summarize_vector(raw_group_corrs),
        "group_srm": summarize_vector(srm_group_corrs),
        "group_delta": summarize_vector(delta_voxelwise_group),
    },
    "outputs": {
        "subject_level_statistics_csv": str(CSV_DIR / "subject_level_statistics_raw_vs_srm.csv"),
        "subject_level_inference_csv": str(CSV_DIR / "subject_level_inference_raw_vs_srm.csv"),
        "voxel_level_statistics_csv": str(CSV_DIR / "voxel_level_statistics_raw_vs_srm.csv"),
        "mean_r_across_subjects_raw": str(NPY_DIR / "mean_r_across_subjects_raw.npy"),
        "mean_r_across_subjects_srm": str(NPY_DIR / "mean_r_across_subjects_srm.npy"),
        "delta_voxelwise_subject_mean": str(NPY_DIR / "delta_voxelwise_subject_mean.npy"),
        "delta_voxelwise_group": str(NPY_DIR / "delta_voxelwise_group.npy"),
    },
    "total_elapsed_minutes": float(total_elapsed),
}

with open(JSON_DIR / "step7d_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved Step 7d summary to: {JSON_DIR / 'step7d_summary.json'}")


# =========================
# 11. Step-end validation checks
# =========================

print("\n" + "=" * 60)
print("STEP-END VALIDATION CHECKS")
print("=" * 60)

subject_shape_ok = subject_stats_df.shape[0] == N_SUBJECTS
inference_shape_ok = subject_inference_df.shape[0] == 1
voxel_shape_ok = voxel_summary_df.shape[0] == 2
range_ok = (
    np.all(raw_voxel_corrs_arr >= -1.0) and np.all(raw_voxel_corrs_arr <= 1.0) and
    np.all(srm_voxel_corrs_arr >= -1.0) and np.all(srm_voxel_corrs_arr <= 1.0)
)
finite_ok = (
    np.isfinite(subject_stats_df.select_dtypes(include=[np.number]).values).all() and
    np.isfinite(subject_inference_df.select_dtypes(include=[np.number]).values).all() and
    np.isfinite(voxel_summary_df.select_dtypes(include=[np.number]).values).all()
)

print(f"subject_shape_ok  : {subject_shape_ok}")
print(f"inference_shape_ok: {inference_shape_ok}")
print(f"voxel_shape_ok    : {voxel_shape_ok}")
print(f"finite_ok         : {finite_ok}")
print(f"range_ok          : {range_ok}")
print(f"mean raw subject r: {np.mean(mean_r_per_subj_raw):.5f}")
print(f"mean srm subject r: {np.mean(mean_r_per_subj_srm):.5f}")
print(f"mean delta        : {np.mean(delta_indiv):.5f}")
print(f"paired t p-value  : {float(ttest_res.pvalue):.6g}")
print(f"wilcoxon p-value  : {float(wilcoxon_p):.6g}")
print(f"sign test p-value : {float(sign_p):.6g}")
print(f"perm p-value      : {float(perm_p):.6g}")
print(f"bootstrap 95% CI  : [{delta_ci_lo:.5f}, {delta_ci_hi:.5f}]")

if not subject_shape_ok:
    raise ValueError("Subject-level statistics table has unexpected number of rows.")
if not inference_shape_ok:
    raise ValueError("Subject-level inference table has unexpected number of rows.")
if not voxel_shape_ok:
    raise ValueError("Voxel-level statistics table has unexpected number of rows.")
if not finite_ok:
    raise ValueError("Non-finite values detected in Step 7d outputs.")
if not range_ok:
    raise ValueError("Correlation values out of [-1, 1] range in Step 7d inputs.")

print(f"\nTotal elapsed: {total_elapsed:.2f} min")
print("\nStep 7d completed successfully.")
# ============================================================================
# Step 7d integrated add-on: one-tailed test for fig3 voxel-wise subject-mean delta
# ============================================================================
# Purpose:
# This block belongs to Step 7d and uses the exact data behind
# fig3_voxel_delta_hist_subjectmean.png:
#   delta_voxelwise_subject_mean = mean_across_subjects(SRM r - Raw r), per voxel
#
# It does not refit the encoding models and does not modify the original Step 7d
# figures or statistical tables. It only adds an independent one-tailed check for
# whether the voxel-wise delta distribution is shifted above zero.
#
# Important interpretation note:
# Voxels are spatially correlated, so this is supporting distribution-level
# evidence. The participant-level paired tests remain the primary inference.

from scipy.stats import ttest_1samp

delta_path_extra = NPY_DIR / "delta_voxelwise_subject_mean.npy"
if not delta_path_extra.exists():
    raise FileNotFoundError(f"Missing fig3 data file: {delta_path_extra}")

delta_extra = np.asarray(np.load(delta_path_extra), dtype=float).ravel()
delta_extra = delta_extra[np.isfinite(delta_extra)]
delta_nonzero_extra = delta_extra[delta_extra != 0]

if delta_extra.size == 0:
    raise ValueError("No finite voxel-wise delta values were found for the one-tailed test.")

def _p_to_stars_step7d_extra(p):
    if not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

try:
    t_res_extra = ttest_1samp(delta_extra, popmean=0.0, alternative="greater")
    t_stat_extra = float(t_res_extra.statistic)
    p_t_greater_extra = float(t_res_extra.pvalue)
except TypeError:
    t_res_extra = ttest_1samp(delta_extra, popmean=0.0)
    t_stat_extra = float(t_res_extra.statistic)
    p_two_extra = float(t_res_extra.pvalue)
    p_t_greater_extra = p_two_extra / 2.0 if t_stat_extra > 0 else 1.0 - (p_two_extra / 2.0)

if delta_nonzero_extra.size > 0:
    try:
        w_res_extra = wilcoxon(delta_nonzero_extra, alternative="greater", zero_method="wilcox")
        wilcoxon_stat_extra = float(w_res_extra.statistic)
        p_wilcoxon_greater_extra = float(w_res_extra.pvalue)
    except Exception:
        wilcoxon_stat_extra = np.nan
        p_wilcoxon_greater_extra = np.nan

    n_positive_nonzero_extra = int(np.sum(delta_nonzero_extra > 0))
    sign_res_extra = binomtest(
        n_positive_nonzero_extra,
        n=int(delta_nonzero_extra.size),
        p=0.5,
        alternative="greater",
    )
    p_sign_greater_extra = float(sign_res_extra.pvalue)
else:
    wilcoxon_stat_extra = np.nan
    p_wilcoxon_greater_extra = np.nan
    n_positive_nonzero_extra = 0
    p_sign_greater_extra = np.nan

mean_delta_extra = float(np.mean(delta_extra))
std_delta_extra = float(np.std(delta_extra, ddof=1)) if delta_extra.size > 1 else np.nan
cohens_d_extra = (
    float(mean_delta_extra / std_delta_extra)
    if np.isfinite(std_delta_extra) and std_delta_extra > 0
    else np.nan
)

fig3_one_tailed_summary = {
    "analysis_name": "fig3_voxel_delta_hist_subjectmean_one_tailed_test",
    "tested_data_file": str(delta_path_extra),
    "null_hypothesis": "voxel-wise subject-mean delta r <= 0",
    "alternative_hypothesis": "voxel-wise subject-mean delta r > 0",
    "important_note": "Voxels are spatially correlated, so this is supporting distribution-level evidence rather than the primary subject-level inference.",
    "n_voxels_finite": int(delta_extra.size),
    "n_voxels_nonzero": int(delta_nonzero_extra.size),
    "mean_delta_r": mean_delta_extra,
    "median_delta_r": float(np.median(delta_extra)),
    "std_delta_r": std_delta_extra,
    "min_delta_r": float(np.min(delta_extra)),
    "max_delta_r": float(np.max(delta_extra)),
    "positive_voxel_fraction": float(np.mean(delta_extra > 0)),
    "positive_nonzero_voxel_fraction": (
        float(n_positive_nonzero_extra / delta_nonzero_extra.size)
        if delta_nonzero_extra.size > 0
        else np.nan
    ),
    "one_sample_t_greater_statistic": t_stat_extra,
    "one_sample_t_greater_p": p_t_greater_extra,
    "one_sample_t_greater_sig": _p_to_stars_step7d_extra(p_t_greater_extra),
    "wilcoxon_greater_statistic": wilcoxon_stat_extra,
    "wilcoxon_greater_p": p_wilcoxon_greater_extra,
    "wilcoxon_greater_sig": _p_to_stars_step7d_extra(p_wilcoxon_greater_extra),
    "sign_test_positive_greater_p": p_sign_greater_extra,
    "sign_test_positive_greater_sig": _p_to_stars_step7d_extra(p_sign_greater_extra),
    "cohens_d_against_zero": cohens_d_extra,
    "how_to_read": "Positive and significant values indicate that the voxel-wise delta distribution in fig3 is shifted above zero, meaning SRM-reconstructed targets tend to improve encoding r relative to raw targets.",
}

fig3_one_tailed_df = pd.DataFrame([fig3_one_tailed_summary])
fig3_one_tailed_csv = CSV_DIR / "fig3_voxel_delta_hist_subjectmean_one_tailed_test.csv"
fig3_one_tailed_json = JSON_DIR / "fig3_voxel_delta_hist_subjectmean_one_tailed_test.json"
fig3_one_tailed_df.to_csv(
    fig3_one_tailed_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.5f",
)
with open(fig3_one_tailed_json, "w", encoding="utf-8") as f:
    json.dump(fig3_one_tailed_summary, f, indent=2, ensure_ascii=False)

print("\nStep 7d integrated add-on completed: one-tailed voxel-wise delta test")
print(f"Saved CSV : {fig3_one_tailed_csv}")
print(f"Saved JSON: {fig3_one_tailed_json}")
display(fig3_one_tailed_df)


step8_brain_mapping_nilearn_glassbrain

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from nilearn import image

from matplotlib.ticker import FormatStrFormatter
from nilearn.plotting import plot_glass_brain, plot_stat_map


"""
Step 8: Brain Mapping with Nilearn
==================================
This step visualizes the final encoding-model outputs in brain space.

Main goals:
    1. Show group-level raw observed BOLD signal strength.
    2. Show group-level LLM-predicted BOLD signal strength.
    3. Show group-level LLM + SRM predicted BOLD signal strength.

In addition, to make the improvement pattern more interpretable, this step
also visualizes voxel-wise encoding performance maps:
    - raw encoding r
    - SRM encoding r
    - delta r (SRM - raw)

All maps are reconstructed back into the retained ROI voxel space and then
embedded into a NIfTI image for nilearn-based plotting.
"""


# =========================
# 1. Project setup
# =========================

DATA_ROOT = Path(r"CHANGE_THIS_TO_YOUR_DATA_ROOT")
BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
ROI_NAME = "CHANGE_THIS_TO_YOUR_ROI_NAME"
ROI_TARGET = f"{ROI_NAME}_only"
ENCODING_RUN_NAME = f"{ROI_NAME}_UBSN_ROI_encoding_cleaned"
RUN_NAME = f"{ROI_NAME}_400_parcels_cleaned"

ENC_PROJECT_DIR = BASE_DIR / "encoding_runs" / ENCODING_RUN_NAME
SRM_PROJECT_DIR = BASE_DIR / "runs"
STEP6_DIR = ENC_PROJECT_DIR / "step6_prepare_fmri_response"
STEP7B_DIR = ENC_PROJECT_DIR / "step7b_raw_bold_encoding"
STEP7C_DIR = ENC_PROJECT_DIR / "step7c_srm_reconstructed_encoding"
STEP8_DIR = ENC_PROJECT_DIR / "step8_brain_mapping_nilearn"

SRM_STEP1_DIR = SRM_PROJECT_DIR / RUN_NAME / "step1_setup_roi_cleaned"
SRM_STEP2_DIR = SRM_PROJECT_DIR / RUN_NAME / "step2_split_zscore"

FIG_DIR = STEP8_DIR / "figures"
CSV_DIR = STEP8_DIR / "csv"
NPY_DIR = STEP8_DIR / "npy"
NII_DIR = STEP8_DIR / "nii"
JSON_DIR = STEP8_DIR / "json"

for d in [STEP8_DIR, FIG_DIR, CSV_DIR, NPY_DIR, NII_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(__doc__)
print(f"Encoding project directory: {ENC_PROJECT_DIR}")
print(f"Step 8 directory: {STEP8_DIR}")


# =========================
# 2. Load inputs
# =========================

Y_TEST_GROUP_PATH = STEP6_DIR / "npy" / "Y_test_group_huthstyle.npy"
RAW_GROUP_PRED_PATH = STEP7B_DIR / "npy" / "group_Y_pred_raw.npy"
SRM_GROUP_PRED_PATH = STEP7C_DIR / "npy" / "group_Y_pred_srm.npy"
RAW_GROUP_CORR_PATH = STEP7B_DIR / "npy" / "group_voxel_corrs_raw.npy"
SRM_GROUP_CORR_PATH = STEP7C_DIR / "npy" / "group_voxel_corrs_srm.npy"
RAW_MEAN_SUBJ_CORR_PATH = STEP7B_DIR / "npy" / "mean_r_across_subjects_raw.npy"
SRM_MEAN_SUBJ_CORR_PATH = STEP7C_DIR / "npy" / "mean_r_across_subjects_srm.npy"

STEP1_CONFIG_PATH = SRM_STEP1_DIR / "step1_config.json"
MANIFEST_PATH = SRM_STEP1_DIR / "csv" / "subject_file_manifest.csv"
SUBJECTS_RETAINED_PATH = SRM_STEP1_DIR / "csv" / "subjects_retained.csv"
ROI_MASK_ATLAS_PATH = SRM_STEP1_DIR / "selected_schaefer400_roi_mask.nii.gz"

for p in [
    Y_TEST_GROUP_PATH,
    RAW_GROUP_PRED_PATH,
    SRM_GROUP_PRED_PATH,
    RAW_GROUP_CORR_PATH,
    SRM_GROUP_CORR_PATH,
    RAW_MEAN_SUBJ_CORR_PATH,
    SRM_MEAN_SUBJ_CORR_PATH,
    STEP1_CONFIG_PATH,
    MANIFEST_PATH,
    SUBJECTS_RETAINED_PATH,
    ROI_MASK_ATLAS_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

start_total = time.time()

Y_test_group = np.load(Y_TEST_GROUP_PATH).astype(np.float32)
Y_pred_raw = np.load(RAW_GROUP_PRED_PATH).astype(np.float32)
Y_pred_srm = np.load(SRM_GROUP_PRED_PATH).astype(np.float32)
group_corr_raw = np.load(RAW_GROUP_CORR_PATH).astype(np.float32)
group_corr_srm = np.load(SRM_GROUP_CORR_PATH).astype(np.float32)
mean_corr_raw = np.load(RAW_MEAN_SUBJ_CORR_PATH).astype(np.float32)
mean_corr_srm = np.load(SRM_MEAN_SUBJ_CORR_PATH).astype(np.float32)

with open(STEP1_CONFIG_PATH, "r", encoding="utf-8") as f:
    step1_config = json.load(f)

manifest_df = pd.read_csv(MANIFEST_PATH)
subjects_retained = pd.read_csv(SUBJECTS_RETAINED_PATH)["subject"].tolist()

print("\nLoaded arrays:")
print(f"Y_test_group shape: {Y_test_group.shape}")
print(f"Y_pred_raw shape:   {Y_pred_raw.shape}")
print(f"Y_pred_srm shape:   {Y_pred_srm.shape}")
print(f"group_corr_raw shape: {group_corr_raw.shape}")
print(f"group_corr_srm shape: {group_corr_srm.shape}")

N_TIME, N_VOXELS = Y_test_group.shape

if Y_pred_raw.shape != Y_test_group.shape:
    raise ValueError(f"Y_pred_raw shape mismatch: {Y_pred_raw.shape} vs {Y_test_group.shape}")
if Y_pred_srm.shape != Y_test_group.shape:
    raise ValueError(f"Y_pred_srm shape mismatch: {Y_pred_srm.shape} vs {Y_test_group.shape}")
if len(group_corr_raw) != N_VOXELS or len(group_corr_srm) != N_VOXELS:
    raise ValueError("Voxelwise correlation map length does not match N_VOXELS.")


# =========================
# 3. Rebuild the retained-voxel brain mask
# =========================

# We reconstruct the exact ROI voxel ordering by repeating the same atlas-to-BOLD
# resampling logic used in the SRM pipeline on one retained subject. Because all
# retained subjects share the common voxel count (1058), this gives us a brain-space
# container into which voxelwise arrays can be written back.

example_subject = subjects_retained[0]
example_bold_path = Path(
    manifest_df.loc[manifest_df["subject"] == example_subject, "bold_path"].iloc[0]
)

roi_mask_atlas_nii = nib.load(str(ROI_MASK_ATLAS_PATH))
bold_nii = nib.load(str(example_bold_path))
bold_data = bold_nii.get_fdata(dtype=np.float32)

story_start_tr = int(step1_config["story_start_tr"])
story_end_tr = int(step1_config["story_end_tr"])
bold_crop = bold_data[..., story_start_tr:story_end_tr]
bold_ref_nii = nib.Nifti1Image(bold_crop[..., 0], bold_nii.affine, bold_nii.header)

roi_resampled_nii = image.resample_to_img(
    roi_mask_atlas_nii,
    bold_ref_nii,
    interpolation="nearest",
    force_resample=True,
    copy_header=True,
)
roi_resampled_mask = roi_resampled_nii.get_fdata() > 0

mask_voxel_count = int(np.sum(roi_resampled_mask))
print(f"\nExample subject for mask reconstruction: {example_subject}")
print(f"Example BOLD path: {example_bold_path}")
print(f"Resampled ROI voxel count: {mask_voxel_count}")

if mask_voxel_count != N_VOXELS:
    raise ValueError(
        f"Resampled mask voxel count ({mask_voxel_count}) does not match expected N_VOXELS ({N_VOXELS})."
    )


# =========================
# 4. Build voxelwise summary maps
# =========================

# Because responses are z-scored over time, temporal mean is not informative.
# We therefore use RMS over time as a signal-strength map for observed/predicted BOLD.

raw_bold_rms = np.sqrt(np.mean(Y_test_group ** 2, axis=0)).astype(np.float32)
raw_pred_rms = np.sqrt(np.mean(Y_pred_raw ** 2, axis=0)).astype(np.float32)
srm_pred_rms = np.sqrt(np.mean(Y_pred_srm ** 2, axis=0)).astype(np.float32)

delta_pred_rms = srm_pred_rms - raw_pred_rms
delta_corr_group = group_corr_srm - group_corr_raw
delta_corr_subject_mean = mean_corr_srm - mean_corr_raw


# =========================
# 5. Helper: write voxel arrays back to NIfTI
# =========================

def vector_to_nifti(values: np.ndarray, mask_bool: np.ndarray, ref_nii: nib.Nifti1Image) -> nib.Nifti1Image:
    values = np.asarray(values, dtype=np.float32).ravel()
    if values.shape[0] != int(np.sum(mask_bool)):
        raise ValueError(
            f"Vector length {values.shape[0]} does not match mask voxel count {int(np.sum(mask_bool))}."
        )
    out = np.zeros(mask_bool.shape, dtype=np.float32)
    out[mask_bool] = values
    return nib.Nifti1Image(out, ref_nii.affine, ref_nii.header)


# =========================
# 6. Create and save NIfTI maps
# =========================

maps = {
    "raw_bold_rms": raw_bold_rms,
    "llm_predicted_bold_rms": raw_pred_rms,
    "llm_srm_predicted_bold_rms": srm_pred_rms,
    "delta_predicted_rms_srm_minus_raw": delta_pred_rms,
    "raw_encoding_r_group": group_corr_raw,
    "srm_encoding_r_group": group_corr_srm,
    "delta_encoding_r_group_srm_minus_raw": delta_corr_group,
    "raw_encoding_r_mean_across_subjects": mean_corr_raw,
    "srm_encoding_r_mean_across_subjects": mean_corr_srm,
    "delta_encoding_r_mean_across_subjects_srm_minus_raw": delta_corr_subject_mean,
}

nii_paths = {}
for map_name, vec in maps.items():
    nii = vector_to_nifti(vec, roi_resampled_mask, bold_ref_nii)
    out_path = NII_DIR / f"{map_name}.nii.gz"
    nib.save(nii, out_path)
    nii_paths[map_name] = str(out_path)

print("\nSaved NIfTI maps:")
for k, v in nii_paths.items():
    print(f"  {k}: {v}")


# =========================
# 7. Common plotting ranges
# =========================

signal_vmin = 0.0
signal_vmax = float(max(raw_bold_rms.max(), raw_pred_rms.max(), srm_pred_rms.max()))
raw_signal_vmin = 0.0
raw_signal_vmax = float(raw_bold_rms.max())
pred_signal_vmin = 0.0
pred_signal_vmax = float(max(raw_pred_rms.max(), srm_pred_rms.max()))
corr_vmin = 0.0
corr_vmax = float(max(group_corr_raw.max(), group_corr_srm.max(), mean_corr_raw.max(), mean_corr_srm.max()))

delta_signal_absmax = float(max(np.abs(delta_pred_rms.min()), np.abs(delta_pred_rms.max())))
delta_corr_absmax = float(max(
    np.abs(delta_corr_group.min()), np.abs(delta_corr_group.max()),
    np.abs(delta_corr_subject_mean.min()), np.abs(delta_corr_subject_mean.max())
))

def robust_threshold(x: np.ndarray, percentile: float) -> float:
    x = np.asarray(x, dtype=np.float64).ravel()
    x = np.abs(x[np.isfinite(x)])
    x = x[x > 0]
    if x.size == 0:
        return 0.0
    return float(np.percentile(x, percentile))

signal_threshold = robust_threshold(
    np.concatenate([raw_bold_rms, raw_pred_rms, srm_pred_rms]),
    percentile=80,
)
raw_signal_threshold = robust_threshold(raw_bold_rms, percentile=80)
pred_signal_threshold = robust_threshold(
    np.concatenate([raw_pred_rms, srm_pred_rms]),
    percentile=60,
)
corr_threshold = robust_threshold(
    np.concatenate([group_corr_raw, group_corr_srm, mean_corr_raw, mean_corr_srm]),
    percentile=75,
)
delta_signal_threshold = robust_threshold(delta_pred_rms, percentile=80)
delta_corr_threshold = robust_threshold(
    np.concatenate([delta_corr_group, delta_corr_subject_mean]),
    percentile=75,
)

signal_summary_df = pd.DataFrame({
    "map_type": [
        "signal_strength_combined",
        "signal_strength_raw_only",
        "signal_strength_predicted_only",
        "encoding_r",
        "delta_signal",
        "delta_encoding_r",
    ],
    "vmin": [signal_vmin, raw_signal_vmin, pred_signal_vmin, corr_vmin, -delta_signal_absmax, -delta_corr_absmax],
    "vmax": [signal_vmax, raw_signal_vmax, pred_signal_vmax, corr_vmax, delta_signal_absmax, delta_corr_absmax],
    "threshold": [signal_threshold, raw_signal_threshold, pred_signal_threshold, corr_threshold, delta_signal_threshold, delta_corr_threshold],
})
signal_summary_df.to_csv(CSV_DIR / "step8_colorbar_ranges.csv", index=False, encoding="utf-8-sig")

print("\nUnified plotting ranges:")
print(signal_summary_df.to_string(index=False))


# =========================
# 8. Main glass-brain figures requested by user
# =========================

main_maps = [
    ("raw_bold_rms", "Group raw BOLD signal strength (RMS)", "viridis", raw_signal_vmin, raw_signal_vmax, raw_signal_threshold),
    ("llm_predicted_bold_rms", "Group LLM-predicted BOLD signal strength (RMS)", "viridis", pred_signal_vmin, pred_signal_vmax, pred_signal_threshold),
    ("llm_srm_predicted_bold_rms", "Group LLM + SRM predicted BOLD signal strength (RMS)", "viridis", pred_signal_vmin, pred_signal_vmax, pred_signal_threshold),
]

for map_name, title, cmap, vmin, vmax, threshold in main_maps:
    nii = nib.load(nii_paths[map_name])
    disp = plot_glass_brain(
        nii,
        display_mode="lyrz",
        colorbar=True,
        cmap=cmap,
        plot_abs=False,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        title=title,
    )
    disp.savefig(FIG_DIR / f"glass_{map_name}.png", dpi=200)
    plt.close()
    print(f"Saved glass_{map_name}.png")


# =========================
# 9. Additional intuitive maps
# =========================

extra_maps = [
    ("raw_encoding_r_group", "Raw encoding performance (group voxel-wise r)", "magma", corr_vmin, corr_vmax, corr_threshold),
    ("srm_encoding_r_group", "SRM encoding performance (group voxel-wise r)", "magma", corr_vmin, corr_vmax, corr_threshold),
    ("delta_encoding_r_group_srm_minus_raw", "Encoding improvement map (SRM - Raw)", "coolwarm", -delta_corr_absmax, delta_corr_absmax, delta_corr_threshold),
    ("delta_predicted_rms_srm_minus_raw", "Predicted signal-strength delta (SRM - Raw)", "coolwarm", -delta_signal_absmax, delta_signal_absmax, delta_signal_threshold),
]

for map_name, title, cmap, vmin, vmax, threshold in extra_maps:
    nii = nib.load(nii_paths[map_name])
    disp = plot_glass_brain(
        nii,
        display_mode="lyrz",
        colorbar=True,
        cmap=cmap,
        plot_abs=False,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        title=title,
    )
    disp.savefig(FIG_DIR / f"glass_{map_name}.png", dpi=200)
    plt.close()
    print(f"Saved glass_{map_name}.png")


# =========================
# 10. Slice-based stat maps for closer inspection
# =========================

slice_maps = [
    ("raw_encoding_r_group", "Slice view: Raw encoding r", "magma", corr_vmin, corr_vmax, corr_threshold),
    ("srm_encoding_r_group", "Slice view: SRM encoding r", "magma", corr_vmin, corr_vmax, corr_threshold),
    ("delta_encoding_r_group_srm_minus_raw", "Slice view: SRM - Raw encoding r", "coolwarm", -delta_corr_absmax, delta_corr_absmax, delta_corr_threshold),
]

for map_name, title, cmap, vmin, vmax, threshold in slice_maps:
    nii = nib.load(nii_paths[map_name])
    disp = plot_stat_map(
        nii,
        display_mode="z",
        cut_coords=10,
        colorbar=True,
        cmap=cmap,
        threshold=threshold,
        vmax=vmax,
        title=title,
    )
    disp.savefig(FIG_DIR / f"slice_{map_name}.png", dpi=200)
    plt.close()
    print(f"Saved slice_{map_name}.png")


# =========================
# 11. Simple distribution summaries
# =========================

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(group_corr_raw, bins=60, color="#4C72B0", alpha=0.65, edgecolor="none", label="Raw")
axes[0].hist(group_corr_srm, bins=60, color="#DD8452", alpha=0.65, edgecolor="none", label="SRM")
axes[0].axvline(float(np.mean(group_corr_raw)), color="#4C72B0", linestyle="--", linewidth=2)
axes[0].axvline(float(np.mean(group_corr_srm)), color="#DD8452", linestyle="--", linewidth=2)
axes[0].axvline(0, color="gray", linewidth=1)
axes[0].set_title("Group voxel-wise encoding r")
axes[0].set_xlabel("Encoding r")
axes[0].set_ylabel("Number of voxels")
axes[0].legend(frameon=False)
axes[0].set_xlim(corr_vmin, corr_vmax)

axes[1].hist(delta_corr_group, bins=60, color="#C44E52", alpha=0.75, edgecolor="none")
axes[1].axvline(0, color="gray", linewidth=1)
axes[1].axvline(float(np.mean(delta_corr_group)), color="crimson", linestyle="--", linewidth=2,
                label=f"Mean delta = {np.mean(delta_corr_group):.5f}")
axes[1].set_title("Group voxel-wise delta r (SRM - Raw)")
axes[1].set_xlabel("Delta encoding r")
axes[1].set_ylabel("Number of voxels")
axes[1].legend(frameon=False)
axes[1].set_xlim(-delta_corr_absmax, delta_corr_absmax)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_distribution_summaries_step8.png", dpi=200)
plt.close()
print("Saved fig_distribution_summaries_step8.png")


# =========================
# 12. Save summary JSON
# =========================

total_elapsed = (time.time() - start_total) / 60.0

summary = {
    "dataset": "21styear",
    "roi_target": ROI_TARGET,
    "pipeline_type": "step8_brain_mapping_nilearn",
    "n_timepoints": int(N_TIME),
    "n_voxels": int(N_VOXELS),
    "example_subject_for_mask": example_subject,
    "example_bold_path": str(example_bold_path),
    "mask_voxel_count": int(mask_voxel_count),
    "colorbar_ranges": {
        "signal_strength_combined": {"vmin": float(signal_vmin), "vmax": float(signal_vmax)},
        "signal_strength_raw_only": {"vmin": float(raw_signal_vmin), "vmax": float(raw_signal_vmax)},
        "signal_strength_predicted_only": {"vmin": float(pred_signal_vmin), "vmax": float(pred_signal_vmax)},
        "encoding_r": {"vmin": float(corr_vmin), "vmax": float(corr_vmax)},
        "delta_signal": {"vmin": float(-delta_signal_absmax), "vmax": float(delta_signal_absmax)},
        "delta_encoding_r": {"vmin": float(-delta_corr_absmax), "vmax": float(delta_corr_absmax)},
    },
    "map_summaries": {
        "raw_bold_rms_mean": float(np.mean(raw_bold_rms)),
        "raw_pred_rms_mean": float(np.mean(raw_pred_rms)),
        "srm_pred_rms_mean": float(np.mean(srm_pred_rms)),
        "raw_group_corr_mean": float(np.mean(group_corr_raw)),
        "srm_group_corr_mean": float(np.mean(group_corr_srm)),
        "delta_group_corr_mean": float(np.mean(delta_corr_group)),
    },
    "outputs": {
        "nii_maps": nii_paths,
        "colorbar_ranges_csv": str(CSV_DIR / "step8_colorbar_ranges.csv"),
    },
    "total_elapsed_minutes": float(total_elapsed),
}

with open(JSON_DIR / "step8_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved Step 8 summary to: {JSON_DIR / 'step8_summary.json'}")


# =========================
# 13. Step-end validation checks
# =========================

print("\n" + "=" * 60)
print("STEP-END VALIDATION CHECKS")
print("=" * 60)

finite_signal_ok = (
    np.isfinite(raw_bold_rms).all() and
    np.isfinite(raw_pred_rms).all() and
    np.isfinite(srm_pred_rms).all()
)
finite_corr_ok = (
    np.isfinite(group_corr_raw).all() and
    np.isfinite(group_corr_srm).all() and
    np.isfinite(mean_corr_raw).all() and
    np.isfinite(mean_corr_srm).all()
)
shape_ok = (
    len(raw_bold_rms) == N_VOXELS and
    len(raw_pred_rms) == N_VOXELS and
    len(srm_pred_rms) == N_VOXELS
)
nii_count_ok = len(nii_paths) == len(maps)

print(f"finite_signal_ok: {finite_signal_ok}")
print(f"finite_corr_ok  : {finite_corr_ok}")
print(f"shape_ok        : {shape_ok}")
print(f"nii_count_ok    : {nii_count_ok}  ({len(nii_paths)} maps)")
print(f"signal range    : [{signal_vmin:.5f}, {signal_vmax:.5f}]")
print(f"corr range      : [{corr_vmin:.5f}, {corr_vmax:.5f}]")
print(f"delta corr mean : {np.mean(delta_corr_group):.5f}")

if not finite_signal_ok:
    raise ValueError("Non-finite values detected in signal-strength maps.")
if not finite_corr_ok:
    raise ValueError("Non-finite values detected in correlation maps.")
if not shape_ok:
    raise ValueError("Voxelwise map shape mismatch in Step 8.")
if not nii_count_ok:
    raise ValueError("Unexpected number of NIfTI outputs in Step 8.")

print(f"\nTotal elapsed: {total_elapsed:.2f} min")
print("\nStep 8 completed successfully.")
# ============================================================================
# Step 8 integrated add-on: X/Y/Z directional brain slice figures
# ============================================================================
# Purpose:
# This block belongs to Step 8 and adds a separate set of direction-specific
# anatomical views from the already saved Step 8 NIfTI maps. It keeps all
# existing Step 8 glass-brain and slice figures unchanged.
#
# X direction = sagittal slices
# Y direction = coronal slices
# Z direction = axial / horizontal slices
#
# Raw and SRM-reconstructed encoding maps use the same color scale. Delta maps
# use a symmetric positive/negative color scale.

from nilearn.plotting import plot_stat_map

FIG_XYZ_DIR = STEP8_DIR / "figures_xyz_directional_slices"
FIG_XYZ_DIR.mkdir(parents=True, exist_ok=True)

xyz_maps = [
    {
        "map_id": "raw_encoding_r_group",
        "path": NII_DIR / "raw_encoding_r_group.nii.gz",
        "title": "Raw voxel-space encoding r",
        "cmap": "viridis",
        "scale_group": "encoding_r",
        "symmetric": False,
        "interpretation": "Baseline LLM encoding performance for raw voxel-space responses.",
    },
    {
        "map_id": "srm_encoding_r_group",
        "path": NII_DIR / "srm_encoding_r_group.nii.gz",
        "title": "SRM-reconstructed voxel-space encoding r",
        "cmap": "viridis",
        "scale_group": "encoding_r",
        "symmetric": False,
        "interpretation": "LLM encoding performance for SRM-reconstructed voxel-space responses.",
    },
    {
        "map_id": "delta_encoding_r_group_srm_minus_raw",
        "path": NII_DIR / "delta_encoding_r_group_srm_minus_raw.nii.gz",
        "title": "Delta r: SRM-reconstructed minus raw",
        "cmap": "cold_hot",
        "scale_group": "delta_r",
        "symmetric": True,
        "interpretation": "Positive values indicate stronger encoding after SRM reconstruction.",
    },
]

for xyz_map in xyz_maps:
    if not xyz_map["path"].exists():
        raise FileNotFoundError(f"Missing Step 8 NIfTI map: {xyz_map['path']}")

def _finite_nonzero_values_step8_xyz(img_path):
    data = np.asanyarray(nib.load(str(img_path)).get_fdata(), dtype=float)

    # Ignore tiny background pseudo-nonzero values caused by NIfTI scaling/header handling.
    valid = np.isfinite(data) & (np.abs(data) > 1e-5)
    return data[valid]

encoding_vals_xyz = []
for xyz_map in xyz_maps:
    if xyz_map["scale_group"] == "encoding_r":
        vals = _finite_nonzero_values_step8_xyz(xyz_map["path"])
        if vals.size > 0:
            encoding_vals_xyz.append(vals)

if encoding_vals_xyz:
    encoding_all_xyz = np.concatenate(encoding_vals_xyz)
    encoding_all_xyz = encoding_all_xyz[np.isfinite(encoding_all_xyz)]
    encoding_positive_xyz = encoding_all_xyz[encoding_all_xyz > 0]

    if encoding_positive_xyz.size > 0:
        shared_encoding_vmax_xyz = float(np.nanpercentile(encoding_positive_xyz, 99))
    elif encoding_all_xyz.size > 0:
        shared_encoding_vmax_xyz = float(np.nanpercentile(np.abs(encoding_all_xyz), 99))
    else:
        shared_encoding_vmax_xyz = 0.00001
else:
    shared_encoding_vmax_xyz = 0.00001

if (not np.isfinite(shared_encoding_vmax_xyz)) or shared_encoding_vmax_xyz <= 0:
    shared_encoding_vmax_xyz = 0.00001


delta_xyz_path = [m for m in xyz_maps if m["scale_group"] == "delta_r"][0]["path"]
delta_vals_xyz = _finite_nonzero_values_step8_xyz(delta_xyz_path)
delta_vals_xyz = delta_vals_xyz[np.isfinite(delta_vals_xyz)]

if delta_vals_xyz.size > 0:
    delta_abs_vmax_xyz = float(np.nanpercentile(np.abs(delta_vals_xyz), 99))
else:
    delta_abs_vmax_xyz = 0.00001

if (not np.isfinite(delta_abs_vmax_xyz)) or delta_abs_vmax_xyz <= 0:
    delta_abs_vmax_xyz = 0.00001

def _format_colorbar_decimal_step8_xyz(disp):
    try:
        disp._cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.5f"))
        disp._cbar.ax.yaxis.offsetText.set_visible(False)
        disp._cbar.update_ticks()
    except Exception:
        pass

def _make_directional_slice_step8_xyz(
    img_path,
    out_path,
    title,
    display_mode,
    cmap,
    vmin,
    vmax,
    symmetric_cbar=False,
):
    disp = plot_stat_map(
    str(img_path),
    display_mode=display_mode,
    cut_coords=7,
    colorbar=True,
    cmap=cmap,
    threshold=1e-5,
    vmin=vmin,
    vmax=vmax,
    black_bg=False,
    symmetric_cbar=False,
    cbar_tick_format="%.5f",
    title=title,
)
    _format_colorbar_decimal_step8_xyz(disp)
    disp.savefig(out_path, dpi=260, bbox_inches="tight", pad_inches=0.05)
    plt.close()

directions_xyz = [
    {"axis": "x", "view_name": "sagittal", "display_mode": "x"},
    {"axis": "y", "view_name": "coronal", "display_mode": "y"},
    {"axis": "z", "view_name": "axial_horizontal", "display_mode": "z"},
]

xyz_rows = []
for xyz_map in xyz_maps:
    if xyz_map["scale_group"] == "encoding_r":
        vmin_xyz, vmax_xyz = 0.0, shared_encoding_vmax_xyz
        scale_note_xyz = "shared_raw_and_srm_encoding_scale"
    else:
        vmin_xyz, vmax_xyz = -delta_abs_vmax_xyz, delta_abs_vmax_xyz
        scale_note_xyz = "symmetric_delta_scale"

    for direction_xyz in directions_xyz:
        out_path_xyz = (
            FIG_XYZ_DIR
            / f"{xyz_map['map_id']}_{direction_xyz['axis']}_{direction_xyz['view_name']}.png"
        )
        fig_title_xyz = (
            f"{xyz_map['title']} "
            f"({direction_xyz['view_name'].replace('_', ' ')})"
        )
        _make_directional_slice_step8_xyz(
            img_path=xyz_map["path"],
            out_path=out_path_xyz,
            title=fig_title_xyz,
            display_mode=direction_xyz["display_mode"],
            cmap=xyz_map["cmap"],
            vmin=vmin_xyz,
            vmax=vmax_xyz,
            symmetric_cbar=False,
        )
        xyz_rows.append({
            "map_id": xyz_map["map_id"],
            "nifti_path": str(xyz_map["path"]),
            "axis": direction_xyz["axis"],
            "view_name": direction_xyz["view_name"],
            "figure_path": str(out_path_xyz),
            "vmin": float(vmin_xyz),
            "vmax": float(vmax_xyz),
            "colorbar_format": "decimal_5_places",
            "scale_note": scale_note_xyz,
            "interpretation": xyz_map["interpretation"],
        })

xyz_manifest_df = pd.DataFrame(xyz_rows)
xyz_manifest_csv = CSV_DIR / "step8_xyz_directional_slice_manifest.csv"
xyz_manifest_json = JSON_DIR / "step8_xyz_directional_slice_manifest.json"
xyz_manifest_df.to_csv(
    xyz_manifest_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.5f",
)
with open(xyz_manifest_json, "w", encoding="utf-8") as f:
    json.dump(xyz_rows, f, indent=2, ensure_ascii=False)

print("\nStep 8 integrated add-on completed: X/Y/Z directional brain slices")
print(f"Saved figures to: {FIG_XYZ_DIR}")
print(f"Saved manifest  : {xyz_manifest_csv}")
display(xyz_manifest_df)
# ============================================================================
# Step 8 integrated add-on: compact orthogonal summary brain figures
# ============================================================================
# Purpose:
# In addition to the existing multi-slice X/Y/Z figures, this block creates
# compact orthogonal summary figures. Each figure shows one sagittal, one
# coronal, and one axial/horizontal view in a single panel, similar to a
# paper-style overview brain map.
#
# This block only reads the already saved Step 8 NIfTI maps and writes new
# figures. It does not change any existing figures, paths, or model outputs.

from nilearn.plotting import plot_stat_map, find_xyz_cut_coords

FIG_ORTHO_DIR = STEP8_DIR / "figures_ortho_summary"
FIG_ORTHO_DIR.mkdir(parents=True, exist_ok=True)

ortho_maps = [
    {
        "map_id": "raw_encoding_r_group",
        "path": NII_DIR / "raw_encoding_r_group.nii.gz",
        "title": "Raw voxel-space encoding r",
        "cmap": "viridis",
        "scale_group": "encoding_r",
        "symmetric": False,
        "interpretation": "Compact orthogonal view of baseline LLM encoding performance for raw voxel-space responses.",
    },
    {
        "map_id": "srm_encoding_r_group",
        "path": NII_DIR / "srm_encoding_r_group.nii.gz",
        "title": "SRM-reconstructed voxel-space encoding r",
        "cmap": "viridis",
        "scale_group": "encoding_r",
        "symmetric": False,
        "interpretation": "Compact orthogonal view of LLM encoding performance for SRM-reconstructed voxel-space responses.",
    },
    {
        "map_id": "delta_encoding_r_group_srm_minus_raw",
        "path": NII_DIR / "delta_encoding_r_group_srm_minus_raw.nii.gz",
        "title": "Delta r: SRM-reconstructed minus raw",
        "cmap": "cold_hot",
        "scale_group": "delta_r",
        "symmetric": True,
        "interpretation": "Compact orthogonal view of the SRM-minus-raw encoding contrast.",
    },
]

for ortho_map in ortho_maps:
    if not ortho_map["path"].exists():
        raise FileNotFoundError(f"Missing Step 8 NIfTI map: {ortho_map['path']}")

def _finite_nonzero_values_step8_ortho(img_path):
    data = np.asanyarray(nib.load(str(img_path)).get_fdata(), dtype=float)

    # Important:
    # Background voxels can become tiny nonzero values after NIfTI scaling/header handling.
    # Therefore we must not use data != 0. We keep only meaningful ROI-level values.
    valid = np.isfinite(data) & (np.abs(data) > 1e-5)
    return data[valid]

encoding_vals_ortho = []
for ortho_map in ortho_maps:
    if ortho_map["scale_group"] == "encoding_r":
        vals = _finite_nonzero_values_step8_ortho(ortho_map["path"])
        if vals.size > 0:
            encoding_vals_ortho.append(vals)

if encoding_vals_ortho:
    encoding_all_ortho = np.concatenate(encoding_vals_ortho)
    encoding_all_ortho = encoding_all_ortho[np.isfinite(encoding_all_ortho)]
    encoding_positive_ortho = encoding_all_ortho[encoding_all_ortho > 0]
    if encoding_positive_ortho.size > 0:
        shared_encoding_vmax_ortho = float(np.nanpercentile(encoding_positive_ortho, 99))
    elif encoding_all_ortho.size > 0:
        shared_encoding_vmax_ortho = float(np.nanpercentile(np.abs(encoding_all_ortho), 99))
    else:
        shared_encoding_vmax_ortho = 0.00001
else:
    shared_encoding_vmax_ortho = 0.00001

if (not np.isfinite(shared_encoding_vmax_ortho)) or shared_encoding_vmax_ortho <= 0:
    shared_encoding_vmax_ortho = 0.00001

delta_ortho_path = [m for m in ortho_maps if m["scale_group"] == "delta_r"][0]["path"]
delta_vals_ortho = _finite_nonzero_values_step8_ortho(delta_ortho_path)
delta_vals_ortho = delta_vals_ortho[np.isfinite(delta_vals_ortho)]
if delta_vals_ortho.size > 0:
    delta_abs_vmax_ortho = float(np.nanpercentile(np.abs(delta_vals_ortho), 99))
else:
    delta_abs_vmax_ortho = 0.00001

if (not np.isfinite(delta_abs_vmax_ortho)) or delta_abs_vmax_ortho <= 0:
    delta_abs_vmax_ortho = 0.00001

def _format_colorbar_decimal_step8_ortho(disp):
    try:
        disp._cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.5f"))
        disp._cbar.ax.yaxis.offsetText.set_visible(False)
        disp._cbar.update_ticks()
    except Exception:
        pass

def _safe_cut_coords_step8_ortho(img_path):
    img = nib.load(str(img_path))
    data = np.asarray(img.get_fdata(), dtype=float)

    # Same logic as above: ignore tiny background pseudo-nonzero values.
    finite_mask = np.isfinite(data)
    valid_mask = finite_mask & (np.abs(data) > 1e-5)

    if np.any(valid_mask):
        abs_data = np.zeros(data.shape, dtype=float)
        abs_data[valid_mask] = np.abs(data[valid_mask])

        # Use the center of the strongest 1% valid voxels, not one isolated max voxel.
        valid_abs = abs_data[valid_mask]
        cutoff = np.percentile(valid_abs, 99)
        strong_mask = valid_mask & (abs_data >= cutoff)

        if np.any(strong_mask):
            ijk_mean = np.mean(np.argwhere(strong_mask), axis=0)
            xyz = nib.affines.apply_affine(img.affine, ijk_mean)
            return tuple(float(v) for v in xyz)

        ijk = np.unravel_index(np.argmax(abs_data), abs_data.shape)
        xyz = nib.affines.apply_affine(img.affine, ijk)
        return tuple(float(v) for v in xyz)

    return (0.0, 0.0, 0.0)



ortho_rows = []

srm_reference_path = NII_DIR / "srm_encoding_r_group.nii.gz"
srm_reference_cut_coords = _safe_cut_coords_step8_ortho(srm_reference_path)

for ortho_map in ortho_maps:
    if ortho_map["scale_group"] == "encoding_r":
        vmin_ortho, vmax_ortho = 0.0, shared_encoding_vmax_ortho
        scale_note_ortho = "shared_raw_and_srm_encoding_scale"
        symmetric_cbar_ortho = False
    else:
        vmin_ortho, vmax_ortho = -delta_abs_vmax_ortho, delta_abs_vmax_ortho
        scale_note_ortho = "symmetric_delta_scale"
        symmetric_cbar_ortho = False

    if ortho_map["scale_group"] == "delta_r":
        cut_coords_ortho = srm_reference_cut_coords
    else:
        cut_coords_ortho = _safe_cut_coords_step8_ortho(ortho_map["path"])

    out_path_ortho = FIG_ORTHO_DIR / f"{ortho_map['map_id']}_ortho_summary.png"


    threshold_ortho = 1e-5 if ortho_map["scale_group"] == "delta_r" else 0.00001
    disp = plot_stat_map(
        str(ortho_map["path"]),
        display_mode="ortho",
        cut_coords=cut_coords_ortho,
        colorbar=True,
        cmap=ortho_map["cmap"],
        threshold=threshold_ortho,
        vmin=vmin_ortho,
        vmax=vmax_ortho,
        black_bg=False,
        dim=-0.4,
        draw_cross=True,
        symmetric_cbar=symmetric_cbar_ortho,
        cbar_tick_format="%.5f",
        title=ortho_map["title"],
    )
    _format_colorbar_decimal_step8_ortho(disp)
    disp.savefig(out_path_ortho, dpi=280, bbox_inches="tight", pad_inches=0.05)
    plt.close()

    ortho_rows.append({
        "map_id": ortho_map["map_id"],
        "nifti_path": str(ortho_map["path"]),
        "figure_path": str(out_path_ortho),
        "display_mode": "ortho",
        "cut_x": float(cut_coords_ortho[0]),
        "cut_y": float(cut_coords_ortho[1]),
        "cut_z": float(cut_coords_ortho[2]),
        "vmin": float(vmin_ortho),
        "vmax": float(vmax_ortho),
        "colorbar_format": "decimal_5_places",
        "scale_note": scale_note_ortho,
        "interpretation": ortho_map["interpretation"],
    })

ortho_manifest_df = pd.DataFrame(ortho_rows)
ortho_manifest_csv = CSV_DIR / "step8_ortho_summary_figure_manifest.csv"
ortho_manifest_json = JSON_DIR / "step8_ortho_summary_figure_manifest.json"
ortho_manifest_df.to_csv(
    ortho_manifest_csv,
    index=False,
    encoding="utf-8-sig",
    float_format="%.5f",
)
with open(ortho_manifest_json, "w", encoding="utf-8") as f:
    json.dump(ortho_rows, f, indent=2, ensure_ascii=False)

print("\nStep 8 integrated add-on completed: compact orthogonal summary brain figures")
print(f"Saved figures to: {FIG_ORTHO_DIR}")
print(f"Saved manifest  : {ortho_manifest_csv}")
display(ortho_manifest_df)
